# CITADEL Single-Notebook Experiment Runner

This notebook is the sole experiment runner for the CITADEL journal-extension workspace. It contains the helper code inline, locates the tracked telemetry data, runs the CITADEL/TCAD full design-space sweep, quantifies lifecycle drift and benign recalibration, attaches hardware-cost estimates, exports FPGA/RTL golden vectors, and shows how future RTL synthesis results can be merged back into the paper tables.

The default settings use the tracked DDR telemetry in `data/telemetry/processed/ddr_data/` with the full TCAD grid. Keep `DATA_MODE = "real"` and `TCAD_PRESET = "full"` for paper results. Apple tier data is located at `data/telemetry/raw/apple_data/` and is kept separate from CINTAS hardware-cost claims.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

EARLY_SEED = "123"
EARLY_THREADS = "1"
for key in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ[key] = EARLY_THREADS
os.environ["PYTHONHASHSEED"] = EARLY_SEED
os.environ["MPLBACKEND"] = "Agg"
_mpl_config_dir = Path(tempfile.gettempdir()) / "citadel-matplotlib"
_mpl_config_dir.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(_mpl_config_dir)

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    markers = (
        Path("notebooks") / "exact_tcad_all_experiments.ipynb",
        Path("configs") / "tcad_grid_smoke.json",
        Path("data") / "external_sources.json",
    )
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError(f"Could not find CITADEL repo root from {start}")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
print(f"Repository: {REPO_ROOT}")
print(f"Python: {platform.python_version()} on {platform.platform()}")


## 1. Reproducibility Configuration

The notebook uses deterministic settings for every experiment:

- seed: `123`
- numerical threads: `1`
- automatic DDR and Apple telemetry discovery
- deterministic sample-data generator when `DATA_MODE = "sample"`
- repo-relative paths
- run manifests with package versions, git commit, input hashes, and output hashes
- live progress cards in long-running notebook cells plus `results/notebook_run/notebook_progress.log`

For the strictest cross-machine comparison, start Jupyter itself with `PYTHONHASHSEED=123`. The notebook still sets the environment variable for downstream tools.


In [ ]:
SEED = 123
THREADS = 1
SAMPLE_ROWS = 600

DDR_DATA_ROOT = REPO_ROOT / "data" / "telemetry" / "processed" / "ddr_data"
APPLE_DATA_ROOT = REPO_ROOT / "data" / "telemetry" / "raw" / "apple_data"
REAL_DATA_ROOT = DDR_DATA_ROOT
APPLE_TIER_DATA_ROOT = APPLE_DATA_ROOT
DATA_SOURCE_CONFIG = REPO_ROOT / "data" / "external_sources.json"

def csv_files(root: Path) -> list[Path]:
    return sorted(root.rglob("*.csv"))


def is_git_lfs_pointer(path: Path) -> bool:
    try:
        with path.open("rb") as f:
            return f.read(64).startswith(b"version https://git-lfs.github.com/spec")
    except OSError:
        return False


DDR_CSVS = csv_files(DDR_DATA_ROOT)
APPLE_CSVS = csv_files(APPLE_DATA_ROOT)
pointer_files = [p for p in [*DDR_CSVS[:3], *APPLE_CSVS[:3]] if is_git_lfs_pointer(p)]
if pointer_files:
    listed = "\n".join(str(p.relative_to(REPO_ROOT)) for p in pointer_files[:6])
    raise RuntimeError(
        "Telemetry CSVs are still Git LFS pointer files. Fetch the Git LFS objects "
        "with your Git client before running the notebook. Examples:\n" + listed
    )

# The repo now tracks real telemetry. Change to "sample" only for a tiny pipeline check.
DATA_MODE = "real" if DDR_CSVS else "sample"  # "real" or "sample"
TCAD_PRESET = "full"  # "full" for TCAD paper results; use "smoke" only for quick debugging
RUN_REPEAT_CHECK = True

RESULTS_ROOT = REPO_ROOT / "results" / "notebook_run"
DATA_ROOT = REPO_ROOT / "data" / "sample" if DATA_MODE == "sample" else REAL_DATA_ROOT
TCAD_OUT = RESULTS_ROOT / "tcad_ablation"
TCAD_REPEAT_OUT = RESULTS_ROOT / "tcad_ablation_repeat"
LIFECYCLE_OUT = RESULTS_ROOT / "lifecycle_drift"
FPGA_OUT = RESULTS_ROOT / "fpga"
RTL_SWEEP_OUT = RESULTS_ROOT / "rtl_sweep"
PROGRESS_LOG = RESULTS_ROOT / "notebook_progress.log"

import random

def configure_reproducibility(seed: int = 123, *, threads: int = 1, matplotlib_backend: str | None = "Agg") -> dict[str, str]:
    env_updates = {"PYTHONHASHSEED": str(seed)}
    for name in (
        "OMP_NUM_THREADS",
        "OPENBLAS_NUM_THREADS",
        "MKL_NUM_THREADS",
        "NUMEXPR_NUM_THREADS",
        "VECLIB_MAXIMUM_THREADS",
    ):
        env_updates[name] = str(threads)
    mpl_config_dir = Path(tempfile.gettempdir()) / "citadel-matplotlib"
    mpl_config_dir.mkdir(parents=True, exist_ok=True)
    env_updates["MPLCONFIGDIR"] = str(mpl_config_dir)
    if matplotlib_backend:
        env_updates["MPLBACKEND"] = matplotlib_backend
    for name, value in env_updates.items():
        os.environ[name] = value
    random.seed(seed)
    try:
        import numpy as _np
        _np.random.seed(seed)
    except Exception:
        pass
    return env_updates

env_updates = configure_reproducibility(seed=SEED, threads=THREADS, matplotlib_backend="Agg")
PROGRESS_LOG.parent.mkdir(parents=True, exist_ok=True)
PROGRESS_LOG.write_text(f"CITADEL notebook progress log\nseed={SEED} threads={THREADS} preset={TCAD_PRESET}\n", encoding="utf-8")
print(f"Progress log: {PROGRESS_LOG.relative_to(REPO_ROOT)}")
for key, value in env_updates.items():
    print(f"{key}={value}")

## 2. Integrated Notebook Utilities

This section keeps the shared CITADEL experiment utilities inside the notebook so data preparation, sample-data generation, TCAD design-space sweeps, figure generation, hardware-cost estimates, lifecycle drift checks, and manifest handling all run from one place.


In [ ]:
import hashlib
import json
import os
import subprocess
import time
import urllib.parse
import urllib.request
from dataclasses import asdict, dataclass
from itertools import product
from pathlib import Path
from typing import Any, Iterable, Sequence

import numpy as np
import pandas as pd


# ---- Inlined from former helper io.py ----

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Tuple

import numpy as np
import pandas as pd


META_COLS_BASE = {
    "setup",
    "scenario",
    "workload",
    "time_idx",
    "label",
    "is_anom",
}


def _sanitize_telemetry_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Drop autogenerated index columns and normalize header whitespace."""
    renamed = {col: str(col).strip() for col in df.columns}
    df = df.rename(columns=renamed)
    keep = [col for col in df.columns if col and not str(col).startswith("Unnamed:")]
    return df.loc[:, keep].copy()


def parse_scenario_workload_from_name(path: Path) -> Tuple[str, str]:
    """Parse (scenario, workload) from a telemetry CSV filename.

    Expected filenames are the same as in the draft notebooks, e.g.:

        DDR4_DROOP_dft.csv
        DDR4_benign_tr.csv
        DDR5_SPECTRE_mm.csv
        DDR5_benign_ni.csv

    Returns
    -------
    scenario : str
        One of BENIGN, DROOP, RH, SPECTRE (uppercased).
    workload : str
        Workload tag (uppercased), e.g., DFT, DJ, ...

    Notes
    -----
    This is intentionally strict: it helps catch data naming issues early.
    """
    stem = path.stem  # e.g. 'DDR4_DROOP_dft'
    parts = stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Cannot parse scenario/workload from filename: {path.name}")

    scen_raw = parts[1].strip().upper()
    wl_raw = parts[2].strip().upper()

    scenario = "BENIGN" if scen_raw == "BENIGN" else scen_raw
    workload = wl_raw
    return scenario, workload


def load_telemetry_for_setup(setup: str, data_root: Path) -> pd.DataFrame:
    """Load and concatenate telemetry CSVs for a given setup.

    Parameters
    ----------
    setup:
        'A' (DDR4) or 'B' (DDR5).
    data_root:
        Folder containing telemetry CSVs.

    Returns
    -------
    pd.DataFrame
        Concatenated telemetry with meta columns:
          - setup ('A' or 'B')
          - scenario ('BENIGN', 'DROOP', 'RH', 'SPECTRE', ...)
          - workload (e.g. 'DFT', 'DP', ...)
          - time_idx (row index inside each file if missing)
          - is_anom (0 for BENIGN else 1)
          - label  (same as is_anom at sample level)
    """
    setup_u = setup.upper()
    if setup_u == "A":
        prefix = "DDR4_"
    elif setup_u == "B":
        prefix = "DDR5_"
    else:
        raise ValueError(f"Unknown setup: {setup}. Expected 'A' or 'B'.")

    data_root = Path(data_root)
    files = sorted(data_root.glob(f"{prefix}*.csv"))
    if not files:
        raise FileNotFoundError(
            f"No files found for setup {setup_u} with prefix {prefix} in {data_root}"
        )

    dfs = []
    for path in files:
        scenario, workload = parse_scenario_workload_from_name(path)
        df = _sanitize_telemetry_columns(pd.read_csv(path))

        # Add meta columns
        df["setup"] = setup_u
        df["scenario"] = scenario
        df["workload"] = workload

        if "time_idx" not in df.columns:
            df["time_idx"] = np.arange(len(df), dtype=int)

        is_anom = 0 if scenario == "BENIGN" else 1
        df["is_anom"] = is_anom
        df["label"] = is_anom

        dfs.append(df)

    full_df = pd.concat(dfs, ignore_index=True)

    # Sort for reproducible windowing later
    full_df = (
        full_df.sort_values(["workload", "scenario", "time_idx"])
        .reset_index(drop=True)
    )
    return full_df


def load_telemetry_two_setups(data_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Convenience loader: returns (setup_A_df, setup_B_df)."""
    data_root = Path(data_root)
    return load_telemetry_for_setup("A", data_root), load_telemetry_for_setup("B", data_root)


# ---- Inlined from former helper preprocessing.py ----

from typing import Iterable

import numpy as np
import pandas as pd



def get_feature_columns(df: pd.DataFrame, meta_cols: Iterable[str] | None = None) -> list[str]:
    """Return numeric telemetry columns used as features.

    The draft notebooks treat all numeric columns as candidate telemetry features
    except for meta/label columns.
    """
    meta = set(META_COLS_BASE if meta_cols is None else meta_cols)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    return [c for c in numeric_cols if c not in meta]


def clean_and_debias_telemetry(
    df: pd.DataFrame,
    scenario_col: str = "scenario",
    benign_name: str = "BENIGN",
    inplace: bool = False,
) -> pd.DataFrame:
    """Simple cleaning + benign de-biasing (from EXACT.ipynb).

    Steps
    -----
    1) Drop rows where *all* numeric feature columns are NaN.
    2) Fill remaining NaNs with BENIGN means if BENIGN exists else global means.
    3) Subtract BENIGN means (or global means) from every sample.

    Notes
    -----
    This is intentionally minimal and matches the draft notebook behavior.
    If you have monotonic counters, you may want to convert them to per-sample
    deltas before calling this function.
    """
    df2 = df if inplace else df.copy()

    feature_cols = get_feature_columns(df2)
    if not feature_cols:
        raise ValueError("No numeric feature columns found for telemetry.")

    # Drop rows where all feature columns are NaN
    mask_all_nan = df2[feature_cols].isna().all(axis=1)
    if mask_all_nan.any():
        df2 = df2.loc[~mask_all_nan].reset_index(drop=True)

    # Compute BENIGN means if available
    if scenario_col in df2.columns:
        ben_mask = df2[scenario_col].astype(str).str.upper() == benign_name.upper()
        if ben_mask.any():
            benign_means = df2.loc[ben_mask, feature_cols].mean(axis=0)
            df2[feature_cols] = df2[feature_cols].fillna(benign_means)
            df2[feature_cols] = df2[feature_cols] - benign_means
        else:
            global_means = df2[feature_cols].mean(axis=0)
            df2[feature_cols] = df2[feature_cols].fillna(global_means)
            df2[feature_cols] = df2[feature_cols] - global_means
    else:
        global_means = df2[feature_cols].mean(axis=0)
        df2[feature_cols] = df2[feature_cols].fillna(global_means)
        df2[feature_cols] = df2[feature_cols] - global_means

    return df2


def drop_constant_features(df: pd.DataFrame, feature_cols: list[str] | None = None) -> list[str]:
    """Return the subset of feature columns that are non-constant."""
    if feature_cols is None:
        feature_cols = get_feature_columns(df)
    keep = []
    for c in feature_cols:
        s = df[c]
        if s.nunique(dropna=True) > 1:
            keep.append(c)
    return keep


# ---- Inlined from former helper metrics.py ----

import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    brier_score_loss,
)


def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    """Expected Calibration Error (ECE) for binary probabilities."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    if y_true.size == 0:
        return float("nan")
    if np.unique(y_true).size < 2:
        return 0.0

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i < n_bins - 1:
            mask = (y_prob >= lo) & (y_prob < hi)
        else:
            mask = (y_prob >= lo) & (y_prob <= hi)
        if not np.any(mask):
            continue
        p_bin = float(y_prob[mask].mean())
        y_bin = float(y_true[mask].mean())
        w_bin = float(mask.mean())
        ece += w_bin * abs(p_bin - y_bin)

    return float(ece)


def _safe_binary_metric(fn, y_true, y_score_or_pred, default=np.nan, **kwargs) -> float:
    y_true = np.asarray(y_true).astype(int)
    if y_true.size == 0 or np.unique(y_true).size < 2:
        return float(default)
    try:
        return float(fn(y_true, y_score_or_pred, **kwargs))
    except Exception:
        return float(default)


def compute_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> dict:
    """Compute detection metrics, including false-positive rate (FPR)."""
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    auc_roc = _safe_binary_metric(roc_auc_score, y_true, y_prob, default=np.nan)
    auc_pr = _safe_binary_metric(average_precision_score, y_true, y_prob, default=np.nan)
    f1 = _safe_binary_metric(f1_score, y_true, y_pred, default=0.0)
    bal_acc = _safe_binary_metric(balanced_accuracy_score, y_true, y_pred, default=0.5)
    mcc = _safe_binary_metric(matthews_corrcoef, y_true, y_pred, default=0.0)
    brier = _safe_binary_metric(brier_score_loss, y_true, y_prob, default=np.nan)
    ece = expected_calibration_error(y_true, y_prob, n_bins=10)

    neg_mask = y_true == 0
    fpr = float(np.mean(y_pred[neg_mask] == 1)) if np.any(neg_mask) else float("nan")

    return {
        "auc_roc": auc_roc,
        "auc_pr": auc_pr,
        "f1": f1,
        "bal_acc": bal_acc,
        "mcc": mcc,
        "fpr": fpr,
        "brier": brier,
        "ece": ece,
    }


# ---- Inlined from former helper cintas.py ----

from dataclasses import dataclass
from typing import Literal, Tuple

import numpy as np
import pandas as pd


@dataclass(frozen=True)
class CINTASModel:
    """CINTAS/CIAS parameter bundle + scoring functions.

    This is the same scoring core used in EXACT.ipynb:

      * Normalize telemetry using BENIGN mean/std
      * Compute:
          E2(t) = Σ_f w_f * z_f(t)^2
          E1(t) = Σ_f w_f * |z_f(t)|
          score(t) = (1 - λ)E2(t) + λE1(t)

    In the draft paper, CINTAS runs in fixed-point on-chip. This class is a
    *software reference* implementation; see FixedPointCINTAS for a small
    Q-format prototype.

    Attributes
    ----------
    feature_cols:
        Feature names (and order) used by this model.
    mu:
        BENIGN mean per feature.
    sigma:
        BENIGN std per feature.
    w:
        Non-negative weights per feature (sum to 1).
    lambda_res:
        Mixing coefficient λ in [0, 1].
    """
    feature_cols: list[str]
    mu: np.ndarray
    sigma: np.ndarray
    w: np.ndarray
    lambda_res: float = 0.5

    def _sigma_safe(self) -> np.ndarray:
        return np.where(self.sigma <= 1e-6, 1.0, self.sigma)

    def score_array(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Compute (score, E1, E2) for X with shape (n_samples, n_features)."""
        s = self._sigma_safe()
        Z = (X - self.mu) / s

        E2 = np.sum(self.w * (Z ** 2), axis=1)
        E1 = np.sum(self.w * np.abs(Z), axis=1)

        score = (1.0 - self.lambda_res) * E2 + self.lambda_res * E1
        return score.astype(float), E1.astype(float), E2.astype(float)

    def score_dataframe(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        X = df[self.feature_cols].to_numpy(dtype=float)
        return self.score_array(X)


def fit_cintas_from_benign(
    df: pd.DataFrame,
    feature_cols: list[str],
    lambda_res: float = 0.5,
    *,
    scenario_col: str = "scenario",
    benign_name: str = "BENIGN",
    weight_mode: Literal["inv_var", "uniform"] = "inv_var",
) -> CINTASModel:
    """Fit CINTAS parameters from BENIGN rows only (unsupervised).

    The draft notebooks use inverse variance as a lightweight unsupervised
    weighting scheme (stable features get larger weight). Use uniform weights
    to match the unweighted form shown in the paper text.
    """
    if scenario_col not in df.columns:
        raise KeyError(f"Expected '{scenario_col}' column to fit from benign.")

    ben = df[df[scenario_col].astype(str).str.upper() == benign_name.upper()].copy()
    if ben.empty:
        raise ValueError("No BENIGN rows found; cannot fit CINTAS.")

    X_ben = ben[feature_cols].to_numpy(dtype=float)
    mu = X_ben.mean(axis=0)
    sigma = X_ben.std(axis=0, ddof=1)

    if weight_mode == "uniform":
        w = np.ones_like(mu, dtype=float) / max(len(mu), 1)
    elif weight_mode == "inv_var":
        inv_var = 1.0 / np.maximum(sigma ** 2, 1e-8)
        w_raw = np.maximum(inv_var, 0.0)
        w = (w_raw / w_raw.sum()) if np.any(w_raw) else (np.ones_like(w_raw) / len(w_raw))
    else:
        raise ValueError(f"Unknown weight_mode: {weight_mode}")

    return CINTASModel(
        feature_cols=list(feature_cols),
        mu=mu,
        sigma=sigma,
        w=w,
        lambda_res=float(lambda_res),
    )


# -------------------------------------------------------------------------
# Fixed-point reference (Q-format) — optional helper for edge-style scoring.
# -------------------------------------------------------------------------

@dataclass(frozen=True)
class FixedPointConfig:
    q: int = 15  # Q15 default

    @property
    def scale(self) -> int:
        return 1 << self.q


@dataclass(frozen=True)
class FixedPointCINTAS:
    """A small fixed-point prototype.

    This is not meant to be bit-exact to any RTL; it is a *reference* that
    keeps the core arithmetic in integers.
    """
    feature_cols: list[str]
    mu_q: np.ndarray        # int
    gamma_q: np.ndarray     # int, approx 1/sigma in Qq
    w_q: np.ndarray         # int, weights in Qq (sum ~= 1.0)
    lambda_q: int           # int, lambda in Qq
    cfg: FixedPointConfig = FixedPointConfig()

    @classmethod
    def from_float_model(cls, model: CINTASModel, cfg: FixedPointConfig = FixedPointConfig()) -> "FixedPointCINTAS":
        q = cfg.q
        scale = cfg.scale
        mu_q = np.round(model.mu * scale).astype(np.int64)
        sigma_safe = np.where(model.sigma <= 1e-6, 1.0, model.sigma)
        gamma = 1.0 / sigma_safe
        gamma_q = np.round(gamma * scale).astype(np.int64)

        w_q = np.round(model.w * scale).astype(np.int64)
        lambda_q = int(round(float(model.lambda_res) * scale))

        return cls(
            feature_cols=list(model.feature_cols),
            mu_q=mu_q,
            gamma_q=gamma_q,
            w_q=w_q,
            lambda_q=lambda_q,
            cfg=cfg,
        )

    def score_sample_int(self, x: np.ndarray) -> int:
        """Score one sample; returns Qq value as int."""
        q = self.cfg.q
        scale = self.cfg.scale

        # x is float → quantize to Qq
        x_q = np.round(np.asarray(x, dtype=float) * scale).astype(np.int64)

        # z_q = (x - mu) * gamma  (all Qq)
        # (x_q - mu_q) is Qq; multiply by gamma_q (Qq) → Q2q; shift back to Qq.
        z_q = ((x_q - self.mu_q) * self.gamma_q) >> q  # Qq

        abs_z = np.abs(z_q)  # Qq

        # E1 = Σ w * |z|  (Qq*Qq→Q2q >>q →Qq)
        e1_q = int(np.sum((self.w_q * abs_z) >> q))

        # E2 = Σ w * z^2
        # z^2: Qq^2 → Q2q; multiply by w_q(Qq) → Q3q; >>2q → Qq
        z2_q2 = (z_q * z_q)  # Q2q
        e2_q = int(np.sum((self.w_q * z2_q2) >> (2 * q)))

        # score = (1-λ)E2 + λE1
        one_q = scale
        score_q = ((one_q - self.lambda_q) * e2_q + self.lambda_q * e1_q) >> q
        return int(score_q)

    def score_dataframe(self, df: pd.DataFrame) -> np.ndarray:
        X = df[self.feature_cols].to_numpy(dtype=float)
        scores = np.zeros(X.shape[0], dtype=np.int64)
        for i in range(X.shape[0]):
            scores[i] = self.score_sample_int(X[i])
        return scores


# ---- Inlined from former helper windowing.py ----

from dataclasses import dataclass
from typing import Literal, Optional

import numpy as np
import pandas as pd



AggMode = Literal["mean", "max", "median"]


def score_samples(df: pd.DataFrame, model: CINTASModel) -> np.ndarray:
    """Compute per-sample CINTAS scores for every row."""
    score, _, _ = model.score_dataframe(df)
    return score


def build_window_dataset(
    df: pd.DataFrame,
    model: CINTASModel,
    *,
    scenario_anom: str,
    window_size: int,
    agg_mode: AggMode = "max",
    balance_per_workload: bool = True,
    seed: int = 123,
    scenario_col: str = "scenario",
    workload_col: str = "workload",
    time_col: str = "time_idx",
) -> pd.DataFrame:
    """Build a benign-vs-anomaly window dataset for one anomaly scenario.

    Matches the logic in EXACT.ipynb (S6):
      * Keep only BENIGN and the specified anomaly scenario.
      * Split into non-overlapping windows per (workload, scenario).
      * Aggregate per-window scores with mean/max/median.
      * Optionally balance benign/anomaly windows per workload.

    Returns a table with one row per window:
      setup, scenario, workload, window_size, agg_mode, score_win, label
    """
    rng = np.random.default_rng(seed)

    if "setup" not in df.columns:
        raise KeyError("df must contain a 'setup' column.")
    setup_vals = df["setup"].unique()
    if len(setup_vals) != 1:
        raise ValueError(f"df must contain exactly one setup; got {setup_vals}.")

    setup = str(setup_vals[0])

    scen_anom_u = scenario_anom.upper()
    df_sub = df[df[scenario_col].astype(str).str.upper().isin(["BENIGN", scen_anom_u])].copy()
    if df_sub.empty:
        return pd.DataFrame()

    if time_col not in df_sub.columns:
        df_sub[time_col] = df_sub.groupby([scenario_col, workload_col]).cumcount()

    df_sub = df_sub.sort_values([workload_col, scenario_col, time_col]).reset_index(drop=True)

    df_sub["score_sample"] = score_samples(df_sub, model)

    rows = []
    for wl, df_w in df_sub.groupby(workload_col):
        df_b = df_w[df_w[scenario_col].astype(str).str.upper() == "BENIGN"]
        df_a = df_w[df_w[scenario_col].astype(str).str.upper() == scen_anom_u]

        s_b = df_b["score_sample"].to_numpy(dtype=float)
        s_a = df_a["score_sample"].to_numpy(dtype=float)

        if s_b.size < window_size or s_a.size < window_size:
            continue

        n_win_b = s_b.size // window_size
        n_win_a = s_a.size // window_size

        if balance_per_workload:
            n_win = min(n_win_b, n_win_a)
            n_win_b = n_win_a = n_win

        def _agg(scores_1d: np.ndarray, n_win: int) -> np.ndarray:
            out = []
            for i in range(n_win):
                start = i * window_size
                end = start + window_size
                w_scores = scores_1d[start:end]
                if w_scores.size < window_size:
                    break
                if agg_mode == "mean":
                    s_win = float(w_scores.mean())
                elif agg_mode == "median":
                    s_win = float(np.median(w_scores))
                else:  # "max"
                    s_win = float(w_scores.max())
                out.append(s_win)
            return np.asarray(out, dtype=float)

        win_scores_b = _agg(s_b, n_win_b)
        win_scores_a = _agg(s_a, n_win_a)

        for s in win_scores_b:
            rows.append({
                "setup": setup,
                "scenario": scen_anom_u,
                "workload": str(wl),
                "window_size": int(window_size),
                "agg_mode": str(agg_mode),
                "score_win": float(s),
                "label": 0,
            })
        for s in win_scores_a:
            rows.append({
                "setup": setup,
                "scenario": scen_anom_u,
                "workload": str(wl),
                "window_size": int(window_size),
                "agg_mode": str(agg_mode),
                "score_win": float(s),
                "label": 1,
            })

    return pd.DataFrame(rows)


# ---- Inlined from former helper evaluation.py ----

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, List, Optional

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold



@dataclass(frozen=True)
class DroopConfig:
    """A small tuning bundle used by the draft notebooks."""
    setup: str
    window_size: int
    n_splits: int
    lambda_res: float
    agg_mode: str
    p_quantile: float


def evaluate_windows_kfold(
    win_df: pd.DataFrame,
    *,
    setup: str,
    scenario_eval: str,
    window_size: int,
    agg_mode: str,
    lambda_res: float,
    p_quantile: float,
    n_splits: int,
    seed: int,
) -> List[dict]:
    """Evaluate window scores with Stratified K-fold CV.

    This mirrors EXACT.ipynb (S6/S7) behavior:
      - Choose threshold τ as p-quantile of BENIGN train scores.
      - Compute predictions on test scores (>= τ).
      - Convert scores to pseudo-probabilities using min-max scaling on train.
      - Report BOTH global (workload='ALL') and per-workload rows.

    Returns a list of row-dicts.
    """
    if win_df.empty:
        return []

    scores = win_df["score_win"].to_numpy(dtype=float)
    labels = win_df["label"].to_numpy(dtype=int)
    workloads = win_df["workload"].astype(str).to_numpy() if "workload" in win_df.columns else np.array(["ALL"] * len(win_df))

    if np.unique(labels).size < 2:
        return []

    requested_n_splits = int(n_splits)
    class_counts = pd.Series(labels).value_counts()
    min_class_count = int(class_counts.min()) if not class_counts.empty else 0
    effective_n_splits = min(requested_n_splits, min_class_count)
    if effective_n_splits < 2:
        return []

    kf = StratifiedKFold(n_splits=effective_n_splits, shuffle=True, random_state=seed)
    rows: list[dict] = []

    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(scores, labels)):
        train_scores = scores[train_idx]
        train_labels = labels[train_idx]

        ben_train = train_scores[train_labels == 0]
        tau = float(np.quantile(ben_train if ben_train.size else train_scores, p_quantile))

        test_scores = scores[test_idx]
        test_labels = labels[test_idx]
        test_workloads = workloads[test_idx]

        # pseudo-probabilities
        s_min, s_max = float(train_scores.min()), float(train_scores.max())
        denom = (s_max - s_min) if (s_max > s_min) else 1.0
        test_probs = np.clip((test_scores - s_min) / denom, 0.0, 1.0)

        test_pred = (test_scores >= tau).astype(int)

        # Global
        metrics_all = compute_binary_metrics(test_labels, test_pred, test_probs)
        row_all = {
            "setup": setup,
            "scenario": scenario_eval,
            "workload": "ALL",
            "window_size": int(window_size),
            "n_splits": int(effective_n_splits),
            "requested_n_splits": int(requested_n_splits),
            "fold_idx": int(fold_idx),
            "agg_mode": str(agg_mode),
            "lambda_res": float(lambda_res),
            "p_quantile": float(p_quantile),
            "n_test": int(test_labels.size),
            "n_benign": int((test_labels == 0).sum()),
            "n_anom": int((test_labels == 1).sum()),
            "tau": float(tau),
        }
        row_all.update(metrics_all)
        rows.append(row_all)

        # Per-workload
        for wl in np.unique(test_workloads):
            mask = test_workloads == wl
            y_w = test_labels[mask]
            if y_w.size < 2 or np.unique(y_w).size < 2:
                continue
            pred_w = test_pred[mask]
            prob_w = test_probs[mask]
            met_w = compute_binary_metrics(y_w, pred_w, prob_w)
            row_w = dict(row_all)
            row_w["workload"] = str(wl)
            row_w["n_test"] = int(y_w.size)
            row_w["n_benign"] = int((y_w == 0).sum())
            row_w["n_anom"] = int((y_w == 1).sum())
            row_w.update(met_w)
            rows.append(row_w)

    return rows


def run_exact_eval_for_setup(
    *,
    setup: str,
    df: pd.DataFrame,
    model: CINTASModel,
    scenarios_eval: Iterable[str],
    window_sizes: Iterable[int],
    n_splits_list: Iterable[int] = (5,),
    default_p_quantile: float = 0.99,
    agg_mode: str = "max",
    droop_cfg: DroopConfig | None = None,
    seed: int = 123,
) -> pd.DataFrame:
    """Run the draft-style evaluation loop for one setup.

    Parameters
    ----------
    df:
        Single-setup telemetry dataframe (with scenario/workload/time_idx).
    model:
        Fitted CINTASModel.
    droop_cfg:
        If provided, overrides settings for DROOP scenario (used in notebook).
    """
    setup_u = setup.upper()

    rows = []
    for scen in scenarios_eval:
        scen_u = scen.upper()
        for W in window_sizes:
            for n_splits in n_splits_list:
                if droop_cfg is not None and scen_u == "DROOP":
                    lambda_res = droop_cfg.lambda_res
                    p_quantile = droop_cfg.p_quantile
                    agg = droop_cfg.agg_mode
                else:
                    lambda_res = model.lambda_res
                    p_quantile = default_p_quantile
                    agg = agg_mode

                win_df = build_window_dataset(
                    df,
                    model=model,
                    scenario_anom=scen_u,
                    window_size=int(W),
                    agg_mode=agg,
                    balance_per_workload=True,
                    seed=seed,
                )
                if win_df.empty:
                    continue

                fold_rows = evaluate_windows_kfold(
                    win_df,
                    setup=setup_u,
                    scenario_eval=scen_u,
                    window_size=int(W),
                    agg_mode=agg,
                    lambda_res=float(lambda_res),
                    p_quantile=float(p_quantile),
                    n_splits=int(n_splits),
                    seed=seed,
                )
                rows.extend(fold_rows)

    return pd.DataFrame(rows)


def save_exact_summaries_for_setup(setup: str, results_df: pd.DataFrame, out_dir: Path) -> None:
    """Save global + per-workload summaries, similar to EXACT.ipynb."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = results_df.copy()
    setup_u = setup.upper()

    # Global summary (workload=ALL), averaged over folds
    global_df = df[df["workload"] == "ALL"].copy()
    if not global_df.empty:
        g = (
            global_df.groupby(["scenario", "window_size"], as_index=False)
            .agg(
                auc_roc=("auc_roc", "mean"),
                auc_pr=("auc_pr", "mean"),
                f1=("f1", "mean"),
                bal_acc=("bal_acc", "mean"),
                mcc=("mcc", "mean"),
                fpr=("fpr", "mean"),
                brier=("brier", "mean"),
                ece=("ece", "mean"),
                n_test=("n_test", "mean"),
            )
            .sort_values(["scenario", "window_size"])
        )
        g.to_csv(out_dir / f"SETUP_{setup_u}_EXACT_summary_global.csv", index=False)

    # Per-workload summary, averaged over folds
    per_wl_df = df[df["workload"] != "ALL"].copy()
    if not per_wl_df.empty:
        p = (
            per_wl_df.groupby(["scenario", "workload", "window_size"], as_index=False)
            .agg(
                auc_roc=("auc_roc", "mean"),
                auc_pr=("auc_pr", "mean"),
                f1=("f1", "mean"),
                bal_acc=("bal_acc", "mean"),
                mcc=("mcc", "mean"),
                fpr=("fpr", "mean"),
                brier=("brier", "mean"),
                ece=("ece", "mean"),
                n_test=("n_test", "mean"),
            )
            .sort_values(["scenario", "workload", "window_size"])
        )
        p.to_csv(out_dir / f"SETUP_{setup_u}_EXACT_summary_per_workload.csv", index=False)


# ---- Inlined from former helper causal_corr.py ----

from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd

try:
    import networkx as nx
except Exception:  # pragma: no cover
    nx = None

import matplotlib.pyplot as plt


def assign_feature_domain(feat: str) -> str:
    """Heuristic mapping from feature name to domain (EXACT.ipynb).

    Returns one of: CORE, MEMORY, SENSOR
    """
    name = feat.lower()
    if any(tok in name for tok in ["temp", "therm", "pk", "pkg", "pwr", "power", "volt", "vcc"]):
        return "SENSOR"
    if any(tok in name for tok in ["dram", "mem", "imc", "unc_m", "llc", "l3", "l2", "dimm"]):
        return "MEMORY"
    return "CORE"


def build_edges_from_corr(
    corr_df: pd.DataFrame,
    corr_threshold: float = 0.35,
) -> pd.DataFrame:
    """Turn a correlation matrix into an edge list (directional heuristic)."""
    features = list(corr_df.columns)
    domains = {f: assign_feature_domain(f) for f in features}
    domain_order = {"CORE": 0, "MEMORY": 1, "SENSOR": 2}

    rows = []
    for i, f_i in enumerate(features):
        for j in range(i + 1, len(features)):
            f_j = features[j]
            rho = float(corr_df.iloc[i, j])
            if not np.isfinite(rho) or abs(rho) < corr_threshold:
                continue

            dom_i = domains[f_i]
            dom_j = domains[f_j]

            if domain_order[dom_i] < domain_order[dom_j]:
                src, dst = f_i, f_j
            elif domain_order[dom_j] < domain_order[dom_i]:
                src, dst = f_j, f_i
            else:
                src, dst = (f_i, f_j) if f_i <= f_j else (f_j, f_i)

            rows.append({"src": src, "dst": dst, "rho": rho, "abs_rho": abs(rho)})
    return pd.DataFrame(rows)


def compute_degree_centrality(
    features: list[str],
    edges_df: pd.DataFrame,
) -> Dict[str, float]:
    """Degree centrality on the undirected skeleton (normalized)."""
    n = max(len(features) - 1, 1)
    deg_counts = {f: 0 for f in features}
    if edges_df is None or edges_df.empty:
        return {f: 0.0 for f in features}
    for row in edges_df.itertuples():
        deg_counts[row.src] += 1
        deg_counts[row.dst] += 1
    return {f: deg_counts[f] / n for f in features}


def compute_feature_cias_alignment(
    df: pd.DataFrame,
    features: list[str],
    *,
    score_col: str = "cias_sample_score",
    scenarios: tuple[str, ...] = ("DROOP", "RH", "SPECTRE"),
) -> Dict[str, float]:
    """Average |Spearman rho| between each feature and score_col over scenarios."""
    corr_map = {f: [] for f in features}
    df2 = df.copy()
    df2["scenario"] = df2["scenario"].astype(str).str.upper()

    for scen in scenarios:
        df_s = df2[df2["scenario"] == scen.upper()]
        if df_s.empty:
            continue
        for f in features:
            if f not in df_s.columns:
                continue
            sub = df_s[[f, score_col]].dropna()
            if sub[f].nunique() <= 1 or sub[score_col].nunique() <= 1:
                continue
            rho = sub.corr(method="spearman").iloc[0, 1]
            if np.isfinite(rho):
                corr_map[f].append(abs(float(rho)))

    return {f: float(np.mean(vals)) if vals else 0.0 for f, vals in corr_map.items()}


def _rank_gaussianize(df: pd.DataFrame) -> np.ndarray:
    """Copula/rank-Gaussian transform for robust conditional graph learning."""
    ranks = df.rank(method="average", pct=True).to_numpy(dtype=float)
    q = np.clip(ranks, 1e-4, 1.0 - 1e-4)
    try:
        from scipy.stats import norm
        z = norm.ppf(q)
    except Exception:
        z = np.log(q / (1.0 - q))
    z = np.asarray(z, dtype=float)
    z[~np.isfinite(z)] = 0.0
    z -= np.nanmean(z, axis=0, keepdims=True)
    std = np.nanstd(z, axis=0, ddof=1, keepdims=True)
    z /= np.maximum(std, 1e-8)
    z[~np.isfinite(z)] = 0.0
    return z


def _workload_stratified_subsample(df: pd.DataFrame, rng: np.random.Generator, frac: float) -> pd.DataFrame:
    if "workload" not in df.columns:
        n = len(df)
        take = max(8, int(np.ceil(float(frac) * n)))
        idx = rng.choice(df.index.to_numpy(), size=min(take, n), replace=False)
        return df.loc[np.sort(idx)].copy()
    parts = []
    for _, group in df.groupby("workload", sort=True):
        n = len(group)
        if n == 0:
            continue
        take = max(4, int(np.ceil(float(frac) * n)))
        take = min(take, n)
        idx = rng.choice(group.index.to_numpy(), size=take, replace=False)
        parts.append(group.loc[np.sort(idx)])
    return pd.concat(parts, axis=0).sort_index().copy() if parts else df.copy()


def feature_telemetry_cost(feature: str) -> float:
    """Small telemetry-acquisition cost prior used only for ranking ties/tradeoffs."""
    domain = assign_feature_domain(feature)
    if domain == "SENSOR":
        return 1.20
    if domain == "MEMORY":
        return 1.08
    if domain == "CORE":
        return 1.00
    return 1.12


def learn_stability_controlled_conditional_graph(
    df_benign: pd.DataFrame,
    features: list[str],
    *,
    corr_threshold: float = 0.35,
    n_bootstraps: int = 8,
    subsample_frac: float = 0.70,
    stability_threshold: float = 0.50,
    seed: int = 123,
    max_edges_per_boot_factor: int = 4,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Learn a CITADEL-specific stable conditional telemetry graph.

    This is intentionally different from the earlier EXACT-style nonlinear test stack.
    CITADEL uses a rank-Gaussian transform, shrinkage precision matrix, workload-
    stratified stability selection, and hardware-aware node metadata. The graph is
    learned from BENIGN telemetry only; anomaly labels are not used to create edges.
    """
    feats = [f for f in features if f in df_benign.columns]
    if not feats:
        empty_nodes = pd.DataFrame(columns=["feature", "domain", "graph_centrality", "edge_stability", "conditional_dependence", "telemetry_cost"])
        return pd.DataFrame(columns=["src", "dst", "rho", "abs_rho", "edge_stability", "dependence_score", "graph_method"]), empty_nodes

    n_feat = len(feats)
    min_abs_partial = max(0.08, float(corr_threshold) * 0.35)
    max_edges_per_boot = max(n_feat, int(max_edges_per_boot_factor) * n_feat)
    rng = np.random.default_rng(int(seed))

    edge_counts: dict[tuple[str, str], int] = {}
    edge_signed_sum: dict[tuple[str, str], float] = {}
    edge_abs_sum: dict[tuple[str, str], float] = {}

    for boot in range(max(int(n_bootstraps), 1)):
        sub = _workload_stratified_subsample(df_benign, rng, subsample_frac)
        x_df = sub[feats].astype(float).replace([np.inf, -np.inf], np.nan)
        x_df = x_df.fillna(x_df.median(numeric_only=True)).fillna(0.0)
        if len(x_df) < max(8, min(2 * n_feat, 32)):
            x_df = df_benign[feats].astype(float).replace([np.inf, -np.inf], np.nan)
            x_df = x_df.fillna(x_df.median(numeric_only=True)).fillna(0.0)

        z = _rank_gaussianize(x_df)
        if z.shape[0] < 4 or z.shape[1] < 2:
            continue
        cov = np.cov(z, rowvar=False)
        cov = np.atleast_2d(cov)
        avg_var = float(np.nanmean(np.diag(cov))) if cov.size else 1.0
        ridge = max(1e-4, 0.05 * avg_var)
        precision = np.linalg.pinv(cov + ridge * np.eye(n_feat))
        diag = np.sqrt(np.maximum(np.diag(precision), 1e-12))
        partial = -precision / np.outer(diag, diag)
        np.fill_diagonal(partial, 0.0)
        partial[~np.isfinite(partial)] = 0.0

        candidates: list[tuple[float, int, int, float]] = []
        for i in range(n_feat):
            for j in range(i + 1, n_feat):
                val = float(partial[i, j])
                aval = abs(val)
                if aval >= min_abs_partial:
                    candidates.append((aval, i, j, val))
        candidates.sort(reverse=True, key=lambda x: x[0])
        for aval, i, j, val in candidates[:max_edges_per_boot]:
            key = (feats[i], feats[j]) if feats[i] <= feats[j] else (feats[j], feats[i])
            sign_val = val if key == (feats[i], feats[j]) else -val
            edge_counts[key] = edge_counts.get(key, 0) + 1
            edge_signed_sum[key] = edge_signed_sum.get(key, 0.0) + sign_val
            edge_abs_sum[key] = edge_abs_sum.get(key, 0.0) + aval

    domains = {f: assign_feature_domain(f) for f in feats}
    domain_order = {"CORE": 0, "MEMORY": 1, "SENSOR": 2, "OTHER": 3}
    rows = []
    for (f_i, f_j), count in edge_counts.items():
        stability = float(count) / max(int(n_bootstraps), 1)
        mean_abs = edge_abs_sum[(f_i, f_j)] / max(count, 1)
        if stability < float(stability_threshold) or mean_abs < min_abs_partial:
            continue
        mean_signed = edge_signed_sum[(f_i, f_j)] / max(count, 1)
        dom_i, dom_j = domains[f_i], domains[f_j]
        if domain_order.get(dom_i, 99) < domain_order.get(dom_j, 99):
            src_f, dst_f, signed = f_i, f_j, mean_signed
        elif domain_order.get(dom_j, 99) < domain_order.get(dom_i, 99):
            src_f, dst_f, signed = f_j, f_i, -mean_signed
        else:
            src_f, dst_f, signed = (f_i, f_j, mean_signed) if f_i <= f_j else (f_j, f_i, -mean_signed)
        rows.append({
            "src": src_f,
            "dst": dst_f,
            "rho": float(signed),
            "abs_rho": float(abs(mean_signed)),
            "edge_stability": float(stability),
            "dependence_score": float(mean_abs * stability),
            "graph_method": "stable_rank_precision",
        })

    edges_df = pd.DataFrame(rows)
    if edges_df.empty and edge_counts:
        # Conservative fallback: keep the most stable conditional edges so sparse data
        # still produces an inspectable graph without reverting to the old method.
        fallback_rows = []
        for (f_i, f_j), count in edge_counts.items():
            stability = float(count) / max(int(n_bootstraps), 1)
            mean_abs = edge_abs_sum[(f_i, f_j)] / max(count, 1)
            mean_signed = edge_signed_sum[(f_i, f_j)] / max(count, 1)
            fallback_rows.append((stability * mean_abs, f_i, f_j, stability, mean_abs, mean_signed))
        fallback_rows.sort(reverse=True, key=lambda x: x[0])
        keep = fallback_rows[: max(1, min(len(fallback_rows), n_feat))]
        rows = []
        for _, f_i, f_j, stability, mean_abs, mean_signed in keep:
            rows.append({
                "src": f_i,
                "dst": f_j,
                "rho": float(mean_signed),
                "abs_rho": float(abs(mean_signed)),
                "edge_stability": float(stability),
                "dependence_score": float(mean_abs * stability),
                "graph_method": "stable_rank_precision_fallback",
            })
        edges_df = pd.DataFrame(rows)

    weighted_degree = {f: 0.0 for f in feats}
    stability_sum = {f: 0.0 for f in feats}
    dependence_sum = {f: 0.0 for f in feats}
    if not edges_df.empty:
        for row in edges_df.itertuples():
            w = float(getattr(row, "dependence_score", getattr(row, "abs_rho", 0.0)))
            st = float(getattr(row, "edge_stability", 0.0))
            dep = float(getattr(row, "abs_rho", 0.0))
            weighted_degree[row.src] += w
            weighted_degree[row.dst] += w
            stability_sum[row.src] += st
            stability_sum[row.dst] += st
            dependence_sum[row.src] += dep
            dependence_sum[row.dst] += dep
    max_deg = max(max(weighted_degree.values()), 1e-8)
    max_st = max(max(stability_sum.values()), 1e-8)
    max_dep = max(max(dependence_sum.values()), 1e-8)
    nodes_df = pd.DataFrame({
        "feature": feats,
        "domain": [domains[f] for f in feats],
        "graph_centrality": [weighted_degree[f] / max_deg for f in feats],
        "edge_stability": [stability_sum[f] / max_st for f in feats],
        "conditional_dependence": [dependence_sum[f] / max_dep for f in feats],
        "telemetry_cost": [feature_telemetry_cost(f) for f in feats],
        "graph_method": "stable_rank_precision",
    })
    return edges_df.sort_values(["dependence_score", "src", "dst"], ascending=[False, True, True], kind="mergesort").reset_index(drop=True), nodes_df


def build_causal_and_rank_features_for_setup(
    *,
    setup: str,
    df: pd.DataFrame,
    feature_cols: list[str],
    out_root: Path,
    corr_threshold: float = 0.35,
    top_k_plot: int = 20,
    score_col: str = "cias_sample_score",
    graph_bootstraps: int = 8,
    graph_subsample_frac: float = 0.70,
    graph_stability_threshold: float = 0.50,
    seed: int = 123,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build CITADEL's stable conditional telemetry graph and rank features."""
    setup_u = setup.upper()
    df_setup = df[df["setup"].astype(str).str.upper() == setup_u].copy()
    if df_setup.empty:
        raise ValueError(f"No rows found for setup={setup_u} in df.")

    df_ben = df_setup[df_setup["scenario"].astype(str).str.upper() == "BENIGN"].copy()
    if df_ben.empty:
        raise ValueError(f"No BENIGN rows found for setup={setup_u}.")

    feats = [f for f in feature_cols if f in df_ben.columns]
    if not feats:
        raise ValueError("None of feature_cols are present in the BENIGN dataframe.")

    edges_df, nodes_base = learn_stability_controlled_conditional_graph(
        df_ben,
        feats,
        corr_threshold=corr_threshold,
        n_bootstraps=graph_bootstraps,
        subsample_frac=graph_subsample_frac,
        stability_threshold=graph_stability_threshold,
        seed=int(seed) + (0 if setup_u == "A" else 1009),
    )
    align = compute_feature_cias_alignment(df_setup, feats, score_col=score_col)

    a = np.array([align[f] for f in feats], dtype=float)
    a_norm = a / max(float(np.max(a)), 1e-8)
    cost = nodes_base.set_index("feature")["telemetry_cost"].reindex(feats).fillna(1.0).to_numpy(dtype=float)
    central = nodes_base.set_index("feature")["graph_centrality"].reindex(feats).fillna(0.0).to_numpy(dtype=float)
    stability = nodes_base.set_index("feature")["edge_stability"].reindex(feats).fillna(0.0).to_numpy(dtype=float)
    dependence = nodes_base.set_index("feature")["conditional_dependence"].reindex(feats).fillna(0.0).to_numpy(dtype=float)
    cost_norm = cost / max(float(np.max(cost)), 1e-8)

    # TCAD-specific ranking: structural value and score alignment, penalized by
    # estimated telemetry-acquisition cost. This is distinct from the earlier
    # EXACT causal-test stack and directly supports hardware-aware DSE.
    raw_importance = (0.35 * central + 0.25 * stability + 0.20 * dependence + 0.20 * a_norm) / np.maximum(cost_norm, 1e-8)
    imp_norm = raw_importance / max(float(np.max(raw_importance)), 1e-8)

    domains = {f: assign_feature_domain(f) for f in feats}
    nodes_df = pd.DataFrame({
        "feature": feats,
        "domain": [domains[f] for f in feats],
        "graph_centrality": central,
        "edge_stability": stability,
        "conditional_dependence": dependence,
        "cias_alignment": a_norm,
        "telemetry_cost": cost,
        "hardware_cost_penalty": cost_norm,
        "importance_score": imp_norm,
        "graph_method": "stable_rank_precision",
    })
    ranks_df = (
        nodes_df.sort_values(
            ["importance_score", "edge_stability", "conditional_dependence", "feature"],
            ascending=[False, False, False, True],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    causal_dir = Path(out_root) / "causal"
    causal_dir.mkdir(parents=True, exist_ok=True)

    nodes_df.to_csv(causal_dir / f"SETUP_{setup_u}_causal_nodes.csv", index=False)
    edges_df.to_csv(causal_dir / f"SETUP_{setup_u}_causal_edges.csv", index=False)
    ranks_df.to_csv(causal_dir / f"SETUP_{setup_u}_feature_ranks.csv", index=False)

    if nx is not None and len(ranks_df):
        try:
            _plot_causal_and_topk(
                setup=setup_u,
                nodes_df=nodes_df,
                edges_df=edges_df,
                ranks_df=ranks_df,
                out_path=causal_dir / f"SETUP_{setup_u}_causal_graph_and_top{top_k_plot}.png",
                top_k=top_k_plot,
            )
        except Exception:
            pass

    return nodes_df, edges_df, ranks_df

def export_edges_for_fig5(edges_df: pd.DataFrame, out_csv: Path) -> None:
    """Export a CSV compatible with the Fig.5 plotting helper from EXACT-2.

    Output columns:
      u, v, strength, domain_u, domain_v
    """
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    def dom(name: str) -> str:
        d = assign_feature_domain(name)
        return {"CORE": "compute", "MEMORY": "memory", "SENSOR": "sensors"}.get(d, "compute")

    rows = []
    for r in edges_df.itertuples():
        u = str(r.src)
        v = str(r.dst)
        strength = float(getattr(r, "abs_rho", getattr(r, "rho", 0.0)))
        rows.append({
            "u": u,
            "v": v,
            "strength": strength,
            "domain_u": dom(u),
            "domain_v": dom(v),
        })
    pd.DataFrame(rows).to_csv(out_csv, index=False)


def _plot_causal_and_topk(
    *,
    setup: str,
    nodes_df: pd.DataFrame,
    edges_df: pd.DataFrame,
    ranks_df: pd.DataFrame,
    out_path: Path,
    top_k: int = 20,
) -> None:
    if nx is None:
        return

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    domains = {r.feature: r.domain for r in nodes_df.itertuples()}
    dom_y = {"CORE": 0.0, "MEMORY": 1.0, "SENSOR": 2.0}

    # Graph
    G = nx.DiGraph()
    for r in nodes_df.itertuples():
        G.add_node(r.feature, domain=r.domain, importance=r.importance_score)
    for r in edges_df.itertuples():
        G.add_edge(r.src, r.dst, weight=r.abs_rho)

    pos = {}
    by_dom = {"CORE": [], "MEMORY": [], "SENSOR": []}
    for n, d in G.nodes(data=True):
        by_dom.setdefault(d.get("domain", "CORE"), []).append(n)
    for dom, ns in by_dom.items():
        if not ns:
            continue
        xs = np.linspace(0.0, 1.0, len(ns))
        y = dom_y.get(dom, 0.0)
        for x, n in zip(xs, ns):
            pos[n] = (float(x), float(y))

    # Bar chart
    top_k = min(top_k, len(ranks_df))
    top = ranks_df.head(top_k)

    fig = plt.figure(figsize=(14, 6))
    ax_g = fig.add_subplot(1, 2, 1)
    ax_b = fig.add_subplot(1, 2, 2)

    color_map = {"CORE": "#1f77b4", "MEMORY": "#2ca02c", "SENSOR": "#ff7f0e"}

    nx.draw_networkx_edges(G, pos, ax=ax_g, alpha=0.2, arrows=False)
    nx.draw_networkx_nodes(
        G, pos, ax=ax_g,
        node_size=[250 + 1200 * float(G.nodes[n].get("importance", 0.0)) for n in G.nodes()],
        node_color=[color_map.get(domains.get(n, "CORE"), "#aaaaaa") for n in G.nodes()],
        linewidths=0.8, edgecolors="#333333"
    )
    ax_g.set_title(f"Setup {setup}: stable conditional telemetry graph")
    ax_g.axis("off")

    ax_b.barh(top["feature"][::-1], top["importance_score"][::-1])
    ax_b.set_title(f"Top-{top_k} ranked telemetry features")
    ax_b.set_xlabel("importance_score")
    ax_b.grid(True, axis="x", alpha=0.3)

    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)


# ---- Inlined from former helper plotting.py ----

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def plot_metrics_vs_window_size(
    *,
    per_workload_csv_A: Path,
    per_workload_csv_B: Path,
    out_png: Path,
    n_splits: int = 5,
    metrics: list[tuple[str, str, float | None, float | None]] | None = None,
    title: str = "Performance vs Decision-Block Length",
) -> None:
    """Recreate the 'MCC / Balanced Accuracy / Brier vs window length' style plot.

    Inputs are the per-workload summary CSVs produced by save_exact_summaries_for_setup().
    """
    if metrics is None:
        metrics = [
            ("mcc", "MCC", 0.5, 1.0),
            ("bal_acc", "Balanced Accuracy", 0.5, 1.0),
            ("fpr", "False-Positive Rate", 0.0, 0.2),
            ("brier", "Brier Score", None, None),
        ]

    df_a = pd.read_csv(per_workload_csv_A)
    df_b = pd.read_csv(per_workload_csv_B)

    if "n_splits" in df_a.columns:
        df_a = df_a[df_a["n_splits"] == n_splits]
    if "n_splits" in df_b.columns:
        df_b = df_b[df_b["n_splits"] == n_splits]

    window_sizes = sorted(set(df_a["window_size"].unique()).union(set(df_b["window_size"].unique())))

    fig, axes = plt.subplots(nrows=len(metrics), ncols=1, figsize=(12, 12), sharex=True)
    if len(metrics) == 1:
        axes = [axes]

    # Colors (match notebook)
    setup_a_color = "#1E88E5"
    setup_b_color = "#8B0000"

    for ax, (metric_col, metric_label, y_min, y_max) in zip(axes, metrics):
        # scatter points per workload
        for W in window_sizes:
            a_w = df_a[df_a["window_size"] == W]
            b_w = df_b[df_b["window_size"] == W]
            ax.scatter([W] * len(a_w), a_w[metric_col], s=18, color=setup_a_color, alpha=0.35, linewidths=0)
            ax.scatter([W] * len(b_w), b_w[metric_col], s=18, color=setup_b_color, alpha=0.35, linewidths=0)

        # mean lines across workloads
        a_mean = df_a.groupby("window_size")[metric_col].mean().reindex(window_sizes)
        b_mean = df_b.groupby("window_size")[metric_col].mean().reindex(window_sizes)
        ax.plot(window_sizes, a_mean.values, color=setup_a_color, linewidth=2.5, label="Setup A (mean)")
        ax.plot(window_sizes, b_mean.values, color=setup_b_color, linewidth=2.5, linestyle="--", label="Setup B (mean)")

        ax.set_ylabel(metric_label)
        ax.grid(True, alpha=0.25)
        if y_min is not None and y_max is not None:
            ax.set_ylim(y_min, y_max)

    axes[-1].set_xlabel("Decision-block length (window_size)")
    axes[0].set_title(title)

    axes[0].legend(loc="lower right")

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)


# ---- Inlined from former helper fig6.py ----

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D


def _domain_color(domain: str) -> str:
    d = str(domain).upper()
    if d == "CORE":
        return "#1f77b4"  # blue
    if d == "MEMORY":
        return "#9467bd"  # purple
    if d == "SENSOR":
        return "#ff7f0e"  # orange
    return "#7f7f7f"


def plot_topk_features_two_panel(
    *,
    ranks_csv_A: Path,
    ranks_csv_B: Path,
    out_png: Path,
    top_k: int = 15,
    title: str = "Feature Importance Analysis",
    setupA_label: str = "Setup A",
    setupB_label: str = "Setup B",
    hatch_align_quantile: float = 0.75,
    star_centrality_quantile: float = 0.75,
) -> None:
    """Produce a Fig. 6-style top-k feature bar plot.

    This uses the rank CSVs produced by the in-notebook stable conditional feature-ranking helper.

    Expected columns in each ranks CSV:
      - feature
      - domain (CORE/MEMORY/SENSOR)
      - importance_score
      - graph_centrality
      - cias_alignment
    """
    dfA = pd.read_csv(ranks_csv_A).copy()
    dfB = pd.read_csv(ranks_csv_B).copy()

    def prep(df: pd.DataFrame) -> pd.DataFrame:
        # tolerate older column naming
        if "score_alignment" in df.columns and "cias_alignment" not in df.columns:
            df = df.rename(columns={"score_alignment": "cias_alignment"})
        for col in ["importance_score", "graph_centrality", "cias_alignment"]:
            if col not in df.columns:
                raise ValueError(f"Missing column '{col}' in ranks CSV. Columns={list(df.columns)}")
        return df

    dfA = prep(dfA)
    dfB = prep(dfB)

    dfA = dfA.head(top_k).iloc[::-1].reset_index(drop=True)
    dfB = dfB.head(top_k).iloc[::-1].reset_index(drop=True)

    # thresholds for hatch/star
    hatch_thr_A = float(dfA["cias_alignment"].quantile(hatch_align_quantile))
    hatch_thr_B = float(dfB["cias_alignment"].quantile(hatch_align_quantile))
    star_thr_A  = float(dfA["graph_centrality"].quantile(star_centrality_quantile))
    star_thr_B  = float(dfB["graph_centrality"].quantile(star_centrality_quantile))

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)
    axA, axB = axes

    def draw(ax, df, setup_label, hatch_thr, star_thr):
        y = np.arange(len(df))
        colors = [_domain_color(d) for d in df["domain"]]
        bars = ax.barh(y, df["importance_score"], color=colors, edgecolor="#333333", linewidth=0.8)

        # Apply hatching for strong alignment
        for bar, align in zip(bars, df["cias_alignment"]):
            if float(align) >= hatch_thr:
                bar.set_hatch("///")
                bar.set_alpha(0.95)

        # Star for high centrality (placed near bar end)
        for yi, (imp, cent) in enumerate(zip(df["importance_score"], df["graph_centrality"])):
            if float(cent) >= star_thr:
                ax.scatter([float(imp)], [yi], marker="*", s=80, color="red", zorder=5)

        ax.set_yticks(y)
        ax.set_yticklabels(df["feature"], fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel("Feature importance")
        ax.set_title(setup_label)
        ax.grid(True, axis="x", alpha=0.25)

    draw(axA, dfA, setupA_label, hatch_thr_A, star_thr_A)
    draw(axB, dfB, setupB_label, hatch_thr_B, star_thr_B)

    # Legends
    legend_domains = [
        Patch(facecolor=_domain_color("CORE"), label="CORE"),
        Patch(facecolor=_domain_color("MEMORY"), label="MEMORY"),
        Patch(facecolor=_domain_color("SENSOR"), label="SENSORS"),
    ]
    legend_markers = [
        Patch(facecolor="white", edgecolor="#333333", hatch="///", label="High CINTAS alignment"),
        Line2D([0], [0], marker="*", color="w", markerfacecolor="red", markersize=12, label="High graph centrality"),
    ]
    axB.legend(handles=legend_domains + legend_markers, loc="lower right", fontsize=9, frameon=True)

    fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.tight_layout(rect=[0, 0.02, 1, 0.95])

    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)


# ---- Inlined from former helper fig5.py ----

import re
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import networkx as nx
from matplotlib.patches import Patch


# ---------------------------------------------------------------------
# Domain inference (copied from EXACT-2-Copy1.ipynb, trimmed)
# ---------------------------------------------------------------------
MEM_PAT = re.compile(r"(L[123]|\bLLC\b|CACHE|L3MISS|LLCMISS|DRAM|MEM(?!S)|BW)", re.IGNORECASE)
SEN_PAT = re.compile(r"(TEMP|THERM|Tdie|FAN|VOLT|POWER|ENERGY|SENSOR|THRM)", re.IGNORECASE)
CMP_PAT = re.compile(r"(ACYC|CPI|IPC|INST|C0res|C7res|TIME\(ticks\)|CYC|UOPS|ALU|CORE|SYSTEM|SOCKET)", re.IGNORECASE)

def infer_domain(name: str) -> str:
    if MEM_PAT.search(name): return "memory"
    if SEN_PAT.search(name): return "sensors"
    if CMP_PAT.search(name): return "compute"
    return "compute"


def _safe_savefig(path: Path, fig=None) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if fig is None:
        plt.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
        plt.close()
    else:
        fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
        plt.close(fig)


def _normalize_pos_to_circle(pos: dict, r: float = 1.0) -> dict:
    if not pos:
        return pos
    xy = np.array(list(pos.values()), dtype=float)
    xy = xy - xy.mean(axis=0, keepdims=True)
    rad = np.sqrt((xy**2).sum(axis=1))
    m = rad.max() if rad.size else 1.0
    if m > 0:
        xy = (r * 0.98) * xy / m
    return {n: tuple(xy[i]) for i, n in enumerate(pos.keys())}


def _pick_majority(dct: dict) -> str:
    order = ["compute", "memory", "sensors", "other"]
    if not dct:
        return "other"
    m = max(dct.values())
    cands = [k for k, v in dct.items() if v == m]
    for k in order:
        if k in cands:
            return k
    return "other"


def _scale_size(deg_map: dict, n: str, lo=220.0, hi=520.0) -> float:
    if not deg_map:
        return (lo + hi) / 2
    dvals = list(deg_map.values())
    dmin, dmax = min(dvals), max(dvals)
    d = deg_map.get(n, dmin)
    if dmax == dmin:
        return (lo + hi) / 2
    return lo + (hi - lo) * (d - dmin) / (dmax - dmin)


def _clean_name(n: str) -> str:
    n = str(n)
    if "Core" in n and "Socket" in n:
        try:
            core_id = n.split("Core", 1)[1].split("(", 1)[0].strip()
            metric = n.split(".", 1)[1].split("_")[-1] if "." in n else n.split("_")[-1]
            return f"C{core_id}.{metric}"
        except Exception:
            return n
    if "System" in n:
        parts = n.split(".", 1)
        return f"Sys.{parts[1]}" if len(parts) > 1 else "System"
    if "Socket" in n and "Core" not in n:
        parts = n.split(".", 1)
        return f"Socket.{parts[1]}" if len(parts) > 1 else "Socket"
    return n


def _draw_labels_with_boxes(ax, pos, label_nodes, fontsize=11, radial_offset=0.03) -> None:
    if not label_nodes:
        return
    xy = np.array([pos[n] for n in label_nodes if n in pos], dtype=float)
    if xy.size == 0:
        return
    center = xy.mean(axis=0)

    for n in label_nodes:
        if n not in pos:
            continue
        x, y = pos[n]
        v = np.array([x, y]) - center
        nv = np.linalg.norm(v)
        if nv > 0:
            v = v / nv
            x2, y2 = (np.array([x, y]) + radial_offset * v).tolist()
        else:
            x2, y2 = x, y

        t = ax.text(
            x2, y2, _clean_name(n),
            fontsize=fontsize,
            fontweight="bold",
            ha="center", va="center",
            zorder=10,
            bbox=dict(
                boxstyle="round,pad=0.25,rounding_size=0.15",
                facecolor="white",
                edgecolor="none",
                alpha=0.90
            )
        )
        t.set_path_effects([pe.withStroke(linewidth=2, foreground="white")])


def _load_edges_csv(edges_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(edges_csv)
    # Required columns: u, v, strength. Domains are optional.
    if not {"u", "v", "strength"}.issubset(df.columns):
        raise ValueError(f"Edges CSV must include columns u,v,strength. Got: {list(df.columns)}")
    return df


def _build_3domain_graph(edges_df: pd.DataFrame) -> tuple[nx.Graph, dict, dict]:
    # Ensure domains exist
    df = edges_df.copy()
    if "domain_u" not in df.columns:
        df["domain_u"] = df["u"].map(lambda s: infer_domain(str(s)))
    if "domain_v" not in df.columns:
        df["domain_v"] = df["v"].map(lambda s: infer_domain(str(s)))

    # normalize strength to [0,1]
    s = df["strength"].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0).values
    if len(s) and np.nanmax(s) > np.nanmin(s):
        s_norm = (s - np.nanmin(s)) / (np.nanmax(s) - np.nanmin(s))
    else:
        s_norm = np.zeros_like(s)
    df["strength_norm"] = s_norm

    G = nx.Graph()
    for _, r in df.iterrows():
        u, v = str(r["u"]), str(r["v"])
        w = float(r["strength_norm"])
        du, dv = str(r["domain_u"]).lower(), str(r["domain_v"]).lower()
        G.add_node(u); G.add_node(v)
        G.add_edge(u, v, strength=w, domain_u=du, domain_v=dv)

    # node domain by majority incidence
    domain_counts = {}
    for u, v, d in G.edges(data=True):
        for node, dn in [(u, d.get("domain_u", "other")), (v, d.get("domain_v", "other"))]:
            domain_counts.setdefault(node, {"compute": 0, "memory": 0, "sensors": 0, "other": 0})
            domain_counts[node][dn if dn in domain_counts[node] else "other"] += 1

    node_domain = {n: _pick_majority(domain_counts.get(n, {})) for n in G.nodes()}
    keep_nodes = [n for n, dn in node_domain.items() if dn in {"compute", "memory", "sensors"}]
    G3 = G.subgraph(keep_nodes).copy()
    deg = dict(G3.degree())
    return G3, node_domain, deg


def _draw_one_panel(ax, G3: nx.Graph, node_domain: dict, deg: dict, title: str, k_full: float, seed: int, label_only_top_n: int) -> None:
    domain_colors = {"compute": "#9ecae1", "memory": "#a1d99b", "sensors": "#fdae6b"}

    if G3.number_of_nodes() == 0:
        ax.set_title(title, fontsize=18, fontweight="bold", pad=10)
        ax.axis("off")
        return

    pos = nx.spring_layout(G3, k=k_full, iterations=300, seed=seed)
    pos = _normalize_pos_to_circle(pos, r=1.0)

    node_sizes = [_scale_size(deg, n) for n in G3.nodes()]
    node_colors = [domain_colors.get(node_domain.get(n, "other"), "#bdbdbd") for n in G3.nodes()]

    edge_strengths = [G3[u][v].get("strength", 0.0) for u, v in G3.edges()]
    edge_colors = [plt.cm.viridis(x) for x in edge_strengths]
    edge_widths = [0.35 + 1.4 * x for x in edge_strengths]

    nx.draw_networkx_edges(G3, pos, edge_color=edge_colors, width=edge_widths, alpha=0.20, ax=ax)
    nx.draw_networkx_nodes(
        G3, pos,
        node_size=node_sizes,
        node_color=node_colors,
        alpha=0.95,
        edgecolors="#2f2f2f",
        linewidths=1.0,
        ax=ax
    )

    # label selection
    top_nodes = [n for n, _ in sorted(deg.items(), key=lambda t: t[1], reverse=True)[:label_only_top_n]]
    _draw_labels_with_boxes(ax, pos, top_nodes, fontsize=10, radial_offset=0.035)

    ax.set_title(title, fontsize=18, fontweight="bold", pad=10)
    ax.axis("off")


def plot_benign_sparse_causal_network_two_panel(
    *,
    edges_csv_A: Path,
    edges_csv_B: Path,
    out_png: Path,
    title_A: str = "Setup A",
    title_B: str = "Setup B",
    label_only_top_n: int = 12,
    spring_k: float = 10.0,
    seed: int = 42,
) -> None:
    """Create the two-panel network plot (Fig. 5 style)."""
    edges_A = _load_edges_csv(Path(edges_csv_A))
    edges_B = _load_edges_csv(Path(edges_csv_B))

    G3A, node_dom_A, deg_A = _build_3domain_graph(edges_A)
    G3B, node_dom_B, deg_B = _build_3domain_graph(edges_B)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    axA, axB = axes

    _draw_one_panel(axA, G3A, node_dom_A, deg_A, title_A, spring_k, seed, label_only_top_n)
    _draw_one_panel(axB, G3B, node_dom_B, deg_B, title_B, spring_k, seed, label_only_top_n)

    # Legend + shared colorbar
    domain_colors = {"compute": "#9ecae1", "memory": "#a1d99b", "sensors": "#fdae6b"}
    legend_elements = [
        Patch(facecolor=domain_colors["compute"], label="Compute", alpha=0.95),
        Patch(facecolor=domain_colors["memory"],  label="Memory",  alpha=0.95),
        Patch(facecolor=domain_colors["sensors"], label="Sensors", alpha=0.95),
    ]
    axB.legend(handles=legend_elements, loc="upper left", bbox_to_anchor=(1.02, 1.02),
               frameon=False, fontsize=12)

    sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=plt.Normalize(vmin=0.0, vmax=1.0))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), fraction=0.03, pad=0.02)
    cbar.set_label("Connection strength (normalized)", rotation=270, labelpad=18)

    fig.tight_layout()
    _safe_savefig(Path(out_png), fig=fig)


# ---- Inlined from former helper hardware.py ----

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


SETUP_B_AREA_MM2 = 215.25
IDLE_POWER_W = 35.5
DEFAULT_HARDWARE_COSTS = REPO_ROOT / "hardware" / "cintas_operator_costs.csv"

WORKLOAD_POWER_W = {
    "DFT": 139.5,
    "DJ": 64.7,
    "DP": 99.3,
    "GS": 96.55,
    "GL": 69.1,
    "HA": 87.5,
    "JA": 89.8,
    "MM": 141.5,
    "NI": 97.5,
    "OE": 114.7,
    "PI": 96.8,
    "SH": 113.0,
    "TR": 94.9,
}

SENSOR_TOKENS = ("sensor", "sen", "temp", "therm", "pkg_power", "power", "pwr", "volt", "vcc", "energy")
MEMORY_TOKENS = ("mem", "dram", "imc", "llc", "l2", "cache", "dimm", "row", "rh")
COMPUTE_TOKENS = ("com", "core", "cpu", "uops", "ipc", "cpi", "cycles", "instr", "branch")


@dataclass(frozen=True)
class OperatorCosts:
    """Add/multiply library costs from Eduardo Ortega's reference calculation.

    The source code reads ``hw.csv`` and divides the area row by ``1000**2``.
    We keep the same conversion here, treating the raw area entries as square
    micrometers even if the legacy CSV label says ``area (mm^2)``.
    """

    add_area_mm2: float
    add_power_mw_at_1ghz: float
    add_delay_ps: float
    add_cycles: int
    mult_area_mm2: float
    mult_power_mw_at_1ghz: float
    mult_delay_ps: float
    mult_cycles: int
    bit_width: int = 16
    source: str = "Eduardo Ortega hardware table"

    @classmethod
    def from_csv(cls, path: Path = DEFAULT_HARDWARE_COSTS) -> "OperatorCosts":
        df = pd.read_csv(path)
        columns = set(df.columns)
        if {"operator", "area_mm2", "power_mw_at_1ghz", "delay_ps", "cycles"}.issubset(columns):
            return cls._from_normalized(df)
        if {"metrics", "add", "mult"}.issubset(columns):
            return cls._from_reference_hw_csv(df)
        raise ValueError(f"Unsupported hardware cost CSV schema: {path}")

    @classmethod
    def _from_normalized(cls, df: pd.DataFrame) -> "OperatorCosts":
        by_op = df.set_index("operator")
        add = by_op.loc["add"]
        mult = by_op.loc["mult"]
        source = str(add.get("source", "Eduardo Ortega hardware table"))
        bit_width = int(add.get("bit_width", 16))
        return cls(
            add_area_mm2=float(add["area_mm2"]),
            add_power_mw_at_1ghz=float(add["power_mw_at_1ghz"]),
            add_delay_ps=float(add["delay_ps"]),
            add_cycles=int(add["cycles"]),
            mult_area_mm2=float(mult["area_mm2"]),
            mult_power_mw_at_1ghz=float(mult["power_mw_at_1ghz"]),
            mult_delay_ps=float(mult["delay_ps"]),
            mult_cycles=int(mult["cycles"]),
            bit_width=bit_width,
            source=source,
        )

    @classmethod
    def _from_reference_hw_csv(cls, df: pd.DataFrame) -> "OperatorCosts":
        def metric(prefix: str, op: str) -> float:
            mask = df["metrics"].astype(str).str.lower().str.startswith(prefix)
            if not mask.any():
                raise ValueError(f"Missing metric row with prefix {prefix!r}")
            return float(df.loc[mask, op].iloc[0])

        area_scale = float(1000**2)
        return cls(
            add_area_mm2=metric("area", "add") / area_scale,
            add_power_mw_at_1ghz=metric("power", "add"),
            add_delay_ps=metric("delay", "add"),
            add_cycles=int(metric("cycles", "add")),
            mult_area_mm2=metric("area", "mult") / area_scale,
            mult_power_mw_at_1ghz=metric("power", "mult"),
            mult_delay_ps=metric("delay", "mult"),
            mult_cycles=int(metric("cycles", "mult")),
        )


@dataclass(frozen=True)
class HardwareCost:
    feature_count: int
    group_feature_counts: dict[str, int]
    aggregator_inputs: int
    frequency_ghz: float
    std_area_mm2: float
    agg_area_mm2: float
    total_area_mm2: float
    total_power_mw: float
    setup_b_area_overhead_pct: float
    idle_power_overhead_pct: float
    median_workload_power_overhead_pct: float
    add_count: int
    mult_count: int
    estimated_serial_cycles: int
    operator_bit_width: int
    add_delay_ps: float
    mult_delay_ps: float

    def to_summary_dict(self, prefix: str = "hw") -> dict[str, float | int]:
        return {
            f"{prefix}_frequency_ghz": self.frequency_ghz,
            f"{prefix}_operator_bit_width": self.operator_bit_width,
            f"{prefix}_std_area_mm2": self.std_area_mm2,
            f"{prefix}_agg_area_mm2": self.agg_area_mm2,
            f"{prefix}_area_mm2": self.total_area_mm2,
            f"{prefix}_power_mw": self.total_power_mw,
            f"{prefix}_setup_b_area_overhead_pct": self.setup_b_area_overhead_pct,
            f"{prefix}_idle_power_overhead_pct": self.idle_power_overhead_pct,
            f"{prefix}_median_workload_power_overhead_pct": self.median_workload_power_overhead_pct,
            f"{prefix}_add_count": self.add_count,
            f"{prefix}_mult_count": self.mult_count,
            f"{prefix}_estimated_serial_cycles": self.estimated_serial_cycles,
            f"{prefix}_add_delay_ps": self.add_delay_ps,
            f"{prefix}_mult_delay_ps": self.mult_delay_ps,
        }


def standard_euclidean_area(n_features: int, costs: OperatorCosts | None = None) -> float:
    costs = OperatorCosts.from_csv() if costs is None else costs
    n_features = int(n_features)
    if n_features <= 0:
        return 0.0
    per_feature = 2.0 * costs.mult_area_mm2 + costs.add_area_mm2
    adder_tree = max(n_features - 1, 0) * costs.add_area_mm2
    return n_features * per_feature + adder_tree


def standard_euclidean_power(
    n_features: int,
    *,
    frequency_ghz: float = 1.0,
    costs: OperatorCosts | None = None,
) -> float:
    costs = OperatorCosts.from_csv() if costs is None else costs
    n_features = int(n_features)
    if n_features <= 0:
        return 0.0
    per_feature = 2.0 * costs.mult_power_mw_at_1ghz + costs.add_power_mw_at_1ghz
    adder_tree = max(n_features - 1, 0) * costs.add_power_mw_at_1ghz
    return float(frequency_ghz) * (n_features * per_feature + adder_tree)


def aggregation_area(n_inputs: int, costs: OperatorCosts | None = None) -> float:
    costs = OperatorCosts.from_csv() if costs is None else costs
    return 2.0 * costs.mult_area_mm2 if int(n_inputs) > 0 else 0.0


def aggregation_power(
    n_inputs: int,
    *,
    frequency_ghz: float = 1.0,
    costs: OperatorCosts | None = None,
) -> float:
    costs = OperatorCosts.from_csv() if costs is None else costs
    return float(frequency_ghz) * 2.0 * costs.mult_power_mw_at_1ghz if int(n_inputs) > 0 else 0.0


def classify_feature_group(feature_name: str) -> str:
    name = str(feature_name).lower()
    if any(token in name for token in SENSOR_TOKENS):
        return "SEN"
    if any(token in name for token in MEMORY_TOKENS):
        return "MEM"
    if any(token in name for token in COMPUTE_TOKENS):
        return "COM"
    return "OTHER"


def count_feature_groups(feature_names: list[str] | tuple[str, ...]) -> dict[str, int]:
    counts = {"COM": 0, "MEM": 0, "SEN": 0, "OTHER": 0}
    for feature in feature_names:
        counts[classify_feature_group(feature)] += 1
    return {key: value for key, value in counts.items() if value > 0}


def format_feature_group_counts(group_feature_counts: dict[str, int]) -> str:
    return ";".join(f"{key}={int(group_feature_counts[key])}" for key in sorted(group_feature_counts))


def estimate_cintas_hardware_cost(
    *,
    feature_names: list[str] | tuple[str, ...] | None = None,
    n_features: int | None = None,
    group_feature_counts: dict[str, int] | None = None,
    frequency_ghz: float = 1.0,
    setup_b_area_mm2: float = SETUP_B_AREA_MM2,
    idle_power_w: float = IDLE_POWER_W,
    workload_powers_w: dict[str, float] | None = None,
    costs: OperatorCosts | None = None,
) -> HardwareCost:
    """Estimate CINTAS cost using the provided add/mult hardware table.

    This ports the reference formulas:

    - STD block per feature: ``2 * mult + 1 * add``
    - STD adder tree: ``n_features - 1`` adders
    - AGG block: ``2 * mult``
    - Power scales linearly with the supplied GHz point
    """
    costs = OperatorCosts.from_csv() if costs is None else costs
    workload_powers_w = WORKLOAD_POWER_W if workload_powers_w is None else workload_powers_w

    if group_feature_counts is None:
        if feature_names is not None:
            group_feature_counts = count_feature_groups(tuple(feature_names))
        elif n_features is not None:
            group_feature_counts = {"ALL": int(n_features)}
        else:
            raise ValueError("Provide feature_names, n_features, or group_feature_counts.")

    clean_counts = {str(k): int(v) for k, v in group_feature_counts.items() if int(v) > 0}
    feature_count = int(sum(clean_counts.values()))
    aggregator_inputs = len(clean_counts) if feature_count > 0 else 0

    std_area = sum(standard_euclidean_area(nf, costs) for nf in clean_counts.values())
    std_power = sum(
        standard_euclidean_power(nf, frequency_ghz=frequency_ghz, costs=costs)
        for nf in clean_counts.values()
    )
    agg_area = aggregation_area(aggregator_inputs, costs)
    agg_power = aggregation_power(aggregator_inputs, frequency_ghz=frequency_ghz, costs=costs)

    total_area = std_area + agg_area
    total_power_mw = std_power + agg_power
    total_power_w = total_power_mw / 1000.0
    median_workload_power = float(np.median(np.array(list(workload_powers_w.values()), dtype=float)))

    add_count = sum(nf + max(nf - 1, 0) for nf in clean_counts.values())
    mult_count = sum(2 * nf for nf in clean_counts.values()) + (2 if aggregator_inputs > 0 else 0)
    estimated_cycles = (
        sum(nf * (2 * costs.mult_cycles + costs.add_cycles) + max(nf - 1, 0) * costs.add_cycles for nf in clean_counts.values())
        + (2 * costs.mult_cycles if aggregator_inputs > 0 else 0)
    )

    return HardwareCost(
        feature_count=feature_count,
        group_feature_counts=clean_counts,
        aggregator_inputs=aggregator_inputs,
        frequency_ghz=float(frequency_ghz),
        std_area_mm2=float(std_area),
        agg_area_mm2=float(agg_area),
        total_area_mm2=float(total_area),
        total_power_mw=float(total_power_mw),
        setup_b_area_overhead_pct=100.0 * float(total_area) / float(setup_b_area_mm2),
        idle_power_overhead_pct=100.0 * float(total_power_w) / float(idle_power_w),
        median_workload_power_overhead_pct=100.0 * float(total_power_w) / median_workload_power,
        add_count=int(add_count),
        mult_count=int(mult_count),
        estimated_serial_cycles=int(estimated_cycles),
        operator_bit_width=int(costs.bit_width),
        add_delay_ps=float(costs.add_delay_ps),
        mult_delay_ps=float(costs.mult_delay_ps),
    )


def compute_power_from_features(num_features: int, ghz: float = 1.0, std: bool = True) -> float:
    if std:
        return standard_euclidean_power(num_features, frequency_ghz=ghz)
    return aggregation_power(num_features, frequency_ghz=ghz)


def compute_area_from_features(num_features: int, std: bool = True) -> float:
    if std:
        return standard_euclidean_area(num_features)
    return aggregation_area(num_features)


def compute_tableIII_setupB(
    *,
    n_features: int = 15,
    frequency_ghz: float = 1.0,
    setup_b_die_area_mm2: float = SETUP_B_AREA_MM2,
    idle_power_w: float = IDLE_POWER_W,
    workload_powers_w: dict[str, float] | None = None,
) -> pd.DataFrame:
    """Create the Setup B overhead table from the add/mult hardware costs."""
    workload_powers_w = WORKLOAD_POWER_W if workload_powers_w is None else workload_powers_w
    cost = estimate_cintas_hardware_cost(
        n_features=int(n_features),
        frequency_ghz=float(frequency_ghz),
        setup_b_area_mm2=float(setup_b_die_area_mm2),
        idle_power_w=float(idle_power_w),
        workload_powers_w=workload_powers_w,
    )

    data = [
        {
            "method": "EXACT (CINTAS)",
            "n_features": int(n_features),
            "area_mm2": cost.total_area_mm2,
            "power_mw": cost.total_power_mw,
            "area_overhead_pct": cost.setup_b_area_overhead_pct,
            "idle_power_overhead_pct": cost.idle_power_overhead_pct,
        },
        {
            "method": "OCTANE",
            "n_features": 227,
            "area_mm2": np.nan,
            "power_mw": np.nan,
            "area_overhead_pct": 1.2,
            "idle_power_overhead_pct": 2.6,
        },
        {
            "method": "E-SCOUT",
            "n_features": 230,
            "area_mm2": np.nan,
            "power_mw": np.nan,
            "area_overhead_pct": 2.2,
            "idle_power_overhead_pct": 1.0,
        },
    ]
    return pd.DataFrame(data)


# ---- Inlined from former helper sample_data.py ----

from pathlib import Path

import numpy as np
import pandas as pd


DEFAULT_WORKLOADS = ("dft", "dj", "mm", "tr")
DEFAULT_ROWS = 2000


def _base_frame(
    *,
    n_rows: int,
    rng: np.random.Generator,
    setup_scale: float,
    workload_scale: float,
) -> pd.DataFrame:
    t = np.arange(n_rows, dtype=float)
    phase = 0.0125 * t * workload_scale
    phase_fast = 0.031 * t

    data = {
        "core_ipc": 1.60 * setup_scale + 0.08 * np.sin(phase) + rng.normal(0.0, 0.020, n_rows),
        "core_cpi": 0.82 / setup_scale + 0.03 * np.cos(phase) + rng.normal(0.0, 0.012, n_rows),
        "uops_retired": 4.10 * workload_scale + 0.18 * np.sin(phase_fast) + rng.normal(0.0, 0.060, n_rows),
        "core_cycles": 3.40 * setup_scale + 0.15 * np.cos(phase_fast / 2.0) + rng.normal(0.0, 0.050, n_rows),
        "l2_miss": 0.22 * workload_scale + 0.04 * np.sin(phase_fast) + rng.normal(0.0, 0.010, n_rows),
        "llc_miss": 0.18 * workload_scale + 0.03 * np.cos(phase_fast) + rng.normal(0.0, 0.008, n_rows),
        "dram_bw": 1.25 * workload_scale + 0.10 * np.sin(phase / 3.0) + rng.normal(0.0, 0.035, n_rows),
        "imc_reads": 0.94 * workload_scale + 0.07 * np.cos(phase / 4.0) + rng.normal(0.0, 0.025, n_rows),
        "pkg_power": 66.0 * setup_scale + 2.2 * np.sin(phase / 5.0) + rng.normal(0.0, 0.40, n_rows),
        "socket_temp": 47.0 + 1.2 * np.sin(phase / 6.0) + rng.normal(0.0, 0.25, n_rows),
        "dimm_temp": 39.0 + 0.9 * np.cos(phase / 7.0) + rng.normal(0.0, 0.20, n_rows),
        "vcc_voltage": 1.04 * setup_scale + 0.006 * np.sin(phase / 8.0) + rng.normal(0.0, 0.002, n_rows),
    }
    return pd.DataFrame(data)


def _apply_scenario_shift(df: pd.DataFrame, scenario: str) -> pd.DataFrame:
    out = df.copy()
    t = np.arange(len(out), dtype=float)
    bursts = ((t // 100) % 2).astype(float)

    if scenario == "DROOP":
        out["vcc_voltage"] -= 0.080 + 0.015 * bursts
        out["core_ipc"] -= 0.180 + 0.025 * bursts
        out["core_cpi"] += 0.160 + 0.020 * bursts
        out["pkg_power"] += 3.5 + 0.8 * bursts
        out["socket_temp"] += 1.3 + 0.2 * bursts
    elif scenario == "RH":
        out["dram_bw"] += 0.330 + 0.040 * bursts
        out["imc_reads"] += 0.260 + 0.025 * bursts
        out["llc_miss"] += 0.070 + 0.015 * bursts
        out["dimm_temp"] += 2.6 + 0.4 * bursts
        out["pkg_power"] += 2.0
    elif scenario == "SPECTRE":
        out["llc_miss"] += 0.130 + 0.025 * bursts
        out["l2_miss"] += 0.085 + 0.020 * bursts
        out["uops_retired"] += 0.320 + 0.050 * bursts
        out["core_ipc"] -= 0.120
        out["pkg_power"] += 2.8

    mins = {
        "core_ipc": 0.10,
        "core_cpi": 0.05,
        "uops_retired": 0.10,
        "core_cycles": 0.10,
        "l2_miss": 0.0,
        "llc_miss": 0.0,
        "dram_bw": 0.0,
        "imc_reads": 0.0,
        "pkg_power": 0.10,
        "socket_temp": 0.10,
        "dimm_temp": 0.10,
        "vcc_voltage": 0.10,
    }
    for column, lower in mins.items():
        out[column] = np.clip(out[column], lower, None)
    return out


def create_sample_dataset(
    out_root: Path,
    *,
    seed: int = 123,
    workloads: tuple[str, ...] = DEFAULT_WORKLOADS,
    n_rows: int = DEFAULT_ROWS,
) -> list[Path]:
    """Create a small deterministic telemetry dataset for smoke tests and demos."""
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    created: list[Path] = []
    scenarios = ("benign", "DROOP", "RH", "SPECTRE")

    for setup_idx, setup_name in enumerate(("DDR4", "DDR5")):
        setup_scale = 1.0 + 0.06 * setup_idx
        for workload_idx, workload in enumerate(workloads):
            workload_scale = 1.0 + 0.04 * workload_idx
            for scenario_idx, scenario_name in enumerate(scenarios):
                file_seed = seed + 10_000 * setup_idx + 1_000 * scenario_idx + 10 * workload_idx
                rng = np.random.default_rng(file_seed)
                frame = _base_frame(
                    n_rows=n_rows,
                    rng=rng,
                    setup_scale=setup_scale,
                    workload_scale=workload_scale,
                )
                if scenario_name != "benign":
                    frame = _apply_scenario_shift(frame, scenario_name)

                out_path = out_root / f"{setup_name}_{scenario_name}_{workload}.csv"
                frame.to_csv(out_path, index=False)
                created.append(out_path)

    return sorted(created)


# ---- Inlined from former helper repro.py ----

import dataclasses
import hashlib
import json
import os
import platform
import random
import subprocess
import tempfile
from importlib import metadata
from pathlib import Path
from typing import Iterable


DEFAULT_SEED = 123
DEFAULT_THREADS = 1
THREAD_ENV_VARS = (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
)
RUNTIME_PACKAGES = (
    "numpy",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "networkx",
    "jupyterlab",
    "nbformat",
    "ipykernel",
)
ENVIRONMENT_FILES = (
    "requirements.txt",
    "environment.yml",
)


def find_repo_root(start: Path | None = None) -> Path:
    """Return the repository root by looking for the notebook/config/data markers."""
    current = Path.cwd() if start is None else Path(start).resolve()
    if current.is_file():
        current = current.parent

    markers = (
        Path("notebooks") / "exact_tcad_all_experiments.ipynb",
        Path("configs") / "tcad_grid_smoke.json",
        Path("data") / "external_sources.json",
    )
    for candidate in (current, *current.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate

    raise FileNotFoundError(f"Could not find repo root from {current}")


def configure_reproducibility(
    seed: int = DEFAULT_SEED,
    *,
    threads: int = DEFAULT_THREADS,
    matplotlib_backend: str | None = "Agg",
) -> dict[str, str]:
    """Set deterministic process knobs before importing numerical libraries."""
    env_updates = {"PYTHONHASHSEED": str(seed)}
    for name in THREAD_ENV_VARS:
        env_updates[name] = str(threads)
    mpl_config_dir = Path(tempfile.gettempdir()) / "citadel-matplotlib"
    mpl_config_dir.mkdir(parents=True, exist_ok=True)
    env_updates["MPLCONFIGDIR"] = str(mpl_config_dir)
    if matplotlib_backend:
        env_updates["MPLBACKEND"] = matplotlib_backend

    for name, value in env_updates.items():
        os.environ[name] = value

    random.seed(seed)

    try:
        import numpy as np

        np.random.seed(seed)
    except Exception:
        pass

    return env_updates


def sha256_file(path: Path, *, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def _git_commit(repo_root: Path) -> str | None:
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=repo_root,
            check=True,
            capture_output=True,
            text=True,
        )
    except Exception:
        return None
    return result.stdout.strip() or None


def _git_dirty(repo_root: Path) -> bool | None:
    try:
        result = subprocess.run(
            ["git", "status", "--porcelain"],
            cwd=repo_root,
            check=True,
            capture_output=True,
            text=True,
        )
    except Exception:
        return None
    return bool(result.stdout.strip())


def _safe_relative(path: Path, repo_root: Path) -> str:
    try:
        return str(path.resolve().relative_to(repo_root.resolve()))
    except ValueError:
        return str(path.resolve())


def runtime_versions() -> dict[str, str]:
    versions: dict[str, str] = {}
    for package in RUNTIME_PACKAGES:
        try:
            versions[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            continue
    return versions


def upstream_method_provenance() -> dict[str, Any]:
    """Record EXACT-to-CITADEL method lineage in every run manifest.

    The ETS baseline is reproduced inside this CITADEL notebook. It is not
    executed from a live checkout of the upstream EXACT repository during a
    notebook run, so the upstream repo/ref/commit are recorded explicitly.
    Override these values with CITADEL_EXACT_REPO, CITADEL_EXACT_REF, or
    CITADEL_EXACT_COMMIT when reproducing from a different EXACT revision.
    """
    exact_commit = os.environ.get("CITADEL_EXACT_COMMIT", "b6b8b17ef825d9f4fad754f88c1de583b09805b9").strip()
    return {
        "upstream_exact": {
            "name": "EXACT",
            "repo": os.environ.get("CITADEL_EXACT_REPO", "https://github.com/ping830616/EXACT"),
            "ref": os.environ.get("CITADEL_EXACT_REF", "HEAD"),
            "commit": exact_commit or None,
            "role": "conference-version methodology and CINTAS baseline reproduced inside CITADEL",
        },
        "citadel_extension": [
            "single-notebook reproducibility manifests",
            "TCAD design-space sweep over feature budget, window size, lambda, aggregation, weighting, and fixed-point precision",
            "false-positive-rate reporting for field SLM practicality",
            "lifecycle drift and benign recalibration experiments",
            "hardware cost, fixed-point golden vectors, and RTL/FPGA integration hooks",
        ],
    }


def _progress_log_path() -> Path:
    return globals().get("PROGRESS_LOG", REPO_ROOT / "results" / "notebook_run" / "notebook_progress.log")


CITADEL_NOTEBOOK_PROGRESS_REV = "2026-05-08-safe-cv-dashboard"


def _progress_display_id(label: str) -> str:
    safe = "".join(ch.lower() if ch.isalnum() else "-" for ch in str(label)).strip("-")
    return f"citadel-progress-{safe or 'run'}"


def _progress_html(label: str, stage: str, *, start: float | None = None, current: int | None = None, total: int | None = None, detail: str | None = None) -> str:
    import html as _html

    pct = None
    if current is not None and total is not None:
        pct = max(0.0, min(100.0, 100.0 * float(current) / max(int(total), 1)))
    elapsed = f"{time.perf_counter() - start:.1f}s" if start is not None else "--"
    pct_text = f"{pct:.1f}%" if pct is not None else "working"
    count_text = f"{current}/{total}" if current is not None and total is not None else ""
    bar_width = pct if pct is not None else 100.0
    bar_color = "#2e7d32" if str(stage).upper() == "DONE" else "#1565c0"
    detail_html = f"<div style='margin-top:6px;color:#455a64'>{_html.escape(str(detail))}</div>" if detail else ""
    return f"""
    <div style='border:1px solid #cfd8dc;border-radius:6px;padding:10px 12px;margin:8px 0;background:#fbfdff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif'>
      <div style='display:flex;justify-content:space-between;gap:12px;align-items:baseline'>
        <div style='font-weight:650;color:#263238'>{_html.escape(str(label))}</div>
        <div style='font-size:12px;color:#607d8b'>elapsed {elapsed}</div>
      </div>
      <div style='margin-top:6px;font-size:13px;color:#37474f'>{_html.escape(str(stage))} {count_text}</div>
      <div style='height:10px;background:#e3edf5;border-radius:999px;overflow:hidden;margin-top:8px'>
        <div style='width:{bar_width:.2f}%;height:10px;background:{bar_color};border-radius:999px'></div>
      </div>
      <div style='margin-top:4px;font-size:12px;color:#607d8b'>{pct_text}</div>
      {detail_html}
    </div>
    """


def _progress_display(label: str, stage: str, *, start: float | None = None, current: int | None = None, total: int | None = None, detail: str | None = None) -> None:
    display_id = _progress_display_id(label)
    html_payload = _progress_html(label, stage, start=start, current=current, total=total, detail=detail)
    try:
        from IPython.display import HTML, display, update_display

        if stage == "START" or display_id not in globals().setdefault("_CITADEL_PROGRESS_DISPLAYS", set()):
            display(HTML(html_payload), display_id=display_id)
            globals().setdefault("_CITADEL_PROGRESS_DISPLAYS", set()).add(display_id)
        else:
            update_display(HTML(html_payload), display_id=display_id)
    except Exception:
        pass


def progress_start(label: str, detail: str | None = None) -> float:
    start = time.perf_counter()
    progress_update(label, "START", start=start, detail=detail)
    return start


def progress_update(label: str, stage: str, *, start: float | None = None, current: int | None = None, total: int | None = None, detail: str | None = None) -> None:
    clock = time.strftime("%Y-%m-%d %H:%M:%S")
    parts = [f"[{clock}]", f"[{label}]", str(stage)]
    if current is not None and total is not None:
        pct = 100.0 * float(current) / max(int(total), 1)
        parts.append(f"{current}/{total} ({pct:.1f}%)")
    if start is not None:
        parts.append(f"elapsed={time.perf_counter() - start:.1f}s")
    if detail:
        parts.append(str(detail))
    line = " | ".join(parts)
    _progress_display(label, stage, start=start, current=current, total=total, detail=detail)
    print(line, flush=True)
    try:
        log_path = _progress_log_path()
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with log_path.open("a", encoding="utf-8") as handle:
            handle.write(line + "\n")
    except Exception:
        pass


def progress_should_log(current: int, total: int, *, steps: int = 20) -> bool:
    if current <= 1 or current >= total:
        return True
    stride = max(1, int(total) // max(int(steps), 1))
    return current % stride == 0


def write_run_manifest(
    out_root: Path,
    *,
    repo_root: Path,
    data_root: Path,
    cfg: object,
    seed: int,
    artifact_paths: Iterable[Path] = (),
) -> Path:
    """Write a deterministic manifest with config, runtime, and file hashes."""
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    repo_root = Path(repo_root).resolve()
    data_root = Path(data_root).resolve()

    if dataclasses.is_dataclass(cfg):
        config_payload = dataclasses.asdict(cfg)
    else:
        config_payload = {"repr": repr(cfg)}

    data_files = sorted(p for p in data_root.rglob("*.csv") if p.is_file())
    artifacts = [Path(path).resolve() for path in artifact_paths if Path(path).exists()]
    environment_files = [repo_root / name for name in ENVIRONMENT_FILES if (repo_root / name).exists()]

    payload = {
        "seed": int(seed),
        "python_version": platform.python_version(),
        "python_implementation": platform.python_implementation(),
        "platform": platform.platform(),
        "git_commit": _git_commit(repo_root),
        "git_dirty": _git_dirty(repo_root),
        "method_provenance": upstream_method_provenance(),
        "config": config_payload,
        "environment": {
            key: os.environ.get(key)
            for key in ("PYTHONHASHSEED", *THREAD_ENV_VARS, "MPLBACKEND", "MPLCONFIGDIR")
            if os.environ.get(key) is not None
        },
        "packages": runtime_versions(),
        "environment_files": [
            {
                "path": _safe_relative(path, repo_root),
                "sha256": sha256_file(path),
                "size_bytes": path.stat().st_size,
            }
            for path in environment_files
        ],
        "data_root": _safe_relative(data_root, repo_root),
        "data_files": [
            {
                "path": _safe_relative(path, repo_root),
                "sha256": sha256_file(path),
                "size_bytes": path.stat().st_size,
            }
            for path in data_files
        ],
        "artifacts": [
            {
                "path": _safe_relative(path, repo_root),
                "sha256": sha256_file(path),
                "size_bytes": path.stat().st_size,
            }
            for path in artifacts
        ],
    }

    manifest_path = out_root / "run_manifest.json"
    manifest_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n")
    return manifest_path




# ---- EXACT baseline reproduction, now owned by this notebook ----




@dataclass(frozen=True)
class ETS2026Config:
    scenarios_eval: tuple[str, ...] = ("DROOP", "RH", "SPECTRE")
    window_sizes: tuple[int, ...] = tuple(range(50, 1001, 50))
    n_splits: int = 5
    lambda_res: float = 0.5
    p_quantile: float = 0.99
    agg_mode: str = "max"
    corr_threshold: float = 0.35
    top_k_features: int = 15
    seed: int = 123


def run_ets2026(
    *,
    data_root: Path,
    out_root: Path,
    cfg: ETS2026Config = ETS2026Config(),
) -> dict:
    """Run the (notebook-derived) EXACT pipeline end-to-end.

    Outputs
    -------
    A dictionary with key artifacts:
      - 'setup_A', 'setup_B': loaded telemetry DataFrames
      - 'shared_features': list[str]
      - 'model_A', 'model_B': fitted CINTAS models
      - 'results_A', 'results_B': evaluation fold-level DataFrames
      - 'ranks_A', 'ranks_B': feature rank DataFrames
    """
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    repo_root = REPO_ROOT
    progress = progress_start("ETS baseline", f"out={_safe_relative(out_root, REPO_ROOT)} windows={cfg.window_sizes} splits={cfg.n_splits}")

    # ------------------------------------------------------------------
    # Load
    # ------------------------------------------------------------------
    progress_update("ETS baseline", "loading telemetry", start=progress, detail=_safe_relative(Path(data_root), REPO_ROOT))
    df_A, df_B = load_telemetry_two_setups(Path(data_root))
    progress_update("ETS baseline", "loaded telemetry", start=progress, detail=f"A rows={len(df_A):,}, B rows={len(df_B):,}")

    # ------------------------------------------------------------------
    # Clean + debias (matches EXACT.ipynb)
    # ------------------------------------------------------------------
    progress_update("ETS baseline", "cleaning and debiasing", start=progress)
    df_A_z = clean_and_debias_telemetry(df_A)
    df_B_z = clean_and_debias_telemetry(df_B)

    feat_A = drop_constant_features(df_A_z, get_feature_columns(df_A_z))
    feat_B = drop_constant_features(df_B_z, get_feature_columns(df_B_z))
    shared_features = sorted(set(feat_A) & set(feat_B))
    progress_update("ETS baseline", "shared features", start=progress, detail=f"count={len(shared_features)}")

    if not shared_features:
        raise RuntimeError("No shared numeric telemetry features between Setup A and Setup B.")

    # ------------------------------------------------------------------
    # Fit CINTAS from BENIGN (same feature list for A and B)
    # ------------------------------------------------------------------
    progress_update("ETS baseline", "fitting CINTAS models", start=progress, detail=f"lambda={cfg.lambda_res}")
    model_A = fit_cintas_from_benign(df_A_z, shared_features, lambda_res=cfg.lambda_res)
    model_B = fit_cintas_from_benign(df_B_z, shared_features, lambda_res=cfg.lambda_res)

    # Attach per-sample score columns with concat to avoid pandas frame.insert
    # fragmentation warnings on wide telemetry tables.
    score_A, _, _ = model_A.score_dataframe(df_A_z)
    score_B, _, _ = model_B.score_dataframe(df_B_z)
    df_A_z = pd.concat([df_A_z.copy(), pd.Series(score_A, index=df_A_z.index, name="cias_sample_score")], axis=1).copy()
    df_B_z = pd.concat([df_B_z.copy(), pd.Series(score_B, index=df_B_z.index, name="cias_sample_score")], axis=1).copy()

    # ------------------------------------------------------------------
    # Offline stable conditional telemetry graph + hardware-aware feature ranking
    # ------------------------------------------------------------------
    progress_update("ETS baseline", "building causal ranks", start=progress, detail="Setup A")
    nodes_A, edges_A, ranks_A = build_causal_and_rank_features_for_setup(
        setup="A",
        df=df_A_z,
        feature_cols=shared_features,
        out_root=out_root,
        corr_threshold=cfg.corr_threshold,
        top_k_plot=20,
        score_col="cias_sample_score",
    )
    progress_update("ETS baseline", "building causal ranks", start=progress, detail="Setup B")
    nodes_B, edges_B, ranks_B = build_causal_and_rank_features_for_setup(
        setup="B",
        df=df_B_z,
        feature_cols=shared_features,
        out_root=out_root,
        corr_threshold=cfg.corr_threshold,
        top_k_plot=20,
        score_col="cias_sample_score",
    )

    # Export edges in Fig.5-compatible format (optional convenience)
    export_edges_for_fig5(edges_A, out_root / "fig5_inputs" / "setupA_edges.csv")
    export_edges_for_fig5(edges_B, out_root / "fig5_inputs" / "setupB_edges.csv")

    # Fig. 6-style feature importance plot (from rank CSVs)
    plot_topk_features_two_panel(
        ranks_csv_A=out_root / "causal" / "SETUP_A_feature_ranks.csv",
        ranks_csv_B=out_root / "causal" / "SETUP_B_feature_ranks.csv",
        out_png=out_root / "figures" / "fig6_top15_features.png",
        top_k=cfg.top_k_features,
    )

    # ------------------------------------------------------------------
    # Evaluation (fold-level)
    # ------------------------------------------------------------------
    progress_update("ETS baseline", "evaluating decision windows", start=progress, detail="Setup A")
    results_A = run_exact_eval_for_setup(
        setup="A",
        df=df_A_z,
        model=model_A,
        scenarios_eval=cfg.scenarios_eval,
        window_sizes=cfg.window_sizes,
        n_splits_list=(cfg.n_splits,),
        default_p_quantile=cfg.p_quantile,
        agg_mode=cfg.agg_mode,
        droop_cfg=None,
        seed=cfg.seed,
    )
    progress_update("ETS baseline", "evaluating decision windows", start=progress, detail="Setup B")
    results_B = run_exact_eval_for_setup(
        setup="B",
        df=df_B_z,
        model=model_B,
        scenarios_eval=cfg.scenarios_eval,
        window_sizes=cfg.window_sizes,
        n_splits_list=(cfg.n_splits,),
        default_p_quantile=cfg.p_quantile,
        agg_mode=cfg.agg_mode,
        droop_cfg=None,
        seed=cfg.seed,
    )

    progress_update("ETS baseline", "writing summaries and figures", start=progress)
    # Save summaries (these CSV names match EXACT.ipynb plots)
    if not results_A.empty:
        save_exact_summaries_for_setup("A", results_A, out_root)
    if not results_B.empty:
        save_exact_summaries_for_setup("B", results_B, out_root)

    # Plot metrics-vs-window-size from the per-workload CSVs
    per_wl_A = out_root / "SETUP_A_EXACT_summary_per_workload.csv"
    per_wl_B = out_root / "SETUP_B_EXACT_summary_per_workload.csv"
    if per_wl_A.exists() and per_wl_B.exists():
        plot_metrics_vs_window_size(
            per_workload_csv_A=per_wl_A,
            per_workload_csv_B=per_wl_B,
            out_png=out_root / "figures" / "fig4_metrics_vs_window_size.png",
            n_splits=cfg.n_splits,
        )

    artifact_paths = (
        out_root / "SETUP_A_EXACT_summary_global.csv",
        out_root / "SETUP_A_EXACT_summary_per_workload.csv",
        out_root / "SETUP_B_EXACT_summary_global.csv",
        out_root / "SETUP_B_EXACT_summary_per_workload.csv",
        out_root / "causal" / "SETUP_A_feature_ranks.csv",
        out_root / "causal" / "SETUP_B_feature_ranks.csv",
        out_root / "fig5_inputs" / "setupA_edges.csv",
        out_root / "fig5_inputs" / "setupB_edges.csv",
        out_root / "figures" / "fig4_metrics_vs_window_size.png",
        out_root / "figures" / "fig6_top15_features.png",
    )
    progress_update("ETS baseline", "writing manifest", start=progress)
    manifest_path = write_run_manifest(
        out_root,
        repo_root=repo_root,
        data_root=Path(data_root),
        cfg=cfg,
        seed=cfg.seed,
        artifact_paths=artifact_paths,
    )

    progress_update("ETS baseline", "DONE", start=progress, detail=_safe_relative(manifest_path, REPO_ROOT))
    return {
        "setup_A": df_A_z,
        "setup_B": df_B_z,
        "shared_features": shared_features,
        "model_A": model_A,
        "model_B": model_B,
        "results_A": results_A,
        "results_B": results_B,
        "ranks_A": ranks_A,
        "ranks_B": ranks_B,
        "run_manifest": manifest_path,
    }


# ---- CITADEL/TCAD ablation, now owned by this notebook ----




@dataclass(frozen=True)
class TCAD2026Config:
    scenarios_eval: tuple[str, ...] = ("DROOP", "RH", "SPECTRE")
    feature_budgets: tuple[int, ...] = (8, 15)
    window_sizes: tuple[int, ...] = (50, 100)
    lambda_res_values: tuple[float, ...] = (0.25, 0.5)
    agg_modes: tuple[str, ...] = ("max",)
    weight_modes: tuple[str, ...] = ("uniform",)
    fixed_point_q: tuple[int, ...] = (12, 15)
    n_splits: int = 3
    p_quantile: float = 0.99
    corr_threshold: float = 0.35
    graph_bootstraps: int = 8
    graph_subsample_frac: float = 0.70
    graph_stability_threshold: float = 0.50
    seed: int = 123

    @classmethod
    def from_json(cls, path: Path) -> "TCAD2026Config":
        data = json.loads(Path(path).read_text(encoding="utf-8"))
        return cls(
            scenarios_eval=tuple(data.get("scenarios_eval", cls.scenarios_eval)),
            feature_budgets=tuple(int(x) for x in data.get("feature_budgets", cls.feature_budgets)),
            window_sizes=tuple(int(x) for x in data.get("window_sizes", cls.window_sizes)),
            lambda_res_values=tuple(float(x) for x in data.get("lambda_res_values", cls.lambda_res_values)),
            agg_modes=tuple(data.get("agg_modes", cls.agg_modes)),
            weight_modes=tuple(data.get("weight_modes", cls.weight_modes)),
            fixed_point_q=tuple(int(x) for x in data.get("fixed_point_q", cls.fixed_point_q)),
            n_splits=int(data.get("n_splits", cls.n_splits)),
            p_quantile=float(data.get("p_quantile", cls.p_quantile)),
            corr_threshold=float(data.get("corr_threshold", cls.corr_threshold)),
            graph_bootstraps=int(data.get("graph_bootstraps", cls.graph_bootstraps)),
            graph_subsample_frac=float(data.get("graph_subsample_frac", cls.graph_subsample_frac)),
            graph_stability_threshold=float(data.get("graph_stability_threshold", cls.graph_stability_threshold)),
            seed=int(data.get("seed", cls.seed)),
        )


def _selected_features(ranks: pd.DataFrame, k: int, fallback: Iterable[str]) -> list[str]:
    if ranks is not None and not ranks.empty and "feature" in ranks.columns:
        feats = [str(x) for x in ranks.head(int(k))["feature"].tolist()]
        if feats:
            return feats
    return list(fallback)[: int(k)]


def _fixed_point_error(df: pd.DataFrame, model, q: int, *, max_rows: int = 5000) -> dict[str, float]:
    if df.empty:
        return {"fp_mae": 0.0, "fp_max_abs": 0.0}
    subset = df.head(max_rows)
    float_score, _, _ = model.score_dataframe(subset)
    fixed = FixedPointCINTAS.from_float_model(model, FixedPointConfig(q=int(q)))
    fixed_score = fixed.score_dataframe(subset).astype(float) / float(fixed.cfg.scale)
    diff = np.abs(float_score - fixed_score)
    return {
        "fp_mae": float(np.mean(diff)) if diff.size else 0.0,
        "fp_max_abs": float(np.max(diff)) if diff.size else 0.0,
    }


def _summarize_fold_results(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    global_df = df[df["workload"] == "ALL"].copy()
    if global_df.empty:
        global_df = df.copy()
    group_cols = [
        "setup",
        "scenario",
        "window_size",
        "agg_mode",
        "lambda_res",
        "p_quantile",
        "top_k",
        "weight_mode",
        "fixed_point_q",
        "n_selected_features",
    ]
    metric_cols = [
        "auc_roc",
        "auc_pr",
        "f1",
        "bal_acc",
        "mcc",
        "fpr",
        "brier",
        "ece",
        "fp_mae",
        "fp_max_abs",
        "hw_frequency_ghz",
        "hw_operator_bit_width",
        "hw_std_area_mm2",
        "hw_agg_area_mm2",
        "hw_area_mm2",
        "hw_power_mw",
        "hw_setup_b_area_overhead_pct",
        "hw_idle_power_overhead_pct",
        "hw_median_workload_power_overhead_pct",
        "hw_add_count",
        "hw_mult_count",
        "hw_estimated_serial_cycles",
        "hw_add_delay_ps",
        "hw_mult_delay_ps",
    ]
    available_metrics = [c for c in metric_cols if c in global_df.columns]
    return (
        global_df.groupby(group_cols, as_index=False)[available_metrics]
        .mean()
        .sort_values(group_cols)
        .reset_index(drop=True)
    )


def run_tcad_ablation(
    *,
    data_root: Path,
    out_root: Path,
    cfg: TCAD2026Config = TCAD2026Config(),
) -> dict[str, Path | pd.DataFrame]:
    """Run the TCAD extension ablation grid.

    This is deliberately a reference orchestration layer. It reuses the ETS
    pipeline components, then varies the journal-specific design axes:
    feature budget, aggregation, decision-block length, lambda, weighting, and
    fixed-point precision.
    """
    data_root = Path(data_root)
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    repo_root = REPO_ROOT
    grid_total = len(("A", "B")) * len(cfg.feature_budgets) * len(cfg.lambda_res_values) * len(cfg.agg_modes) * len(cfg.weight_modes) * len(cfg.fixed_point_q)
    progress = progress_start("TCAD ablation", f"out={_safe_relative(out_root, REPO_ROOT)} configs={grid_total} scenarios={cfg.scenarios_eval} windows={cfg.window_sizes}")

    progress_update("TCAD ablation", "loading telemetry", start=progress, detail=_safe_relative(data_root, REPO_ROOT))
    df_a, df_b = load_telemetry_two_setups(data_root)
    progress_update("TCAD ablation", "loaded telemetry", start=progress, detail=f"A rows={len(df_a):,}, B rows={len(df_b):,}")
    df_by_setup = {
        "A": clean_and_debias_telemetry(df_a),
        "B": clean_and_debias_telemetry(df_b),
    }

    feat_a = drop_constant_features(df_by_setup["A"], get_feature_columns(df_by_setup["A"]))
    feat_b = drop_constant_features(df_by_setup["B"], get_feature_columns(df_by_setup["B"]))
    shared_features = sorted(set(feat_a) & set(feat_b))
    if not shared_features:
        raise RuntimeError("No shared numeric telemetry features between Setup A and Setup B.")
    progress_update("TCAD ablation", "shared features", start=progress, detail=f"count={len(shared_features)}")

    ranks_by_setup: dict[str, pd.DataFrame] = {}
    for setup, df_setup in df_by_setup.items():
        progress_update("TCAD ablation", "building feature ranks", start=progress, detail=f"Setup {setup}")
        base_model = fit_cintas_from_benign(
            df_setup,
            shared_features,
            lambda_res=0.5,
            weight_mode="uniform",
        )
        score, _, _ = base_model.score_dataframe(df_setup)
        score_series = pd.Series(score, index=df_setup.index, name="cias_sample_score")
        df_setup = pd.concat([df_setup.copy(), score_series], axis=1).copy()
        df_by_setup[setup] = df_setup
        _, _, ranks = build_causal_and_rank_features_for_setup(
            setup=setup,
            df=df_setup,
            feature_cols=shared_features,
            out_root=out_root,
            corr_threshold=cfg.corr_threshold,
            top_k_plot=max(cfg.feature_budgets),
            graph_bootstraps=cfg.graph_bootstraps,
            graph_subsample_frac=cfg.graph_subsample_frac,
            graph_stability_threshold=cfg.graph_stability_threshold,
            seed=cfg.seed,
            score_col="cias_sample_score",
        )
        ranks_by_setup[setup] = ranks

    fold_frames: list[pd.DataFrame] = []
    selected_rows: list[dict] = []
    grid_idx = 0

    for setup, df_setup in df_by_setup.items():
        for top_k, lambda_res, agg_mode, weight_mode, q in product(
            cfg.feature_budgets,
            cfg.lambda_res_values,
            cfg.agg_modes,
            cfg.weight_modes,
            cfg.fixed_point_q,
        ):
            grid_idx += 1
            if progress_should_log(grid_idx, grid_total):
                progress_update("TCAD ablation", "evaluating grid", start=progress, current=grid_idx, total=grid_total, detail=f"setup={setup} k={top_k} lambda={lambda_res} agg={agg_mode} weight={weight_mode} q={q}")
            feats = _selected_features(ranks_by_setup[setup], int(top_k), shared_features)
            hw_cost = estimate_cintas_hardware_cost(feature_names=feats, frequency_ghz=1.0)
            model = fit_cintas_from_benign(
                df_setup,
                feats,
                lambda_res=float(lambda_res),
                weight_mode=str(weight_mode),
            )
            fp_err = _fixed_point_error(df_setup, model, int(q))
            selected_rows.append({
                "setup": setup,
                "top_k": int(top_k),
                "lambda_res": float(lambda_res),
                "agg_mode": str(agg_mode),
                "weight_mode": str(weight_mode),
                "fixed_point_q": int(q),
                "n_selected_features": int(hw_cost.feature_count),
                "feature_group_counts": format_feature_group_counts(hw_cost.group_feature_counts),
                "features": ",".join(feats),
            })

            eval_df = run_exact_eval_for_setup(
                setup=setup,
                df=df_setup,
                model=model,
                scenarios_eval=cfg.scenarios_eval,
                window_sizes=cfg.window_sizes,
                n_splits_list=(cfg.n_splits,),
                default_p_quantile=cfg.p_quantile,
                agg_mode=str(agg_mode),
                droop_cfg=None,
                seed=cfg.seed,
            )
            if eval_df.empty:
                continue
            eval_df["top_k"] = int(top_k)
            eval_df["n_selected_features"] = int(hw_cost.feature_count)
            eval_df["weight_mode"] = str(weight_mode)
            eval_df["fixed_point_q"] = int(q)
            eval_df["fp_mae"] = fp_err["fp_mae"]
            eval_df["fp_max_abs"] = fp_err["fp_max_abs"]
            for key, value in hw_cost.to_summary_dict().items():
                eval_df[key] = value
            fold_frames.append(eval_df)

    progress_update("TCAD ablation", "summarizing fold results", start=progress, detail=f"fold_frames={len(fold_frames)}")
    fold_results = pd.concat(fold_frames, ignore_index=True) if fold_frames else pd.DataFrame()
    summary = _summarize_fold_results(fold_results)

    fold_path = out_root / "tcad_ablation_fold_results.csv"
    summary_path = out_root / "tcad_ablation_summary.csv"
    selected_path = out_root / "tcad_selected_features.csv"
    config_path = out_root / "tcad_config_resolved.json"

    progress_update("TCAD ablation", "writing CSV artifacts", start=progress)
    fold_results.to_csv(fold_path, index=False)
    summary.to_csv(summary_path, index=False)
    pd.DataFrame(selected_rows).to_csv(selected_path, index=False)
    config_path.write_text(json.dumps(asdict(cfg), indent=2, sort_keys=True), encoding="utf-8")

    progress_update("TCAD ablation", "writing manifest", start=progress)
    manifest_path = write_run_manifest(
        out_root,
        repo_root=repo_root,
        data_root=data_root,
        cfg=cfg,
        seed=cfg.seed,
        artifact_paths=(
            fold_path,
            summary_path,
            selected_path,
            config_path,
            out_root / "causal" / "SETUP_A_feature_ranks.csv",
            out_root / "causal" / "SETUP_B_feature_ranks.csv",
        ),
    )

    progress_update("TCAD ablation", "DONE", start=progress, detail=_safe_relative(summary_path, REPO_ROOT))
    return {
        "fold_results": fold_results,
        "summary": summary,
        "fold_path": fold_path,
        "summary_path": summary_path,
        "selected_path": selected_path,
        "manifest_path": manifest_path,
    }


# ---- Lifecycle drift and benign recalibration ----


@dataclass(frozen=True)
class LifecycleDriftConfig:
    scenarios_eval: tuple[str, ...] = ("DROOP", "RH", "SPECTRE")
    top_k: int = 15
    window_size: int = 100
    lambda_res: float = 0.5
    agg_mode: str = "max"
    weight_mode: str = "uniform"
    p_quantile: float = 0.99
    calibration_frac: float = 0.60
    corr_threshold: float = 0.35
    seed: int = 123

    @classmethod
    def from_tcad(cls, cfg: TCAD2026Config) -> "LifecycleDriftConfig":
        scenarios = tuple(s for s in cfg.scenarios_eval if str(s).upper() in {"DROOP", "RH", "SPECTRE"})
        return cls(
            scenarios_eval=scenarios or cls.scenarios_eval,
            top_k=15 if 15 in cfg.feature_budgets else int(cfg.feature_budgets[0]),
            window_size=100 if 100 in cfg.window_sizes else int(cfg.window_sizes[0]),
            lambda_res=0.5 if 0.5 in cfg.lambda_res_values else float(cfg.lambda_res_values[0]),
            agg_mode="max" if "max" in cfg.agg_modes else str(cfg.agg_modes[0]),
            weight_mode="uniform" if "uniform" in cfg.weight_modes else str(cfg.weight_modes[0]),
            p_quantile=float(cfg.p_quantile),
            corr_threshold=float(cfg.corr_threshold),
            seed=int(cfg.seed),
        )


def notebook_lifecycle_config(cfg: TCAD2026Config | None = None) -> LifecycleDriftConfig:
    return LifecycleDriftConfig.from_tcad(cfg if cfg is not None else notebook_tcad_config(TCAD_PRESET, seed=SEED))


def _split_benign_epochs(df: pd.DataFrame, *, calibration_frac: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    ben = df[df["scenario"].astype(str).str.upper() == "BENIGN"].copy()
    if ben.empty:
        return ben.copy(), ben.copy()
    ben = ben.sort_values(["workload", "time_idx"]).reset_index(drop=True)
    counts = ben.groupby("workload")["workload"].transform("size").to_numpy(dtype=int)
    pos = ben.groupby("workload").cumcount().to_numpy(dtype=int)
    raw_cut = np.floor(counts * float(calibration_frac)).astype(int)
    cut = np.where(counts >= 2, np.clip(raw_cut, 1, counts - 1), counts)
    mask = pos < cut
    initial = ben.loc[mask].copy()
    later = ben.loc[~mask].copy()
    initial["lifecycle_epoch"] = "initial_calibration"
    later["lifecycle_epoch"] = "later_benign"
    return initial, later


def _rank_features_from_benign_only(*, setup: str, df_benign: pd.DataFrame, feature_cols: list[str], out_root: Path, corr_threshold: float, tag: str) -> pd.DataFrame:
    feats = [f for f in feature_cols if f in df_benign.columns]
    if not feats:
        return pd.DataFrame(columns=["feature", "domain", "graph_centrality", "edge_stability", "conditional_dependence", "importance_score"])
    edges_df, nodes_df = learn_stability_controlled_conditional_graph(
        df_benign,
        feats,
        corr_threshold=corr_threshold,
        n_bootstraps=6,
        subsample_frac=0.70,
        stability_threshold=0.50,
        seed=123 + (0 if str(setup).upper() == "A" else 1009),
    )
    cost = nodes_df["telemetry_cost"].to_numpy(dtype=float) if "telemetry_cost" in nodes_df.columns else np.ones(len(nodes_df))
    cost_norm = cost / max(float(np.max(cost)), 1e-8) if len(cost) else cost
    raw = (
        0.45 * nodes_df.get("graph_centrality", 0.0).to_numpy(dtype=float)
        + 0.35 * nodes_df.get("edge_stability", 0.0).to_numpy(dtype=float)
        + 0.20 * nodes_df.get("conditional_dependence", 0.0).to_numpy(dtype=float)
    ) / np.maximum(cost_norm, 1e-8)
    nodes_df["hardware_cost_penalty"] = cost_norm
    nodes_df["importance_score"] = raw / max(float(np.max(raw)), 1e-8) if len(raw) else []
    ranks_df = nodes_df.sort_values(["importance_score", "edge_stability", "feature"], ascending=[False, False, True], kind="mergesort").reset_index(drop=True)
    causal_dir = Path(out_root) / "lifecycle_causal"
    causal_dir.mkdir(parents=True, exist_ok=True)
    setup_u = str(setup).upper()
    safe_tag = str(tag).replace("/", "_")
    edges_df.to_csv(causal_dir / f"SETUP_{setup_u}_{safe_tag}_edges.csv", index=False)
    nodes_df.to_csv(causal_dir / f"SETUP_{setup_u}_{safe_tag}_nodes.csv", index=False)
    ranks_df.to_csv(causal_dir / f"SETUP_{setup_u}_{safe_tag}_feature_ranks.csv", index=False)
    return ranks_df

def _aggregate_epoch_scores(scores: np.ndarray, *, window_size: int, agg_mode: str) -> np.ndarray:
    scores = np.asarray(scores, dtype=float)
    n_win = scores.size // int(window_size)
    out: list[float] = []
    for i in range(n_win):
        block = scores[i * int(window_size):(i + 1) * int(window_size)]
        if agg_mode == "mean":
            out.append(float(np.mean(block)))
        elif agg_mode == "median":
            out.append(float(np.median(block)))
        else:
            out.append(float(np.max(block)))
    return np.asarray(out, dtype=float)


def _window_scores_for_epoch(df: pd.DataFrame, model: CINTASModel, *, phase: str, label: int, window_size: int, agg_mode: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    tmp = df.copy()
    if "time_idx" not in tmp.columns:
        tmp["time_idx"] = tmp.groupby(["scenario", "workload"]).cumcount()
    tmp = tmp.sort_values(["workload", "scenario", "time_idx"]).reset_index(drop=True)
    tmp["score_sample"] = score_samples(tmp, model)
    rows: list[dict[str, Any]] = []
    setup = str(tmp["setup"].iloc[0]) if "setup" in tmp.columns and len(tmp) else "UNKNOWN"
    for (scenario, workload), group in tmp.groupby(["scenario", "workload"], sort=True):
        win_scores = _aggregate_epoch_scores(group["score_sample"].to_numpy(dtype=float), window_size=window_size, agg_mode=agg_mode)
        for idx, score in enumerate(win_scores):
            rows.append({
                "setup": setup,
                "phase": str(phase),
                "scenario": str(scenario).upper(),
                "workload": str(workload),
                "window_idx": int(idx),
                "window_size": int(window_size),
                "agg_mode": str(agg_mode),
                "score_win": float(score),
                "label": int(label),
            })
    return pd.DataFrame(rows)


def _jaccard(a: Sequence[str], b: Sequence[str]) -> float:
    sa, sb = set(a), set(b)
    return float(len(sa & sb) / len(sa | sb)) if (sa or sb) else 1.0


def _pseudo_probs_from_reference(scores: np.ndarray, reference_scores: np.ndarray) -> np.ndarray:
    scores = np.asarray(scores, dtype=float)
    ref = np.asarray(reference_scores, dtype=float)
    if ref.size == 0:
        ref = scores
    if ref.size == 0:
        return np.asarray([], dtype=float)
    s_min, s_max = float(np.min(ref)), float(np.max(ref))
    denom = (s_max - s_min) if s_max > s_min else 1.0
    return np.clip((scores - s_min) / denom, 0.0, 1.0)


def _summarize_lifecycle_windows(windows: pd.DataFrame, *, method: str, threshold: float, reference_scores: np.ndarray, selected_features: Sequence[str], recalibrated_features: Sequence[str], rank_jaccard: float, cfg: LifecycleDriftConfig, calibration_mean: float) -> tuple[dict[str, Any], pd.DataFrame]:
    if windows.empty:
        return {"method": method, "fpr": float("nan"), "tpr": float("nan"), "n_eval_windows": 0}, windows.copy()
    out = windows.copy()
    scores = out["score_win"].to_numpy(dtype=float)
    labels = out["label"].to_numpy(dtype=int)
    pred = (scores >= float(threshold)).astype(int)
    prob = _pseudo_probs_from_reference(scores, reference_scores)
    metrics = compute_binary_metrics(labels, pred, prob)
    benign_mask = labels == 0
    anomaly_mask = labels == 1
    tpr = float(np.mean(pred[anomaly_mask] == 1)) if np.any(anomaly_mask) else float("nan")
    later_benign_mean = float(np.mean(scores[benign_mask])) if np.any(benign_mask) else float("nan")
    out["method"] = method
    out["threshold"] = float(threshold)
    out["pred"] = pred
    out["prob"] = prob
    row = {
        "setup": str(out["setup"].iloc[0]) if "setup" in out.columns and len(out) else "UNKNOWN",
        "method": method,
        "window_size": int(cfg.window_size),
        "agg_mode": cfg.agg_mode,
        "lambda_res": float(cfg.lambda_res),
        "p_quantile": float(cfg.p_quantile),
        "top_k": int(cfg.top_k),
        "weight_mode": cfg.weight_mode,
        "threshold": float(threshold),
        "n_initial_features": int(len(selected_features)),
        "n_recalibrated_features": int(len(recalibrated_features)),
        "rank_jaccard": float(rank_jaccard),
        "n_eval_windows": int(labels.size),
        "n_benign_windows": int(np.sum(benign_mask)),
        "n_anomaly_windows": int(np.sum(anomaly_mask)),
        "tpr": tpr,
        "calibration_benign_score_mean": float(calibration_mean),
        "later_benign_score_mean": later_benign_mean,
        "benign_score_mean_delta": later_benign_mean - float(calibration_mean) if np.isfinite(later_benign_mean) else float("nan"),
        "selected_features": ",".join(selected_features),
        "recalibrated_features": ",".join(recalibrated_features),
    }
    row.update(metrics)
    return row, out


def _scenario_lifecycle_rates(windows: pd.DataFrame) -> pd.DataFrame:
    if windows.empty:
        return pd.DataFrame()
    rows: list[dict[str, Any]] = []
    for (setup, method, scenario), group in windows.groupby(["setup", "method", "scenario"], sort=True):
        labels = group["label"].to_numpy(dtype=int)
        pred = group["pred"].to_numpy(dtype=int)
        benign_mask = labels == 0
        anomaly_mask = labels == 1
        rows.append({
            "setup": setup,
            "method": method,
            "scenario": scenario,
            "n_windows": int(len(group)),
            "fpr": float(np.mean(pred[benign_mask] == 1)) if np.any(benign_mask) else float("nan"),
            "tpr": float(np.mean(pred[anomaly_mask] == 1)) if np.any(anomaly_mask) else float("nan"),
            "score_mean": float(group["score_win"].mean()),
        })
    return pd.DataFrame(rows)


def plot_lifecycle_recalibration(summary: pd.DataFrame, out_png: Path) -> Path | None:
    if summary.empty or not {"setup", "method", "fpr", "tpr"}.issubset(summary.columns):
        return None
    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    methods = [m for m in ["frozen", "threshold_only", "full_feature_rank"] if m in set(summary["method"])]
    setups = sorted(summary["setup"].astype(str).unique())
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for ax, metric, label in zip(axes, ["fpr", "tpr"], ["False-positive rate", "Anomaly true-positive rate"]):
        x = np.arange(len(setups), dtype=float)
        width = 0.24
        for offset, method in enumerate(methods):
            vals = []
            for setup in setups:
                sub = summary[(summary["setup"].astype(str) == setup) & (summary["method"] == method)]
                vals.append(float(sub[metric].iloc[0]) if not sub.empty else np.nan)
            ax.bar(x + (offset - (len(methods) - 1) / 2) * width, vals, width=width, label=method.replace("_", " "))
        ax.set_xticks(x)
        ax.set_xticklabels(setups)
        ax.set_ylabel(label)
        ax.set_ylim(0.0, 1.0)
        ax.grid(axis="y", alpha=0.25)
    axes[0].legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return out_png


def run_lifecycle_drift_recalibration(*, data_root: Path, out_root: Path, cfg: LifecycleDriftConfig) -> dict[str, Path | pd.DataFrame]:
    data_root = Path(data_root)
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    progress = progress_start("Lifecycle drift", f"out={_safe_relative(out_root, REPO_ROOT)} top_k={cfg.top_k} window={cfg.window_size} scenarios={cfg.scenarios_eval}")
    progress_update("Lifecycle drift", "loading telemetry", start=progress, detail=_safe_relative(data_root, REPO_ROOT))
    df_a, df_b = load_telemetry_two_setups(data_root)
    df_by_setup = {"A": clean_and_debias_telemetry(df_a), "B": clean_and_debias_telemetry(df_b)}
    feat_a = drop_constant_features(df_by_setup["A"], get_feature_columns(df_by_setup["A"]))
    feat_b = drop_constant_features(df_by_setup["B"], get_feature_columns(df_by_setup["B"]))
    shared_features = sorted(set(feat_a) & set(feat_b))
    if not shared_features:
        raise RuntimeError("No shared numeric telemetry features between Setup A and Setup B.")
    progress_update("Lifecycle drift", "shared features", start=progress, detail=f"count={len(shared_features)}")

    summary_rows: list[dict[str, Any]] = []
    window_frames: list[pd.DataFrame] = []
    rank_rows: list[dict[str, Any]] = []

    setup_total = len(df_by_setup)
    for setup_idx, (setup, df_setup) in enumerate(df_by_setup.items(), start=1):
        progress_update("Lifecycle drift", "setup start", start=progress, current=setup_idx, total=setup_total, detail=f"Setup {setup}")
        initial_benign, later_benign = _split_benign_epochs(df_setup, calibration_frac=cfg.calibration_frac)
        if initial_benign.empty or later_benign.empty:
            progress_update("Lifecycle drift", "setup skipped", start=progress, detail=f"Setup {setup}: insufficient benign epochs")
            continue
        progress_update("Lifecycle drift", "epoch split", start=progress, detail=f"Setup {setup}: initial={len(initial_benign):,}, later={len(later_benign):,}")
        initial_ranks = _rank_features_from_benign_only(setup=setup, df_benign=initial_benign, feature_cols=shared_features, out_root=out_root, corr_threshold=cfg.corr_threshold, tag="initial_benign")
        initial_features = _selected_features(initial_ranks, cfg.top_k, shared_features)
        frozen_model = fit_cintas_from_benign(initial_benign, initial_features, lambda_res=cfg.lambda_res, weight_mode=cfg.weight_mode)
        calibration_windows = _window_scores_for_epoch(initial_benign, frozen_model, phase="initial_calibration", label=0, window_size=cfg.window_size, agg_mode=cfg.agg_mode)
        later_benign_frozen = _window_scores_for_epoch(later_benign, frozen_model, phase="later_benign", label=0, window_size=cfg.window_size, agg_mode=cfg.agg_mode)
        if calibration_windows.empty or later_benign_frozen.empty:
            continue
        anomaly_frozen_frames = []
        for scenario_idx, scenario in enumerate(cfg.scenarios_eval, start=1):
            progress_update("Lifecycle drift", "frozen anomaly windows", start=progress, current=scenario_idx, total=len(cfg.scenarios_eval), detail=f"Setup {setup} scenario={scenario}")
            df_anom = df_setup[df_setup["scenario"].astype(str).str.upper() == str(scenario).upper()].copy()
            anomaly_frozen_frames.append(_window_scores_for_epoch(df_anom, frozen_model, phase="anomaly_eval", label=1, window_size=cfg.window_size, agg_mode=cfg.agg_mode))
        anomaly_frozen = pd.concat([x for x in anomaly_frozen_frames if not x.empty], ignore_index=True) if anomaly_frozen_frames else pd.DataFrame()
        eval_frozen = pd.concat([later_benign_frozen, anomaly_frozen], ignore_index=True)
        calibration_scores = calibration_windows["score_win"].to_numpy(dtype=float)
        frozen_threshold = float(np.quantile(calibration_scores, cfg.p_quantile))
        calibration_mean = float(np.mean(calibration_scores))

        row, windows = _summarize_lifecycle_windows(eval_frozen, method="frozen", threshold=frozen_threshold, reference_scores=calibration_scores, selected_features=initial_features, recalibrated_features=initial_features, rank_jaccard=1.0, cfg=cfg, calibration_mean=calibration_mean)
        summary_rows.append(row)
        window_frames.append(windows)

        later_scores = later_benign_frozen["score_win"].to_numpy(dtype=float)
        threshold_only = float(np.quantile(later_scores, cfg.p_quantile))
        row, windows = _summarize_lifecycle_windows(eval_frozen, method="threshold_only", threshold=threshold_only, reference_scores=later_scores, selected_features=initial_features, recalibrated_features=initial_features, rank_jaccard=1.0, cfg=cfg, calibration_mean=calibration_mean)
        summary_rows.append(row)
        window_frames.append(windows)

        recal_ranks = _rank_features_from_benign_only(setup=setup, df_benign=later_benign, feature_cols=shared_features, out_root=out_root, corr_threshold=cfg.corr_threshold, tag="later_benign")
        recal_features = _selected_features(recal_ranks, cfg.top_k, shared_features)
        recal_model = fit_cintas_from_benign(later_benign, recal_features, lambda_res=cfg.lambda_res, weight_mode=cfg.weight_mode)
        later_benign_recal = _window_scores_for_epoch(later_benign, recal_model, phase="later_benign", label=0, window_size=cfg.window_size, agg_mode=cfg.agg_mode)
        anomaly_recal_frames = []
        for scenario_idx, scenario in enumerate(cfg.scenarios_eval, start=1):
            progress_update("Lifecycle drift", "recalibrated anomaly windows", start=progress, current=scenario_idx, total=len(cfg.scenarios_eval), detail=f"Setup {setup} scenario={scenario}")
            df_anom = df_setup[df_setup["scenario"].astype(str).str.upper() == str(scenario).upper()].copy()
            anomaly_recal_frames.append(_window_scores_for_epoch(df_anom, recal_model, phase="anomaly_eval", label=1, window_size=cfg.window_size, agg_mode=cfg.agg_mode))
        anomaly_recal = pd.concat([x for x in anomaly_recal_frames if not x.empty], ignore_index=True) if anomaly_recal_frames else pd.DataFrame()
        if later_benign_recal.empty:
            continue
        recal_scores = later_benign_recal["score_win"].to_numpy(dtype=float)
        recal_threshold = float(np.quantile(recal_scores, cfg.p_quantile))
        eval_recal = pd.concat([later_benign_recal, anomaly_recal], ignore_index=True)
        feature_jaccard = _jaccard(initial_features, recal_features)
        row, windows = _summarize_lifecycle_windows(eval_recal, method="full_feature_rank", threshold=recal_threshold, reference_scores=recal_scores, selected_features=initial_features, recalibrated_features=recal_features, rank_jaccard=feature_jaccard, cfg=cfg, calibration_mean=calibration_mean)
        summary_rows.append(row)
        window_frames.append(windows)
        rank_rows.append({"setup": setup, "top_k": int(cfg.top_k), "initial_features": ",".join(initial_features), "recalibrated_features": ",".join(recal_features), "rank_jaccard": float(feature_jaccard)})

    progress_update("Lifecycle drift", "summarizing", start=progress, detail=f"summary_rows={len(summary_rows)}")
    summary = pd.DataFrame(summary_rows)
    windows_all = pd.concat(window_frames, ignore_index=True) if window_frames else pd.DataFrame()
    scenario_rates = _scenario_lifecycle_rates(windows_all)
    ranks = pd.DataFrame(rank_rows)
    summary_path = out_root / "lifecycle_recalibration_summary.csv"
    windows_path = out_root / "lifecycle_recalibration_windows.csv"
    scenario_path = out_root / "lifecycle_recalibration_by_scenario.csv"
    ranks_path = out_root / "lifecycle_feature_rank_stability.csv"
    config_path = out_root / "lifecycle_config_resolved.json"
    figure_path = out_root / "figures" / "fig_lifecycle_recalibration_fpr_tpr.png"
    progress_update("Lifecycle drift", "writing CSV artifacts", start=progress)
    summary.to_csv(summary_path, index=False)
    windows_all.to_csv(windows_path, index=False)
    scenario_rates.to_csv(scenario_path, index=False)
    ranks.to_csv(ranks_path, index=False)
    config_path.write_text(json.dumps(asdict(cfg), indent=2, sort_keys=True), encoding="utf-8")
    plotted = plot_lifecycle_recalibration(summary, figure_path)
    artifact_paths = [summary_path, windows_path, scenario_path, ranks_path, config_path]
    if plotted is not None:
        artifact_paths.append(plotted)
    progress_update("Lifecycle drift", "writing manifest", start=progress)
    manifest_path = write_run_manifest(out_root, repo_root=REPO_ROOT, data_root=data_root, cfg=cfg, seed=cfg.seed, artifact_paths=artifact_paths)
    progress_update("Lifecycle drift", "DONE", start=progress, detail=_safe_relative(summary_path, REPO_ROOT))
    return {"summary": summary, "windows": windows_all, "scenario_rates": scenario_rates, "rank_stability": ranks, "summary_path": summary_path, "windows_path": windows_path, "scenario_path": scenario_path, "ranks_path": ranks_path, "config_path": config_path, "figure_path": figure_path, "manifest_path": manifest_path}


def notebook_run_lifecycle_drift(*, data_root: Path, out_root: Path, cfg: LifecycleDriftConfig | None = None, seed: int = SEED, threads: int = THREADS) -> dict[str, Path | pd.DataFrame]:
    configure_reproducibility(seed=int(seed), threads=int(threads), matplotlib_backend="Agg")
    resolved_cfg = cfg if cfg is not None else notebook_lifecycle_config()
    return run_lifecycle_drift_recalibration(data_root=Path(data_root), out_root=Path(out_root), cfg=resolved_cfg)


def notebook_write_paper_tbd_replacements(*, tcad_summary: pd.DataFrame, lifecycle_summary: pd.DataFrame, total_features: int, out_path: Path) -> Path:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    rows: list[dict[str, Any]] = []
    if tcad_summary is not None and not tcad_summary.empty:
        sort_cols = [c for c in ["mcc", "bal_acc", "fpr", "hw_area_mm2", "window_size"] if c in tcad_summary.columns]
        ascending = [False if c in {"mcc", "bal_acc"} else True for c in sort_cols]
        best = tcad_summary.sort_values(sort_cols, ascending=ascending, kind="mergesort").iloc[0] if sort_cols else tcad_summary.iloc[0]
        n_features = int(best.get("n_selected_features", best.get("top_k", 0)))
        reduction = 100.0 * (1.0 - (n_features / max(int(total_features), 1)))
        rows.extend([
            {"paper_placeholder": "runtime telemetry feature-set reduction", "suggested_value": f"{reduction:.2f}%", "source_artifact": "results/notebook_run/tcad_ablation/tcad_ablation_summary.csv", "notes": f"best-row {n_features} selected features out of {int(total_features)} shared features"},
            {"paper_placeholder": "CINTAS area overhead", "suggested_value": f"{float(best.get('hw_setup_b_area_overhead_pct', np.nan)):.6f}%", "source_artifact": "results/notebook_run/tcad_ablation/tcad_ablation_summary.csv", "notes": "best-row Setup-B-normalized area overhead"},
            {"paper_placeholder": "CINTAS idle-power overhead", "suggested_value": f"{float(best.get('hw_idle_power_overhead_pct', np.nan)):.6f}%", "source_artifact": "results/notebook_run/tcad_ablation/tcad_ablation_summary.csv", "notes": "best-row idle-power-normalized overhead"},
            {"paper_placeholder": "fixed-point sensitivity maximum score delta", "suggested_value": f"{float(tcad_summary.get('fp_max_abs', pd.Series([np.nan])).max()):.6g}", "source_artifact": "results/notebook_run/tcad_ablation/tcad_ablation_summary.csv", "notes": "maximum absolute floating-point versus fixed-point score delta over the sweep"},
        ])
    if lifecycle_summary is not None and not lifecycle_summary.empty:
        pivot = lifecycle_summary.pivot_table(index="setup", columns="method", values="fpr", aggfunc="mean")
        for method in ["threshold_only", "full_feature_rank"]:
            if {"frozen", method}.issubset(pivot.columns):
                improvement_pp = 100.0 * (pivot["frozen"] - pivot[method]).mean()
                rows.append({"paper_placeholder": f"lifecycle FPR improvement with {method.replace('_', ' ')}", "suggested_value": f"{improvement_pp:.2f} percentage points", "source_artifact": "results/notebook_run/lifecycle_drift/lifecycle_recalibration_summary.csv", "notes": "positive means lower false positives than frozen initial calibration"})
    pd.DataFrame(rows).to_csv(out_path, index=False)
    return out_path


SOURCE_REGISTRY = DATA_SOURCE_CONFIG
MANIFEST_ROOT = REPO_ROOT / "data" / "external_manifests"


@dataclass(frozen=True)
class SourceFile:
    repo: str
    ref: str
    source_id: str
    source_path: str
    rel_path: str
    name: str
    size: int
    git_sha: str
    raw_url: str
    target_path: Path


def load_source_registry(path: Path = SOURCE_REGISTRY) -> dict[str, dict[str, Any]]:
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def github_token() -> str | None:
    token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
    if token:
        return token.strip()
    try:
        proc = subprocess.run(
            ["gh", "auth", "token"],
            check=False,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError:
        return None
    token = proc.stdout.strip()
    return token or None


def api_json(url: str, token: str | None) -> Any:
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "CITADEL-data-prep",
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode("utf-8"))


def quote_path(path: str) -> str:
    return urllib.parse.quote(path, safe="/")


def raw_url(repo: str, ref: str, path: str) -> str:
    return f"https://raw.githubusercontent.com/{repo}/{ref}/{quote_path(path)}"


def repo_tree(repo: str, ref: str, token: str | None) -> list[dict[str, Any]]:
    url = f"https://api.github.com/repos/{repo}/git/trees/{quote_path(ref)}?recursive=1"
    payload = api_json(url, token)
    if payload.get("truncated"):
        raise RuntimeError(f"GitHub returned a truncated tree for {repo}@{ref}; use a narrower source path.")
    return payload.get("tree", [])


def source_files(source_id: str, spec: dict[str, Any], token: str | None) -> list[SourceFile]:
    repo = spec["repo"]
    ref = spec.get("ref", "main")
    root = spec["path"].strip("/")
    target_root = REPO_ROOT / spec["target"]
    preserve_tree = bool(spec.get("preserve_tree", True))
    include_prefixes = spec.get("include_prefixes")

    files: list[SourceFile] = []
    for entry in repo_tree(repo, ref, token):
        if entry.get("type") != "blob":
            continue
        path = str(entry.get("path", ""))
        if not path.startswith(root + "/"):
            continue
        rel = path[len(root) + 1 :]
        if include_prefixes and not any(rel.startswith(prefix) for prefix in include_prefixes):
            continue
        local_rel = rel if preserve_tree else Path(rel).name
        files.append(
            SourceFile(
                repo=repo,
                ref=ref,
                source_id=source_id,
                source_path=path,
                rel_path=rel,
                name=Path(rel).name,
                size=int(entry.get("size", 0)),
                git_sha=str(entry.get("sha", "")),
                raw_url=raw_url(repo, ref, path),
                target_path=target_root / local_rel,
            )
        )
    return sorted(files, key=lambda f: f.rel_path)


def split_csv_arg(value: str | None) -> set[str] | None:
    if not value:
        return None
    items = {x.strip().upper() for x in value.split(",") if x.strip()}
    return items or None


def filter_ddr_files(
    files: Iterable[SourceFile],
    setups: set[str] | None,
    scenarios: set[str] | None,
    workloads: set[str] | None,
) -> list[SourceFile]:
    selected = []
    for item in files:
        parts = Path(item.name).stem.split("_")
        if len(parts) < 3:
            continue
        setup, scenario, workload = parts[0].upper(), parts[1].upper(), parts[2].upper()
        if setups and setup not in setups:
            continue
        if scenarios and scenario not in scenarios:
            continue
        if workloads and workload not in workloads:
            continue
        selected.append(item)
    return selected


def filter_apple_tiers(files: Iterable[SourceFile], tiers: set[str] | None) -> list[SourceFile]:
    if not tiers:
        return list(files)
    selected = []
    for item in files:
        top = item.rel_path.split("/", 1)[0].upper()
        if top in tiers:
            selected.append(item)
    return selected


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_one_source_file(item: SourceFile, token: str | None, force: bool = False) -> dict[str, Any]:
    item.target_path.parent.mkdir(parents=True, exist_ok=True)
    if item.target_path.exists() and not force:
        return {
            "status": "exists",
            "sha256": sha256_file(item.target_path),
            "local_size": item.target_path.stat().st_size,
        }

    headers = {"User-Agent": "CITADEL-data-prep"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    req = urllib.request.Request(item.raw_url, headers=headers)
    tmp_path = item.target_path.with_suffix(item.target_path.suffix + ".tmp")
    with urllib.request.urlopen(req, timeout=240) as resp, tmp_path.open("wb") as out:
        while True:
            chunk = resp.read(1024 * 1024)
            if not chunk:
                break
            out.write(chunk)
    tmp_path.replace(item.target_path)
    return {
        "status": "downloaded",
        "sha256": sha256_file(item.target_path),
        "local_size": item.target_path.stat().st_size,
    }


def manifest_entry(item: SourceFile, downloaded: dict[str, Any] | None = None) -> dict[str, Any]:
    entry = {
        "source_id": item.source_id,
        "repo": item.repo,
        "ref": item.ref,
        "source_path": item.source_path,
        "relative_path": item.rel_path,
        "name": item.name,
        "size": item.size,
        "git_sha": item.git_sha,
        "raw_url": item.raw_url,
        "target_path": str(item.target_path.relative_to(REPO_ROOT)),
    }
    if downloaded:
        entry.update(downloaded)
    return entry


def write_external_manifest(
    source_id: str,
    spec: dict[str, Any],
    files: list[SourceFile],
    entries: list[dict[str, Any]],
) -> Path:
    MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
    path = MANIFEST_ROOT / f"{source_id}.json"
    payload = {
        "source_id": source_id,
        "repo": spec["repo"],
        "ref": spec.get("ref", "main"),
        "source_path": spec["path"],
        "target": spec["target"],
        "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "file_count": len(files),
        "total_bytes": sum(f.size for f in files),
        "files": entries,
    }
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return path


def select_sources(registry: dict[str, dict[str, Any]], requested: str) -> list[str]:
    if requested == "all":
        return list(registry.keys())
    if requested not in registry:
        raise KeyError(f"Unknown source {requested!r}. Available: {', '.join(registry)}")
    return [requested]


def prepare_external_data_notebook(
    source: str = "all",
    *,
    download: bool = False,
    force: bool = False,
    setups: str | None = None,
    scenarios: str | None = None,
    workloads: str | None = None,
    apple_tiers: str | None = None,
    max_files: int | None = None,
) -> pd.DataFrame:
    """Create source manifests and, when requested, fetch DDR/Apple telemetry into this repo."""
    registry = load_source_registry()
    token = github_token()
    requested = select_sources(registry, source)

    setup_filter = split_csv_arg(setups)
    scenario_filter = split_csv_arg(scenarios)
    workload_filter = split_csv_arg(workloads)
    tier_filter = split_csv_arg(apple_tiers)

    rows: list[dict[str, Any]] = []
    for source_id in requested:
        spec = registry[source_id]
        files = source_files(source_id, spec, token)
        if spec.get("kind") == "hardware_counter":
            files = filter_ddr_files(files, setup_filter, scenario_filter, workload_filter)
        if spec.get("kind") == "limited_observability_host":
            files = filter_apple_tiers(files, tier_filter)
        if max_files is not None:
            files = files[: int(max_files)]

        entries: list[dict[str, Any]] = []
        for idx, item in enumerate(files, start=1):
            downloaded = download_one_source_file(item, token=token, force=force) if download else None
            entries.append(manifest_entry(item, downloaded=downloaded))
            rows.append({
                "source": source_id,
                "index": idx,
                "path": str(item.target_path.relative_to(REPO_ROOT)),
                "status": (downloaded or {}).get("status", "manifest_only"),
                "bytes": item.size,
            })

        manifest = write_external_manifest(source_id, spec, files, entries)
        rows.append({
            "source": source_id,
            "index": None,
            "path": str(manifest.relative_to(REPO_ROOT)),
            "status": "manifest_written",
            "bytes": sum(f.size for f in files),
        })
    return pd.DataFrame(rows)


def notebook_generate_sample_data(
    out_root: Path = REPO_ROOT / "data" / "sample",
    *,
    seed: int = SEED,
    rows: int = SAMPLE_ROWS,
    workloads: tuple[str, ...] = ("dft", "dj", "mm", "tr"),
) -> list[Path]:
    configure_reproducibility(seed=int(seed), threads=THREADS, matplotlib_backend="Agg")
    return create_sample_dataset(Path(out_root), seed=int(seed), workloads=tuple(workloads), n_rows=int(rows))


def notebook_run_exact_ets2026(
    *,
    data_root: Path,
    out_root: Path,
    seed: int = SEED,
    threads: int = THREADS,
    lambda_res: float = 0.5,
    agg_mode: str = "max",
    p_quantile: float = 0.99,
    corr_threshold: float = 0.35,
    window_sizes: tuple[int, ...] = (50, 100, 200),
    n_splits: int = 3,
) -> dict[str, Any]:
    configure_reproducibility(seed=int(seed), threads=int(threads), matplotlib_backend="Agg")
    cfg = ETS2026Config(
        window_sizes=tuple(int(x) for x in window_sizes),
        n_splits=int(n_splits),
        lambda_res=float(lambda_res),
        agg_mode=str(agg_mode),
        p_quantile=float(p_quantile),
        corr_threshold=float(corr_threshold),
        seed=int(seed),
    )
    return run_ets2026(data_root=Path(data_root), out_root=Path(out_root), cfg=cfg)


def notebook_tcad_config(
    preset: str = TCAD_PRESET,
    *,
    config_path: Path | None = None,
    seed: int = SEED,
) -> TCAD2026Config:
    resolved = Path(config_path) if config_path is not None else REPO_ROOT / "configs" / f"tcad_grid_{preset}.json"
    cfg = TCAD2026Config.from_json(resolved)
    return TCAD2026Config(
        scenarios_eval=cfg.scenarios_eval,
        feature_budgets=cfg.feature_budgets,
        window_sizes=cfg.window_sizes,
        lambda_res_values=cfg.lambda_res_values,
        agg_modes=cfg.agg_modes,
        weight_modes=cfg.weight_modes,
        fixed_point_q=cfg.fixed_point_q,
        n_splits=cfg.n_splits,
        p_quantile=cfg.p_quantile,
        corr_threshold=cfg.corr_threshold,
        graph_bootstraps=cfg.graph_bootstraps,
        graph_subsample_frac=cfg.graph_subsample_frac,
        graph_stability_threshold=cfg.graph_stability_threshold,
        seed=int(seed),
    )


def notebook_run_tcad_ablation(
    *,
    data_root: Path,
    out_root: Path,
    cfg: TCAD2026Config | None = None,
    preset: str = TCAD_PRESET,
    config_path: Path | None = None,
    seed: int = SEED,
    threads: int = THREADS,
) -> dict[str, Path | pd.DataFrame]:
    configure_reproducibility(seed=int(seed), threads=int(threads), matplotlib_backend="Agg")
    resolved_cfg = cfg if cfg is not None else notebook_tcad_config(preset, config_path=config_path, seed=seed)
    return run_tcad_ablation(data_root=Path(data_root), out_root=Path(out_root), cfg=resolved_cfg)


def notebook_plot_fig5_benign_causal_network(
    edges_a: Path,
    edges_b: Path,
    out_png: Path = RESULTS_ROOT / "figures" / "fig5_benign_sparse_causal_network.png",
    *,
    title_a: str = "Setup A",
    title_b: str = "Setup B",
    seed: int = 42,
) -> Path:
    configure_reproducibility(seed=int(seed), threads=THREADS, matplotlib_backend="Agg")
    plot_benign_sparse_causal_network_two_panel(
        edges_csv_A=Path(edges_a),
        edges_csv_B=Path(edges_b),
        out_png=Path(out_png),
        title_A=title_a,
        title_B=title_b,
        seed=int(seed),
    )
    return Path(out_png)


## 3. Prepare Data

This cell verifies the tracked telemetry folders used by the notebook. DDR4/DDR5 data is read from `data/telemetry/processed/ddr_data/`. Apple tier-0/1/2 data is read from `data/telemetry/raw/apple_data/` for limited-observability studies; it is not mixed into CINTAS hardware-cost claims. If `DATA_MODE` is set to `sample`, the notebook generates deterministic synthetic telemetry for a tiny pipeline check.


### Workload and Anomaly Labels

The hardware-counter experiments use the workload set $\mathcal{W}_{L}$, which contains 13 workload tags: DFT, DJ, DP, GS, GL, HA, JA, MM, NI, OE, PI, SH, and TR. These tags are inherited from the telemetry CSV filenames and are treated as stable workload labels throughout the notebook. Each workload is evaluated under benign operation and under SLM-relevant anomaly classes. The anomaly set is $\mathcal{A}=\{\mathrm{DROOP},\mathrm{RH},\mathrm{SPECTRE}\}$. DROOP represents voltage droop behavior. RH represents RowHammer/TRRespass-like memory disturbance. SPECTRE represents a speculative-execution security condition. BENIGN data is used for calibration and thresholding; anomaly labels are used only for evaluation.

| Label group | Labels | Notebook role |
|---|---|---|
| Workloads | DFT, DJ, DP, GS, GL, HA, JA, MM, NI, OE, PI, SH, TR | Repeated operating conditions used to test whether CITADEL is stable across workload behavior. |
| Benign class | BENIGN | Used to compute normalization constants, feature ranks, and thresholds. |
| Anomaly classes | DROOP, RH, SPECTRE | Used only after calibration to evaluate detection quality. |


In [ ]:
if DATA_SOURCE_CONFIG.exists():
    sources = json.loads(DATA_SOURCE_CONFIG.read_text())
    display(pd.DataFrame([
        {"source": key, "repo": val["repo"], "target": val["target"], "kind": val.get("kind", "")}
        for key, val in sources.items()
    ]))

display(pd.DataFrame([
    {
        "dataset": "ddr_data",
        "root": str(DDR_DATA_ROOT.relative_to(REPO_ROOT)),
        "csv_files": len(DDR_CSVS),
        "role": "CITADEL CINTAS experiments",
    },
    {
        "dataset": "apple_data",
        "root": str(APPLE_DATA_ROOT.relative_to(REPO_ROOT)),
        "csv_files": len(APPLE_CSVS),
        "role": "limited-observability analysis",
    },
]))

if DATA_MODE == "sample":
    created = notebook_generate_sample_data(DATA_ROOT, seed=SEED, rows=SAMPLE_ROWS)
    print(f"Generated {len(created)} deterministic telemetry CSVs under {DATA_ROOT}")
else:
    csvs = sorted(DATA_ROOT.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in {DATA_ROOT}")
    expected = 78
    if len(csvs) != expected:
        print(f"Warning: expected {expected} DDR CSVs, found {len(csvs)}")
    print(f"Using {len(csvs)} DDR telemetry CSVs from {DATA_ROOT}")

sample_files = sorted(DATA_ROOT.glob("*.csv"))[:8]
display(pd.DataFrame({"example_input_files": [str(p.relative_to(REPO_ROOT)) for p in sample_files]}))


### Methodology Roadmap: CITADEL From Start To Finish

This notebook is the executable methodology for the TCAD paper. The flow below keeps the paper story aligned with the runnable artifacts: benign telemetry is calibrated, stable conditional telemetry structure is learned, compact features are ranked under hardware constraints, CINTAS is swept and quantized, lifecycle drift is checked, and RTL/FPGA hooks are exported.

| Step | Notebook section | Method role | Main artifact |
|---:|---|---|---|
| 1 | Reproducibility Configuration | Fix seed, threads, paths, backend | deterministic run settings |
| 2 | Prepare Data | Load DDR telemetry and remove non-feature columns | input hashes and schema checks |
| 3 | Integrated Full DSE | Learn $G_{\mathcal{B}}$, sweep $\Omega$, and use the DROOP-adaptive branch for voltage transients | `tcad_ablation_summary.csv` |
| 4 | Detection Quality | Report AUROC, AUPRC, F1, balanced accuracy, MCC, FPR | anomaly-class tables |
| 5 | Feature-Budget Trade-Off | Sweep $k\in\mathcal{K}$ for accuracy--bandwidth cost | budget tables and figures |
| 6 | Stable Conditional Graph | Interpret edge stability, dependence, and top features | graph/rank CSVs and figures |
| 7 | Fixed-Point CINTAS | Sweep $Q\in\mathcal{Q}$ and report $\Delta_{\mathrm{MAE}}$, $\Delta_{\max}$ | fixed-point sensitivity tables |
| 8 | Hardware Cost | Estimate $A_{\mathrm{CINTAS}}$, $P_{\mathrm{CINTAS}}$, latency, bandwidth | hardware-cost summary |
| 9 | Lifecycle Drift | Check $\operatorname{FPR}_r>\eta$ and recalibrate | drift/recalibration CSVs |
| 10 | TCAD Results Gallery | Display five paper tables and five paper figures | `paper_figures/` gallery |
| 11 | RTL/FPGA Hooks | Export golden vectors and merge future synthesis results | `fpga/` and `rtl_sweep/` |
| 12 | Apple Supplement | Evaluate limited-observability portability separately | supplemental Apple artifacts |

#### Method Upgrade From EXACT To CITADEL

| Axis | EXACT | CITADEL |
|---|---|---|
| Telemetry map | $G_{\mathcal{B}}$ from benign causal/dependency learning | $G_{\mathcal{B}}=(V,E,D,\Pi)$ with strength $D$ and stability $\Pi$ |
| Feature compass | Compact causal top-$k$ features | $r_f=(\alpha_c c_f+\alpha_e e_f+\alpha_d d_f+\alpha_a a_f)/(\kappa_f+\epsilon)$ |
| Budget knob | One compact operating point | $k\in\mathcal{K}$ swept for accuracy--bandwidth trade-off |
| Score engine | CINTAS edge score $s(t)$ | $s(t)=(1-\lambda)E_2(t)+\lambda E_1(t)$, $\lambda\in\Lambda$ |
| Time lens | Selected decision windows | $N\in\mathcal{N}$, $\Phi\in\{\max,\mathrm{mean},\mathrm{median}\}$ |
| Number system | Fixed-point deployment path | $Q\in\mathcal{Q}$ with $\Delta_{\mathrm{MAE}}$ and $\Delta_{\max}$ |
| Silicon view | Compact detector motivation | $A_{\mathrm{CINTAS}}$, $P_{\mathrm{CINTAS}}$, latency, bandwidth, RTL/FPGA |
| Lifecycle mode | Frozen benign reference | $\operatorname{FPR}_r>\eta$ triggers threshold/full recalibration |
| Run contract | Portable code and results | Single notebook, manifests, hashes, deterministic seed/thread settings |

CITADEL keeps edge CINTAS scoring but upgrades the workflow to stable conditional graph learning, hardware-aware ranking, full design-space exploration, fixed-point sensitivity, lifecycle recalibration, and RTL/FPGA-oriented validation.


### Overleaf-Ready Upgrade Table

Use this compact table near the end of the Introduction or at the start of the Proposed Method section.

```latex
\begin{table*}[!t]
\centering
\caption{From EXACT to CITADEL: upgrading edge telemetry scoring into a hardware-aware SLM workflow.}
\label{tab:exact_to_citadel_upgrade}

\vspace{0.5mm}
\footnotesize
\setlength{\tabcolsep}{4.2pt}
\renewcommand{\arraystretch}{1.08}

\begin{tabularx}{0.97\textwidth}{@{}p{0.16\textwidth}XX@{}}
\toprule
\textbf{Axis} &
\textbf{EXACT} &
\textbf{CITADEL} \\
\midrule

Telemetry map &
$G_{\mathcal{B}}$ from benign causal/dependency learning &
$G_{\mathcal{B}}=(V,E,D,\Pi)$: strength $D$ plus stability $\Pi$ \\

Feature compass &
Compact causal top-$k$ features &
$r_f=(\alpha_c c_f+\alpha_e e_f+\alpha_d d_f+\alpha_a a_f)/(\kappa_f+\epsilon)$ \\

Budget knob &
One compact operating point &
$k\in\mathcal{K}$ swept for accuracy--bandwidth trade-off \\

Score engine &
CINTAS edge score $s(t)$ &
$s(t)=(1-\lambda)E_2(t)+\lambda E_1(t)$, $\lambda\in\Lambda$ \\

Time lens &
Selected decision windows &
$N\in\mathcal{N}$, $\Phi\in\{\max,\mathrm{mean},\mathrm{median}\}$ \\

Number system &
Fixed-point deployment path &
$Q\in\mathcal{Q}$ with $\Delta_{\mathrm{MAE}}$ and $\Delta_{\max}$ \\

Silicon view &
Compact detector motivation &
$A_{\mathrm{CINTAS}}$, $P_{\mathrm{CINTAS}}$, latency, bandwidth, RTL/FPGA \\

Lifecycle mode &
Frozen benign reference &
$\operatorname{FPR}_r>\eta$ triggers threshold/full recalibration \\

Run contract &
Portable code and results &
Single notebook, manifests, hashes, deterministic seed/thread settings \\

\bottomrule
\end{tabularx}

\vspace{0.6mm}
\begin{minipage}{0.92\textwidth}
\scriptsize
CITADEL keeps edge CINTAS scoring but adds stable conditional graph learning, hardware-aware ranking, full DSE, fixed-point sensitivity, and lifecycle recalibration.
\end{minipage}

\vspace{-0.5mm}
\end{table*}
```


## 4. Full Design-Space Sweep

This section runs the complete CITADEL design-space sweep used by the TCAD results. The standard CINTAS branch evaluates non-DROOP anomaly classes, while the DROOP branch uses transient-aware telemetry features inside the same DSE workflow. Thus, DROOP is not evaluated twice; the adaptive DROOP results are merged back into the main `tcad_ablation_summary.csv`. For final paper numbers, use `TCAD_PRESET = "full"` in the reproducibility configuration. After this section finishes once, the result-display subsections below can be rerun quickly from the saved CSV files.


In [ ]:
cfg_path = REPO_ROOT / "configs" / f"tcad_grid_{TCAD_PRESET}.json"

expected_progress_rev = "2026-05-08-safe-cv-dashboard"
if globals().get("CITADEL_NOTEBOOK_PROGRESS_REV") != expected_progress_rev:
    raise RuntimeError(
        "This kernel is using an older notebook helper cell. "
        "Restart the Jupyter kernel, pull the latest GitHub repo, then run the notebook from the top. "
        f"Expected progress revision: {expected_progress_rev}; loaded: {globals().get('CITADEL_NOTEBOOK_PROGRESS_REV')}"
    )

required_progress_helpers = ("progress_start", "progress_update", "progress_should_log")
missing_progress_helpers = [name for name in required_progress_helpers if name not in globals()]
if missing_progress_helpers:
    raise RuntimeError(
        "Progress-enabled helper functions are not loaded in this kernel. "
        "Restart the Jupyter kernel, pull the latest GitHub repo, then run the notebook from the top. "
        f"Missing helpers: {missing_progress_helpers}"
    )

progress_notice = (
    "TCAD progress is enabled. A live progress card and timestamped [TCAD ablation] lines "
    "should appear below before the long sweep starts."
)
print(progress_notice)
try:
    display(Markdown(f"**{progress_notice}**"))
except Exception:
    pass

tcad_cell_progress = progress_start("TCAD notebook cell", f"preset={TCAD_PRESET} config={cfg_path.relative_to(REPO_ROOT)}")
tcad_cfg = notebook_tcad_config(TCAD_PRESET, config_path=cfg_path, seed=SEED)
DROOP_ADAPTIVE_INCLUDED_IN_DSE = True
standard_scenarios = tuple(
    s for s in tcad_cfg.scenarios_eval
    if not (DROOP_ADAPTIVE_INCLUDED_IN_DSE and str(s).upper() == "DROOP")
)
tcad_run_cfg = TCAD2026Config(**{**asdict(tcad_cfg), "scenarios_eval": standard_scenarios})

print(f"TCAD config: {cfg_path.relative_to(REPO_ROOT)}")
print(f"Standard DSE scenarios: {tcad_run_cfg.scenarios_eval}")
if DROOP_ADAPTIVE_INCLUDED_IN_DSE and any(str(s).upper() == "DROOP" for s in tcad_cfg.scenarios_eval):
    print("DROOP standard branch is skipped here and evaluated by the integrated DROOP-adaptive branch below.")

progress_update(
    "TCAD notebook cell",
    "calling standard DSE helper",
    start=tcad_cell_progress,
    detail=f"scenarios={tcad_run_cfg.scenarios_eval}",
)

if tcad_run_cfg.scenarios_eval:
    tcad_artifacts = notebook_run_tcad_ablation(
        data_root=DATA_ROOT,
        out_root=TCAD_OUT,
        cfg=tcad_run_cfg,
        seed=SEED,
        threads=THREADS,
    )
    summary_standard = tcad_artifacts["summary"].copy()
    summary_standard = summary_standard[summary_standard["scenario"].astype(str).str.upper() != "DROOP"].copy()
    summary_standard["sweep_source"] = "standard_dse"
    standard_summary_path = TCAD_OUT / "tcad_ablation_standard_summary.csv"
    summary_standard.to_csv(standard_summary_path, index=False)

    standard_fold_results = tcad_artifacts.get("fold_results", pd.DataFrame()).copy()
    if not standard_fold_results.empty:
        standard_fold_results = standard_fold_results[standard_fold_results["scenario"].astype(str).str.upper() != "DROOP"].copy()
        standard_fold_results["sweep_source"] = "standard_dse"
        standard_fold_path = TCAD_OUT / "tcad_ablation_standard_fold_results.csv"
        standard_fold_results.to_csv(standard_fold_path, index=False)
        tcad_artifacts["fold_results"] = standard_fold_results
    else:
        standard_fold_path = None

    summary = summary_standard.copy()
    print(f"Standard DSE summary: {standard_summary_path.relative_to(REPO_ROOT)}")
    print(f"TCAD manifest: {tcad_artifacts['manifest_path'].relative_to(REPO_ROOT)}")
else:
    summary_standard = pd.DataFrame()
    standard_fold_results = pd.DataFrame()
    tcad_artifacts = {
        "summary": summary_standard,
        "fold_results": standard_fold_results,
        "summary_path": TCAD_OUT / "tcad_ablation_summary.csv",
        "fold_path": TCAD_OUT / "tcad_ablation_fold_results.csv",
    }
    summary = summary_standard.copy()
    print("No non-DROOP standard DSE scenarios were requested; only the DROOP-adaptive branch will run.")

progress_update(
    "TCAD notebook cell",
    "standard DSE branch complete",
    start=tcad_cell_progress,
    current=1,
    total=2 if DROOP_ADAPTIVE_INCLUDED_IN_DSE else 1,
    detail="DROOP adaptive branch follows" if DROOP_ADAPTIVE_INCLUDED_IN_DSE else "DONE",
)
display(summary.head(10))


**Integrated DROOP-adaptive branch.** The following cell completes the full DSE by replacing the standard DROOP branch with transient-aware DROOP features, lower benign threshold quantiles, and a broader feature-budget range. It then merges DROOP-adaptive rows with the non-DROOP standard DSE rows so downstream result sections read one integrated DSE summary.


In [ ]:
# DROOP-adaptive CINTAS experiment.
# Edit here if you want to change the DROOP-specific block lengths or feature budgets.
# The requested paper-scale block lengths are exactly 50, 100, 150, ..., 950, 1000.

import re
from pathlib import Path
from IPython.display import Image, Markdown, display

DROOP_BLOCK_LENGTHS = tuple(range(50, 1001, 50))
DROOP_FEATURE_BUDGETS = (5, 8, 10, 15, 20, 30, 50, 75, 100)
DROOP_P_QUANTILES = (0.95, 0.975, 0.99)
DROOP_FIXED_POINT_Q = (15, 18)
DROOP_N_SPLITS = 5  # Sparse long-window cases automatically use fewer folds when needed.
DROOP_MAX_BASE_COLUMNS = 80
DROOP_REBUILD_AUGMENTED_DATA = False  # Set True to force rebuilding cached augmented CSVs.

DROOP_ADAPTIVE_DATA_ROOT = RESULTS_ROOT / "droop_adaptive_data"
DROOP_ADAPTIVE_OUT = RESULTS_ROOT / "droop_adaptive_ablation"
DROOP_ADAPTIVE_DATA_ROOT.mkdir(parents=True, exist_ok=True)
DROOP_ADAPTIVE_OUT.mkdir(parents=True, exist_ok=True)

droop_progress = progress_start(
    "DROOP-adaptive CINTAS",
    f"windows={DROOP_BLOCK_LENGTHS[0]}..{DROOP_BLOCK_LENGTHS[-1]} step=50",
)


def _safe_feature_name(name: str) -> str:
    safe = re.sub(r"[^A-Za-z0-9_]+", "_", str(name)).strip("_")
    return safe or "feature"


def droop_priority_columns(df: pd.DataFrame, max_cols: int = 80) -> list[str]:
    """Prefer columns likely to expose voltage droop or its system-level symptoms."""
    exclude = {"setup", "scenario", "workload", "label", "is_anom", "time_idx"}
    numeric_cols = [
        c for c in df.select_dtypes(include=[np.number]).columns
        if str(c) not in exclude and not str(c).startswith("Unnamed:")
    ]
    if not numeric_cols:
        return []

    droop_regex = re.compile(
        r"volt|vdd|vcc|vid|freq|clock|aperf|mperf|ratio|power|pwr|rapl|energy|"
        r"temp|thermal|throttle|stall|cycle|ipc|instr|uops|miss|cache|llc|l2|l3|"
        r"mem|dram|imc|lat|retry|error|margin",
        re.IGNORECASE,
    )
    preferred = [c for c in numeric_cols if droop_regex.search(str(c))]
    remaining = [c for c in numeric_cols if c not in preferred]

    if remaining:
        variances = df[remaining].astype(float).var(numeric_only=True).sort_values(ascending=False)
        remaining = variances.index.tolist()

    return (preferred + remaining)[: min(max_cols, len(numeric_cols))]


def add_droop_transient_features(df: pd.DataFrame, max_cols: int = 80) -> pd.DataFrame:
    """Add transient-aware columns within each setup/scenario/workload trace."""
    out = df.copy()
    cols = droop_priority_columns(out, max_cols=max_cols)
    if not cols:
        return out

    group_cols = [c for c in ["setup", "scenario", "workload"] if c in out.columns]
    group_key = None
    if group_cols:
        group_key = out[group_cols].astype(str).agg("|".join, axis=1)

    def _clean_series(series: pd.Series) -> pd.Series:
        return (
            pd.to_numeric(series, errors="coerce")
            .astype(float)
            .replace([np.inf, -np.inf], np.nan)
            .ffill()
            .bfill()
            .fillna(0.0)
        )

    def _trace_transform(series: pd.Series, fn) -> pd.Series:
        if group_key is None:
            return fn(series)
        return series.groupby(group_key, sort=False).transform(fn)

    new_cols: dict[str, pd.Series] = {}
    for idx, col in enumerate(cols):
        x = _clean_series(out[col])
        dx = _trace_transform(x, lambda s: s.diff().fillna(0.0))
        roll5_max = _trace_transform(x, lambda s: s.rolling(window=5, min_periods=1).max())
        roll5_min = _trace_transform(x, lambda s: s.rolling(window=5, min_periods=1).min())
        roll10_max = _trace_transform(x, lambda s: s.rolling(window=10, min_periods=1).max())
        roll10_std = _trace_transform(x, lambda s: s.rolling(window=10, min_periods=1).std().fillna(0.0))
        med20 = _trace_transform(x, lambda s: s.rolling(window=20, min_periods=1).median())
        mad20 = _trace_transform((x - med20).abs(), lambda s: s.rolling(window=20, min_periods=1).median())
        mad20 = mad20.replace(0.0, np.nan)
        fallback = float(mad20.dropna().median()) if mad20.notna().any() else 0.0
        safe = f"{idx:03d}_{_safe_feature_name(col)}"

        # delta/abs-delta expose sudden motion; rolling range/std expose bursts;
        # drop-from-recent-max helps voltage/frequency-like downward transients;
        # robust-z exposes local deviation without needing anomaly labels.
        new_cols[f"{safe}__delta1"] = dx
        new_cols[f"{safe}__abs_delta1"] = dx.abs()
        new_cols[f"{safe}__range5"] = roll5_max - roll5_min
        new_cols[f"{safe}__std10"] = roll10_std
        new_cols[f"{safe}__drop_from_max10"] = roll10_max - x
        new_cols[f"{safe}__robust_z20"] = ((x - med20).abs() / (mad20.fillna(fallback) + 1e-9)).fillna(0.0)

    if new_cols:
        out = pd.concat([out, pd.DataFrame(new_cols, index=out.index)], axis=1).copy()
    return out


# Build a DROOP-only data view so causal/ranking alignment is not dominated by RH or SPECTRE.
progress_update("DROOP-adaptive CINTAS", "building DROOP-only augmented telemetry", start=droop_progress)
eligible_sources = [
    src for src in sorted(Path(DATA_ROOT).glob("DDR*.csv"))
    if "_benign_" in src.name.lower() or "_droop_" in src.name.lower()
]
if not eligible_sources:
    raise RuntimeError(f"No BENIGN/DROOP telemetry CSVs found in {DATA_ROOT}")

created = []
reused = []
for file_idx, src in enumerate(eligible_sources, start=1):
    dst = DROOP_ADAPTIVE_DATA_ROOT / src.name
    if (
        dst.exists()
        and not DROOP_REBUILD_AUGMENTED_DATA
        and dst.stat().st_mtime >= src.stat().st_mtime
    ):
        progress_update(
            "DROOP-adaptive CINTAS",
            "reusing cached augmented telemetry",
            start=droop_progress,
            current=file_idx,
            total=len(eligible_sources),
            detail=src.name,
        )
        created.append(dst)
        reused.append(dst)
        continue

    progress_update(
        "DROOP-adaptive CINTAS",
        "loading source telemetry",
        start=droop_progress,
        current=file_idx,
        total=len(eligible_sources),
        detail=src.name,
    )
    raw = _sanitize_telemetry_columns(pd.read_csv(src))
    progress_update(
        "DROOP-adaptive CINTAS",
        "adding transient features",
        start=droop_progress,
        current=file_idx,
        total=len(eligible_sources),
        detail=f"{src.name}: rows={len(raw):,} cols={raw.shape[1]}",
    )
    aug = add_droop_transient_features(raw, max_cols=DROOP_MAX_BASE_COLUMNS)
    progress_update(
        "DROOP-adaptive CINTAS",
        "writing augmented telemetry",
        start=droop_progress,
        current=file_idx,
        total=len(eligible_sources),
        detail=f"{dst.name}: cols={aug.shape[1]}",
    )
    aug.to_csv(dst, index=False)
    created.append(dst)

if not created:
    raise RuntimeError(f"No augmented BENIGN/DROOP telemetry files were created in {DROOP_ADAPTIVE_DATA_ROOT}")

print(f"Created {len(created)} DROOP-adaptive CSVs in {DROOP_ADAPTIVE_DATA_ROOT.relative_to(REPO_ROOT)}")
print(f"Block lengths: {DROOP_BLOCK_LENGTHS}")
print(f"Feature budgets: {DROOP_FEATURE_BUDGETS}")
print(f"Requested folds: {DROOP_N_SPLITS} (auto-reduced only when a long-window class has fewer blocks)")

# Run multiple benign threshold quantiles. Lower quantiles may improve DROOP recall;
# the table keeps FPR visible so the paper can choose an honest operating point.
summary_frames = []
selected_frames = []
fold_frames = []
for p_quantile in DROOP_P_QUANTILES:
    out_root = DROOP_ADAPTIVE_OUT / f"p{str(p_quantile).replace('.', '_')}"
    cfg = TCAD2026Config(
        scenarios_eval=("DROOP",),
        feature_budgets=DROOP_FEATURE_BUDGETS,
        window_sizes=DROOP_BLOCK_LENGTHS,
        lambda_res_values=(0.0, 0.25, 0.5, 0.75, 1.0),
        agg_modes=("max", "mean", "median"),
        weight_modes=("uniform", "inv_var"),
        fixed_point_q=DROOP_FIXED_POINT_Q,
        n_splits=DROOP_N_SPLITS,
        p_quantile=float(p_quantile),
        corr_threshold=0.35,
        seed=SEED,
    )
    progress_update(
        "DROOP-adaptive CINTAS",
        "running DROOP-only sweep",
        start=droop_progress,
        detail=f"p={p_quantile} windows={len(DROOP_BLOCK_LENGTHS)} budgets={len(DROOP_FEATURE_BUDGETS)}",
    )
    artifacts = notebook_run_tcad_ablation(
        data_root=DROOP_ADAPTIVE_DATA_ROOT,
        out_root=out_root,
        cfg=cfg,
        seed=SEED,
        threads=THREADS,
    )
    s = artifacts["summary"].copy()
    s["sweep_source"] = "droop_adaptive_dse"
    s["droop_adaptive_p_quantile"] = float(p_quantile)
    s["droop_adaptive_out_root"] = str(out_root.relative_to(REPO_ROOT))
    summary_frames.append(s)

    selected_path = artifacts.get("selected_path")
    if selected_path is not None and Path(selected_path).exists():
        sf = pd.read_csv(selected_path)
        sf["sweep_source"] = "droop_adaptive_dse"
        sf["droop_adaptive_p_quantile"] = float(p_quantile)
        selected_frames.append(sf)

    fold_df = artifacts.get("fold_results")
    if isinstance(fold_df, pd.DataFrame) and not fold_df.empty:
        ff = fold_df.copy()
        ff["sweep_source"] = "droop_adaptive_dse"
        ff["droop_adaptive_p_quantile"] = float(p_quantile)
        ff["droop_adaptive_out_root"] = str(out_root.relative_to(REPO_ROOT))
        fold_frames.append(ff)

if not summary_frames:
    raise RuntimeError("DROOP-adaptive sweep produced no summary rows; check BENIGN/DROOP data, block lengths, and n_splits.")

adaptive_summary = pd.concat(summary_frames, ignore_index=True)
adaptive_summary_path = DROOP_ADAPTIVE_OUT / "droop_adaptive_summary_all_quantiles.csv"
adaptive_summary.to_csv(adaptive_summary_path, index=False)

if selected_frames:
    adaptive_selected = pd.concat(selected_frames, ignore_index=True)
    adaptive_selected_path = DROOP_ADAPTIVE_OUT / "droop_adaptive_selected_features_all_quantiles.csv"
    adaptive_selected.to_csv(adaptive_selected_path, index=False)
else:
    adaptive_selected_path = None

if fold_frames:
    adaptive_fold_results = pd.concat(fold_frames, ignore_index=True, sort=False)
    adaptive_fold_path = DROOP_ADAPTIVE_OUT / "droop_adaptive_fold_results_all_quantiles.csv"
    adaptive_fold_results.to_csv(adaptive_fold_path, index=False)
else:
    adaptive_fold_results = pd.DataFrame()
    adaptive_fold_path = None

# Merge the DROOP-adaptive branch into the main DSE artifacts so downstream
# sections see one integrated DSE instead of standard-DROOP plus adaptive-DROOP.
standard_summary_for_dse = globals().get("summary_standard", pd.DataFrame()).copy()
standard_summary_path = TCAD_OUT / "tcad_ablation_standard_summary.csv"
if standard_summary_for_dse.empty and standard_summary_path.exists():
    standard_summary_for_dse = pd.read_csv(standard_summary_path)
elif standard_summary_for_dse.empty and "summary" in globals() and isinstance(summary, pd.DataFrame):
    standard_summary_for_dse = summary.copy()
if not standard_summary_for_dse.empty and "scenario" in standard_summary_for_dse.columns:
    standard_summary_for_dse = standard_summary_for_dse[standard_summary_for_dse["scenario"].astype(str).str.upper() != "DROOP"].copy()
if not standard_summary_for_dse.empty:
    if "sweep_source" not in standard_summary_for_dse.columns:
        standard_summary_for_dse["sweep_source"] = "standard_dse"
    else:
        standard_summary_for_dse["sweep_source"] = standard_summary_for_dse["sweep_source"].fillna("standard_dse")

summary = pd.concat([standard_summary_for_dse, adaptive_summary], ignore_index=True, sort=False)
tcad_integrated_summary_path = TCAD_OUT / "tcad_ablation_summary.csv"
tcad_integrated_summary_copy_path = TCAD_OUT / "tcad_ablation_summary_integrated.csv"
summary.to_csv(tcad_integrated_summary_path, index=False)
summary.to_csv(tcad_integrated_summary_copy_path, index=False)

standard_fold_for_dse = pd.DataFrame()
if "tcad_artifacts" in globals() and isinstance(tcad_artifacts, dict) and isinstance(tcad_artifacts.get("fold_results"), pd.DataFrame):
    standard_fold_for_dse = tcad_artifacts["fold_results"].copy()
standard_fold_path = TCAD_OUT / "tcad_ablation_standard_fold_results.csv"
if standard_fold_for_dse.empty and standard_fold_path.exists():
    standard_fold_for_dse = pd.read_csv(standard_fold_path)
if not standard_fold_for_dse.empty and "scenario" in standard_fold_for_dse.columns:
    standard_fold_for_dse = standard_fold_for_dse[standard_fold_for_dse["scenario"].astype(str).str.upper() != "DROOP"].copy()
if not standard_fold_for_dse.empty:
    if "sweep_source" not in standard_fold_for_dse.columns:
        standard_fold_for_dse["sweep_source"] = "standard_dse"
    else:
        standard_fold_for_dse["sweep_source"] = standard_fold_for_dse["sweep_source"].fillna("standard_dse")

fold_results_integrated = pd.concat([standard_fold_for_dse, adaptive_fold_results], ignore_index=True, sort=False)
if not fold_results_integrated.empty:
    tcad_integrated_fold_path = TCAD_OUT / "tcad_ablation_fold_results.csv"
    tcad_integrated_fold_copy_path = TCAD_OUT / "tcad_ablation_fold_results_integrated.csv"
    fold_results_integrated.to_csv(tcad_integrated_fold_path, index=False)
    fold_results_integrated.to_csv(tcad_integrated_fold_copy_path, index=False)
else:
    tcad_integrated_fold_path = TCAD_OUT / "tcad_ablation_fold_results.csv"

if "tcad_artifacts" not in globals() or not isinstance(tcad_artifacts, dict):
    tcad_artifacts = {}
tcad_artifacts["summary"] = summary
tcad_artifacts["summary_path"] = tcad_integrated_summary_path
tcad_artifacts["fold_results"] = fold_results_integrated
tcad_artifacts["fold_path"] = tcad_integrated_fold_path
DROOP_ADAPTIVE_ALREADY_IN_TCAD_SUMMARY = True

rank_cols = ["mcc", "bal_acc", "auc_roc", "f1", "fpr"]
rank_ascending = [False, False, False, False, True]
adaptive_best = (
    adaptive_summary
    .sort_values(rank_cols, ascending=rank_ascending, kind="mergesort")
    .groupby(["setup", "scenario"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

paper_cols = [
    "setup", "scenario", "droop_adaptive_p_quantile", "top_k", "n_selected_features",
    "window_size", "agg_mode", "lambda_res", "weight_mode", "fixed_point_q",
    "auc_roc", "auc_pr", "f1", "bal_acc", "mcc", "fpr", "brier", "ece",
    "fp_mae", "fp_max_abs", "hw_setup_b_area_overhead_pct", "hw_idle_power_overhead_pct",
]
adaptive_best_table = adaptive_best[[c for c in paper_cols if c in adaptive_best.columns]].copy()
adaptive_best_path = DROOP_ADAPTIVE_OUT / "droop_adaptive_best_by_setup.csv"
adaptive_best_table.to_csv(adaptive_best_path, index=False)

print(f"DROOP-adaptive summary: {adaptive_summary_path.relative_to(REPO_ROOT)}")
print(f"DROOP-adaptive best table: {adaptive_best_path.relative_to(REPO_ROOT)}")
print(f"Integrated DSE summary: {tcad_integrated_summary_path.relative_to(REPO_ROOT)}")
print(f"Integrated DSE fold results: {tcad_integrated_fold_path.relative_to(REPO_ROOT)}")
display(adaptive_best_table.round(6))
display(Markdown("### Integrated DSE preview: standard non-DROOP + adaptive DROOP"))
display(summary.head(12).round(6))

# Standard DROOP is intentionally skipped in the main branch when integrated
# DROOP-adaptive DSE is enabled, so there is no duplicate standard-DROOP
# comparison in the paper-facing flow.
print("Integrated DSE uses adaptive DROOP only; standard DROOP is not run twice.")

# Figure: best DROOP MCC over the requested 50-step block-length grid.
plot_df = (
    adaptive_summary
    .sort_values(rank_cols, ascending=rank_ascending, kind="mergesort")
    .groupby(["setup", "scenario", "window_size"], as_index=False)
    .head(1)
    .sort_values(["setup", "window_size"])
)

fig_path = DROOP_ADAPTIVE_OUT / "fig_droop_adaptive_mcc_vs_block_length.png"
fig, ax = plt.subplots(figsize=(8.0, 4.2))
for setup_name, group in plot_df.groupby("setup", sort=True):
    ax.plot(group["window_size"], group["mcc"], marker="o", linewidth=2.4, label=f"Setup {setup_name} / DROOP")
ax.set_title("DROOP-Adaptive CINTAS: Best MCC versus Decision-Block Length")
ax.set_xlabel("Decision-block length $N$")
ax.set_ylabel("Best MCC")
ax.set_xticks(list(DROOP_BLOCK_LENGTHS))
ax.tick_params(axis="x", labelrotation=45)
ax.set_ylim(-0.05, 1.02)
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)
display(Image(filename=str(fig_path)))
print(f"Saved figure: {fig_path.relative_to(REPO_ROOT)}")

progress_update(
    "DROOP-adaptive CINTAS",
    "DONE",
    start=droop_progress,
    current=1,
    total=2,
    detail=tcad_integrated_summary_path.relative_to(REPO_ROOT),
)
if "tcad_cell_progress" in globals():
    progress_update("TCAD notebook cell", "DONE", start=tcad_cell_progress, current=2, total=2, detail=tcad_integrated_summary_path.relative_to(REPO_ROOT))


### Section 4 Paper Figures: Full DSE and Workload Sweep

Run this cell after the full design-space sweep. It converts the saved sweep CSVs into paper-facing Section 4 artifacts: a full DSE heatmap over feature budget and decision-block length, plus a workload-sweep heatmap/table that shows robustness across workloads.


In [ ]:
# Section 4 paper-facing DSE and workload-sweep figures.
# Run this after Section 4 finishes. It reads saved CSVs and does not rerun the full sweep.

from IPython.display import Image, Markdown, display
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DSE_FIG_DIR = TCAD_OUT / "paper_figures"
DSE_FIG_DIR.mkdir(parents=True, exist_ok=True)
DSE_SUMMARY_PATH = TCAD_OUT / "tcad_ablation_summary.csv"
DSE_FOLD_PATH = TCAD_OUT / "tcad_ablation_fold_results.csv"

if "summary" in globals() and isinstance(summary, pd.DataFrame) and not summary.empty:
    dse_summary = summary.copy()
elif DSE_SUMMARY_PATH.exists():
    dse_summary = pd.read_csv(DSE_SUMMARY_PATH)
else:
    raise RuntimeError("Run Section 4 first, or make sure tcad_ablation_summary.csv exists.")

fold_cols = [
    "setup", "scenario", "workload", "sweep_source", "droop_adaptive_p_quantile", "window_size", "agg_mode", "lambda_res", "p_quantile",
    "n_splits", "requested_n_splits", "top_k", "n_selected_features", "weight_mode", "fixed_point_q",
    "auc_roc", "auc_pr", "f1", "bal_acc", "mcc", "fpr", "brier", "ece",
    "fp_mae", "fp_max_abs", "hw_setup_b_area_overhead_pct", "hw_idle_power_overhead_pct",
]
if "tcad_artifacts" in globals() and isinstance(tcad_artifacts, dict) and isinstance(tcad_artifacts.get("fold_results"), pd.DataFrame):
    dse_folds = tcad_artifacts["fold_results"].copy()
    dse_folds = dse_folds[[c for c in fold_cols if c in dse_folds.columns]]
elif DSE_FOLD_PATH.exists():
    dse_folds = pd.read_csv(DSE_FOLD_PATH, usecols=lambda c: c in fold_cols)
else:
    raise RuntimeError("Run Section 4 first, or make sure tcad_ablation_fold_results.csv exists.")

rank_cols = [c for c in ["mcc", "bal_acc", "auc_roc", "f1", "fpr"] if c in dse_summary.columns]
rank_ascending = [False, False, False, False, True][: len(rank_cols)]

def _save_table(df: pd.DataFrame, name: str) -> Path:
    path = DSE_FIG_DIR / name
    df.to_csv(path, index=False)
    return path

def _save_show(fig, name: str) -> Path:
    path = DSE_FIG_DIR / name
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    display(Image(filename=str(path)))
    return path

def _best_rows(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    local_rank_cols = [c for c in rank_cols if c in df.columns]
    local_ascending = [rank_ascending[rank_cols.index(c)] for c in local_rank_cols]
    if not local_rank_cols:
        return df.groupby(group_cols, as_index=False).head(1).reset_index(drop=True)
    return (
        df.sort_values(local_rank_cols, ascending=local_ascending, kind="mergesort", na_position="last")
        .groupby(group_cols, as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

# Table: best operating point per setup/anomaly class from the full sweep.
best_pair_cols = [
    "setup", "scenario", "sweep_source", "droop_adaptive_p_quantile", "top_k", "n_selected_features", "window_size", "agg_mode",
    "lambda_res", "weight_mode", "fixed_point_q", "auc_roc", "auc_pr", "f1",
    "bal_acc", "mcc", "fpr", "fp_mae", "fp_max_abs",
    "hw_setup_b_area_overhead_pct", "hw_idle_power_overhead_pct",
]
best_by_pair = _best_rows(dse_summary, ["setup", "scenario"])
best_by_pair = best_by_pair[[c for c in best_pair_cols if c in best_by_pair.columns]]
best_by_pair_path = _save_table(best_by_pair, "section4_full_dse_best_by_setup_scenario.csv")

display(Markdown("#### Section 4 table: best full-DSE operating point by setup and anomaly class"))
display(best_by_pair.round(6))
print(f"Saved table: {best_by_pair_path.relative_to(REPO_ROOT)}")

# Figure 1: full DSE map over feature budget and decision-block length.
dse_envelope = (
    dse_summary.groupby(["setup", "scenario", "top_k", "window_size"], as_index=False)
    .agg(best_mcc=("mcc", "max"), best_bal_acc=("bal_acc", "max"), min_fpr=("fpr", "min"))
)
dse_envelope_path = _save_table(dse_envelope, "section4_full_dse_budget_window_envelope.csv")

pairs = sorted(set(zip(dse_envelope["setup"].astype(str), dse_envelope["scenario"].astype(str))))
n_pairs = len(pairs)
ncols = 2 if n_pairs > 1 else 1
nrows = int(np.ceil(max(n_pairs, 1) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6.4 * ncols, 3.7 * nrows), squeeze=False, sharex=False, sharey=False)
all_vals = dse_envelope["best_mcc"].replace([np.inf, -np.inf], np.nan).dropna()
vmin = min(-0.05, float(all_vals.min())) if not all_vals.empty else -0.05
vmax = 1.0
last_im = None
for ax, (setup_name, scenario_name) in zip(axes.flat, pairs):
    sub = dse_envelope[(dse_envelope["setup"].astype(str) == setup_name) & (dse_envelope["scenario"].astype(str) == scenario_name)].copy()
    pivot = sub.pivot_table(index="top_k", columns="window_size", values="best_mcc", aggfunc="max").sort_index(ascending=False)
    data = pivot.to_numpy(dtype=float)
    last_im = ax.imshow(data, aspect="auto", cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(f"Setup {setup_name} / {scenario_name}")
    ax.set_xlabel("Decision-block length N")
    ax.set_ylabel("Feature budget k")
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels([str(int(x)) for x in pivot.index])
    xticks = np.arange(len(pivot.columns))
    keep = max(1, int(np.ceil(len(xticks) / 8)))
    ax.set_xticks(xticks[::keep])
    ax.set_xticklabels([str(int(x)) for x in pivot.columns[::keep]], rotation=35, ha="right")
    # Mark each panel's best DSE point.
    if not sub.empty and sub["best_mcc"].notna().any():
        best = sub.sort_values(["best_mcc", "best_bal_acc", "min_fpr"], ascending=[False, False, True], kind="mergesort").iloc[0]
        y = list(pivot.index).index(best["top_k"])
        x = list(pivot.columns).index(best["window_size"])
        ax.scatter([x], [y], marker="*", s=180, c="white", edgecolors="black", linewidths=0.8)
        ax.text(x, y, f" {best['best_mcc']:.2f}", va="center", ha="left", fontsize=8, color="white", fontweight="bold")
for ax in axes.flat[n_pairs:]:
    ax.axis("off")
if last_im is not None:
    cbar = fig.colorbar(last_im, ax=axes.ravel().tolist(), shrink=0.92, pad=0.015)
    cbar.set_label("Best MCC")
fig.suptitle("Full Design-Space Sweep: Accuracy Envelope over Feature Budget and Block Length", y=1.02, fontsize=13, fontweight="bold")
full_dse_fig_path = _save_show(fig, "fig_section4_full_dse_budget_window_heatmap.png")
print(f"Saved table: {dse_envelope_path.relative_to(REPO_ROOT)}")
print(f"Saved figure: {full_dse_fig_path.relative_to(REPO_ROOT)}")

# Workload sweep: select the best DSE point per setup/scenario/workload.
per_workload = dse_folds[dse_folds["workload"].astype(str).str.upper() != "ALL"].copy()
if per_workload.empty:
    raise RuntimeError("No per-workload rows found in fold results. Make sure Section 4 completed successfully.")

cfg_cols = [c for c in [
    "setup", "scenario", "workload", "window_size", "agg_mode", "lambda_res", "p_quantile",
    "top_k", "n_selected_features", "weight_mode", "fixed_point_q"
] if c in per_workload.columns]
metric_cols = [c for c in [
    "auc_roc", "auc_pr", "f1", "bal_acc", "mcc", "fpr", "brier", "ece",
    "fp_mae", "fp_max_abs", "hw_setup_b_area_overhead_pct", "hw_idle_power_overhead_pct"
] if c in per_workload.columns]
per_workload_mean = per_workload.groupby(cfg_cols, as_index=False)[metric_cols].mean(numeric_only=True)
workload_best = _best_rows(per_workload_mean, ["setup", "scenario", "workload"])
workload_best["setup_scenario"] = workload_best["setup"].astype(str) + "-" + workload_best["scenario"].astype(str)
workload_best_path = _save_table(workload_best, "section4_workload_sweep_best_by_workload.csv")

workload_summary = (
    workload_best.groupby(["setup", "scenario"], as_index=False)
    .agg(
        workloads=("workload", "nunique"),
        mean_mcc=("mcc", "mean"),
        median_mcc=("mcc", "median"),
        min_mcc=("mcc", "min"),
        max_mcc=("mcc", "max"),
        mean_fpr=("fpr", "mean"),
        max_fpr=("fpr", "max"),
        median_top_k=("top_k", "median"),
        median_window_size=("window_size", "median"),
    )
)
workload_summary_path = _save_table(workload_summary, "section4_workload_sweep_summary.csv")

weak_workloads = workload_best.sort_values(["mcc", "bal_acc", "fpr"], ascending=[True, True, False], kind="mergesort").head(15)
weak_workloads_path = _save_table(weak_workloads, "section4_workload_sweep_lowest_margin_cases.csv")

display(Markdown("#### Section 4 table: workload-sweep robustness summary"))
display(workload_summary.round(6))
display(Markdown("#### Lowest-margin workload cases to discuss or improve"))
display(weak_workloads[[c for c in best_pair_cols + ["workload"] if c in weak_workloads.columns]].round(6))
print(f"Saved table: {workload_best_path.relative_to(REPO_ROOT)}")
print(f"Saved table: {workload_summary_path.relative_to(REPO_ROOT)}")
print(f"Saved table: {weak_workloads_path.relative_to(REPO_ROOT)}")

# Figure 2: workload x setup/scenario heatmap with mean-MCC side bars.
workload_order = (
    workload_best.groupby("workload")["mcc"].mean().sort_values(ascending=False).index.tolist()
)
pair_order = (
    workload_best.groupby("setup_scenario")["mcc"].mean().sort_values(ascending=True).index.tolist()
)
heat = (
    workload_best.pivot_table(index="setup_scenario", columns="workload", values="mcc", aggfunc="max")
    .reindex(index=pair_order, columns=workload_order)
)
fig = plt.figure(figsize=(max(10.0, 0.62 * len(workload_order)), max(4.0, 0.55 * len(pair_order))))
grid = fig.add_gridspec(1, 2, width_ratios=[5.0, 1.15], wspace=0.04)
ax = fig.add_subplot(grid[0, 0])
ax_bar = fig.add_subplot(grid[0, 1], sharey=ax)
im = ax.imshow(heat.to_numpy(dtype=float), aspect="auto", cmap="magma", vmin=vmin, vmax=vmax)
ax.set_title("Workload Sweep: Best MCC by Workload and Anomaly Class", fontweight="bold")
ax.set_xlabel("Workload")
ax.set_ylabel("Setup / anomaly")
ax.set_xticks(np.arange(len(workload_order)))
ax.set_xticklabels(workload_order, rotation=45, ha="right")
ax.set_yticks(np.arange(len(pair_order)))
ax.set_yticklabels(pair_order)
for y in range(heat.shape[0]):
    for x in range(heat.shape[1]):
        val = heat.iloc[y, x]
        if pd.notna(val):
            ax.text(x, y, f"{val:.2f}", ha="center", va="center", fontsize=7, color="white" if val < 0.62 else "black")
mean_by_pair = heat.mean(axis=1, skipna=True)
ax_bar.barh(np.arange(len(pair_order)), mean_by_pair.to_numpy(dtype=float), color="#4c78a8", alpha=0.9)
ax_bar.set_xlim(vmin, vmax)
ax_bar.set_xlabel("Mean")
ax_bar.grid(axis="x", alpha=0.25)
ax_bar.tick_params(axis="y", labelleft=False)
for y, val in enumerate(mean_by_pair):
    if pd.notna(val):
        ax_bar.text(val + 0.02, y, f"{val:.2f}", va="center", fontsize=8)
cbar = fig.colorbar(im, ax=[ax, ax_bar], shrink=0.88, pad=0.015)
cbar.set_label("Best MCC")
workload_fig_path = _save_show(fig, "fig_section4_workload_sweep_mcc_heatmap.png")
print(f"Saved figure: {workload_fig_path.relative_to(REPO_ROOT)}")


## 5. Detection Quality Across Anomaly Classes

Run this cell after the full design-space sweep. It loads the saved sweep outputs, integrates the DROOP-adaptive results when available, and displays the paper-selected detection table and figure.


In [ ]:
# Detection quality across anomaly classes.
# This cell also defines shared result-loading helpers used by the following result subsections.

from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt

PAPER_FIG_DIR = TCAD_OUT / "paper_figures"
PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = TCAD_OUT / "tcad_ablation_summary.csv"
FOLD_PATH = TCAD_OUT / "tcad_ablation_fold_results.csv"
DROOP_SUMMARY_PATH = RESULTS_ROOT / "droop_adaptive_ablation" / "droop_adaptive_summary_all_quantiles.csv"
DROOP_BEST_PATH = RESULTS_ROOT / "droop_adaptive_ablation" / "droop_adaptive_best_by_setup.csv"

if "summary" in globals() and isinstance(summary, pd.DataFrame) and not summary.empty:
    tcad_summary = summary.copy()
elif SUMMARY_PATH.exists():
    tcad_summary = pd.read_csv(SUMMARY_PATH)
else:
    raise RuntimeError("Run Section 4 first, or make sure results/notebook_run/tcad_ablation/tcad_ablation_summary.csv exists.")

fold_needed_cols = [
    "setup", "scenario", "workload", "sweep_source", "droop_adaptive_p_quantile", "window_size", "agg_mode", "lambda_res", "p_quantile",
    "n_splits", "requested_n_splits", "top_k", "n_selected_features", "weight_mode", "fixed_point_q",
    "auc_roc", "auc_pr", "f1", "bal_acc", "mcc", "fpr", "brier", "ece",
    "fp_mae", "fp_max_abs", "hw_setup_b_area_overhead_pct", "hw_idle_power_overhead_pct",
]
if "tcad_artifacts" in globals() and isinstance(tcad_artifacts, dict) and isinstance(tcad_artifacts.get("fold_results"), pd.DataFrame):
    fold_results = tcad_artifacts["fold_results"].copy()
    fold_results = fold_results[[c for c in fold_needed_cols if c in fold_results.columns]]
elif FOLD_PATH.exists():
    fold_results = pd.read_csv(FOLD_PATH, usecols=lambda c: c in fold_needed_cols)
else:
    fold_results = pd.DataFrame()

DROOP_ADAPTIVE_ALREADY_IN_TCAD_SUMMARY = (
    "sweep_source" in tcad_summary.columns
    and tcad_summary["sweep_source"].astype(str).str.contains("droop_adaptive", case=False, na=False).any()
)
if DROOP_ADAPTIVE_ALREADY_IN_TCAD_SUMMARY:
    droop_adaptive_summary = pd.DataFrame()
elif "adaptive_summary" in globals() and isinstance(adaptive_summary, pd.DataFrame) and not adaptive_summary.empty:
    droop_adaptive_summary = adaptive_summary.copy()
elif DROOP_SUMMARY_PATH.exists():
    droop_adaptive_summary = pd.read_csv(DROOP_SUMMARY_PATH)
else:
    droop_adaptive_summary = pd.DataFrame()

rank_cols = [c for c in ["mcc", "bal_acc", "auc_roc", "f1", "fpr"] if c in tcad_summary.columns]
rank_ascending = [False, False, False, False, True][: len(rank_cols)]

def best_rows(df: pd.DataFrame, group_cols: list[str], *, compact: bool = False) -> pd.DataFrame:
    tmp = df.copy()
    if compact and {"top_k", "fixed_point_q"}.issubset(tmp.columns):
        compact_df = tmp[(tmp["top_k"] <= 15) & (tmp["fixed_point_q"] >= 15)].copy()
        if not compact_df.empty:
            tmp = compact_df
    if tmp.empty:
        return tmp
    local_rank_cols = [c for c in rank_cols if c in tmp.columns]
    local_ascending = [rank_ascending[rank_cols.index(c)] for c in local_rank_cols]
    return (
        tmp.sort_values(local_rank_cols, ascending=local_ascending, kind="mergesort")
        .groupby(group_cols, as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

def save_table(df: pd.DataFrame, name: str) -> Path:
    path = PAPER_FIG_DIR / name
    df.to_csv(path, index=False)
    return path

def save_show(fig, name: str) -> Path:
    path = PAPER_FIG_DIR / name
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    display(Image(filename=str(path)))
    return path

paper_metric_cols = [
    "setup", "scenario", "sweep_source", "droop_adaptive_p_quantile", "top_k", "n_selected_features",
    "window_size", "agg_mode", "lambda_res", "weight_mode", "fixed_point_q",
    "auc_roc", "auc_pr", "f1", "bal_acc", "mcc", "fpr", "brier", "ece",
    "fp_mae", "fp_max_abs", "hw_setup_b_area_overhead_pct", "hw_idle_power_overhead_pct",
]

main_best = best_rows(tcad_summary, ["setup", "scenario"])
if "sweep_source" not in main_best.columns:
    main_best["sweep_source"] = "standard_dse"
else:
    main_best["sweep_source"] = main_best["sweep_source"].fillna("standard_dse")
if "droop_adaptive_p_quantile" not in main_best.columns:
    main_best["droop_adaptive_p_quantile"] = np.nan

if not droop_adaptive_summary.empty:
    droop_best = best_rows(droop_adaptive_summary, ["setup", "scenario"])
    droop_best["sweep_source"] = "droop_adaptive_sweep"
    droop_candidates = pd.concat([
        main_best[main_best["scenario"].astype(str).str.upper() == "DROOP"],
        droop_best,
    ], ignore_index=True, sort=False)
    paper_droop = best_rows(droop_candidates, ["setup", "scenario"])
else:
    droop_best = pd.DataFrame()
    paper_droop = main_best[main_best["scenario"].astype(str).str.upper() == "DROOP"].copy()

paper_detection = pd.concat([
    main_best[main_best["scenario"].astype(str).str.upper() != "DROOP"],
    paper_droop,
], ignore_index=True, sort=False).sort_values(["setup", "scenario"]).reset_index(drop=True)
paper_detection_cols = [c for c in paper_metric_cols if c in paper_detection.columns]
paper_detection_table = paper_detection[paper_detection_cols]
paper_detection_path = save_table(paper_detection_table, "tcad_detection_quality_paper_selected.csv")

all_best_path = save_table(main_best[[c for c in paper_metric_cols if c in main_best.columns]], "tcad_main_sweep_best_by_setup_scenario.csv")
if not droop_best.empty:
    droop_best_path = save_table(droop_best[[c for c in paper_metric_cols if c in droop_best.columns]], "tcad_droop_adaptive_best_by_setup.csv")

# Workload robustness table.
if not fold_results.empty and "workload" in fold_results.columns:
    per_workload = fold_results[fold_results["workload"].astype(str).str.upper() != "ALL"].copy()
else:
    per_workload = pd.DataFrame()

if not per_workload.empty:
    cfg_cols = [c for c in [
        "setup", "scenario", "workload", "window_size", "agg_mode", "lambda_res", "p_quantile",
        "top_k", "n_selected_features", "weight_mode", "fixed_point_q"
    ] if c in per_workload.columns]
    metric_cols = [c for c in [
        "auc_roc", "auc_pr", "f1", "bal_acc", "mcc", "fpr", "brier", "ece",
        "fp_mae", "fp_max_abs", "hw_setup_b_area_overhead_pct", "hw_idle_power_overhead_pct"
    ] if c in per_workload.columns]
    per_workload_mean = per_workload.groupby(cfg_cols, as_index=False)[metric_cols].mean(numeric_only=True)
    wl_rank_cols = [c for c in rank_cols if c in per_workload_mean.columns]
    wl_rank_ascending = [rank_ascending[rank_cols.index(c)] for c in wl_rank_cols]
    best_by_workload = (
        per_workload_mean.sort_values(wl_rank_cols, ascending=wl_rank_ascending, kind="mergesort")
        .groupby(["setup", "scenario", "workload"], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    workload_summary = (
        best_by_workload.groupby(["setup", "scenario"], as_index=False)
        .agg(
            workloads=("workload", "nunique"),
            mean_best_mcc=("mcc", "mean"),
            median_best_mcc=("mcc", "median"),
            min_best_mcc=("mcc", "min"),
            mean_best_bal_acc=("bal_acc", "mean"),
            mean_best_f1=("f1", "mean"),
            mean_best_fpr=("fpr", "mean"),
        )
    )
    best_by_workload_path = save_table(best_by_workload, "tcad_best_dse_by_setup_scenario_workload.csv")
    workload_summary_path = save_table(workload_summary, "tcad_workload_robustness_summary.csv")
else:
    best_by_workload = pd.DataFrame()
    workload_summary = pd.DataFrame()

# Detection quality figure.
plot_best = paper_detection_table.copy()
labels = [f"{r.setup}-{r.scenario}" for r in plot_best.itertuples()]
metrics_to_plot = [c for c in ["mcc", "f1", "bal_acc", "auc_roc"] if c in plot_best.columns]
fig, ax = plt.subplots(figsize=(max(8.0, 1.25 * len(labels)), 4.2))
x = np.arange(len(labels))
width = 0.18 if len(metrics_to_plot) >= 4 else 0.24
for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + (i - (len(metrics_to_plot)-1)/2) * width, plot_best[metric], width, label=metric.upper())
ax.set_title("CITADEL Detection Quality Across Anomaly Classes")
ax.set_ylabel("Metric value")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylim(0.0, 1.02)
ax.grid(axis="y", alpha=0.25)
ax.legend(ncol=min(4, len(metrics_to_plot)), frameon=False)
detection_fig_path = save_show(fig, "fig_results_detection_quality.png")

display(Markdown("### Paper-selected detection table"))
display(paper_detection_table.round(6))
if not workload_summary.empty:
    display(Markdown("### Workload robustness summary"))
    display(workload_summary.round(6))

print(f"Saved table: {paper_detection_path.relative_to(REPO_ROOT)}")
print(f"Saved table: {all_best_path.relative_to(REPO_ROOT)}")
if not droop_best.empty:
    print(f"Saved table: {droop_best_path.relative_to(REPO_ROOT)}")
if not workload_summary.empty:
    print(f"Saved table: {workload_summary_path.relative_to(REPO_ROOT)}")
print(f"Saved figure: {detection_fig_path.relative_to(REPO_ROOT)}")
print(f"All paper figures/tables are collected in: {PAPER_FIG_DIR.relative_to(REPO_ROOT)}")


## 6. Feature-Budget and Telemetry-Cost Trade-Off

This subsection shows how detection quality changes as the selected feature budget changes. It is the main figure/table pair for arguing that CITADEL reduces telemetry bandwidth while preserving anomaly sensitivity.


In [ ]:
# Feature-budget and telemetry-cost trade-off.
if "tcad_summary" not in globals():
    raise RuntimeError("Run Section 5 first; it loads the saved TCAD result tables.")

trade_sources = []
trade_summary = tcad_summary.copy()
if "sweep_source" not in trade_summary.columns:
    trade_summary["sweep_source"] = "standard_dse"
else:
    trade_summary["sweep_source"] = trade_summary["sweep_source"].fillna("standard_dse")
main_trade = (
    trade_summary.groupby(["sweep_source", "setup", "scenario", "top_k"], as_index=False)
    .agg(
        best_mcc=("mcc", "max"),
        best_bal_acc=("bal_acc", "max"),
        best_f1=("f1", "max"),
        area_overhead_pct=("hw_setup_b_area_overhead_pct", "mean"),
        idle_power_overhead_pct=("hw_idle_power_overhead_pct", "mean"),
    )
)
trade_sources.append(main_trade)

if not globals().get("DROOP_ADAPTIVE_ALREADY_IN_TCAD_SUMMARY", False) and "droop_adaptive_summary" in globals() and not droop_adaptive_summary.empty:
    droop_trade = (
        droop_adaptive_summary.groupby(["setup", "scenario", "top_k"], as_index=False)
        .agg(
            best_mcc=("mcc", "max"),
            best_bal_acc=("bal_acc", "max"),
            best_f1=("f1", "max"),
            area_overhead_pct=("hw_setup_b_area_overhead_pct", "mean"),
            idle_power_overhead_pct=("hw_idle_power_overhead_pct", "mean"),
        )
    )
    droop_trade["sweep_source"] = "droop_adaptive_sweep"
    trade_sources.append(droop_trade)

feature_budget_tradeoff = pd.concat(trade_sources, ignore_index=True, sort=False)
tradeoff_path = save_table(feature_budget_tradeoff, "tcad_feature_budget_tradeoff.csv")

fig, ax = plt.subplots(figsize=(8.5, 4.8))
for (source, setup_name, scenario_name), group in feature_budget_tradeoff.groupby(["sweep_source", "setup", "scenario"], sort=True):
    group = group.sort_values("top_k")
    line_style = "--" if "droop_adaptive" in str(source) else "-"
    alpha = 0.8 if "droop_adaptive" in str(source) else 0.95
    ax.plot(group["top_k"], group["best_mcc"], marker="o", linestyle=line_style, linewidth=2.0, alpha=alpha, label=f"{setup_name}-{scenario_name} ({source.replace('_', ' ')})")
ax.set_title("Feature-Budget Sweep: Best MCC versus Selected Telemetry Features")
ax.set_xlabel("Selected feature budget k")
ax.set_ylabel("Best MCC")
ax.set_ylim(-0.05, 1.02)
ax.grid(True, alpha=0.25)
ax.legend(ncol=2, frameon=False, fontsize=8)
tradeoff_fig_path = save_show(fig, "fig_results_feature_budget_tradeoff.png")

display(Markdown("### Feature-budget trade-off table"))
display(feature_budget_tradeoff.round(6))
print(f"Saved table: {tradeoff_path.relative_to(REPO_ROOT)}")
print(f"Saved figure: {tradeoff_fig_path.relative_to(REPO_ROOT)}")


## 7. Stable Conditional Telemetry Graph Interpretation

This subsection displays the learned stable conditional telemetry graphs and the highest-ranked CITADEL features. Use it to explain why selected features are structurally stable under benign telemetry, conditionally connected to other monitored signals, aligned with CINTAS scoring, and compatible with telemetry-cost constraints.


In [ ]:
# Causal telemetry graph interpretation.
if "PAPER_FIG_DIR" not in globals():
    raise RuntimeError("Run Section 5 first; it defines the paper artifact directory and helpers.")

causal_dir = TCAD_OUT / "causal"
rank_tables = []
if causal_dir.exists():
    for rank_path in sorted(causal_dir.glob("SETUP_*_feature_ranks.csv")):
        setup_name = rank_path.stem.split("_")[1]
        ranks = pd.read_csv(rank_path)
        ranks.insert(0, "setup", setup_name)
        rank_tables.append(ranks.head(20))

if rank_tables:
    top_causal_features = pd.concat(rank_tables, ignore_index=True)
    causal_rank_path = save_table(top_causal_features, "tcad_top_causal_features.csv")
    display(Markdown("### Top causal telemetry features"))
    display(top_causal_features.round(6))
    print(f"Saved table: {causal_rank_path.relative_to(REPO_ROOT)}")
else:
    display(Markdown("No stable conditional feature-rank files found yet. Run Section 4 first."))

graph_paths = sorted(causal_dir.glob("SETUP_*_causal_graph_and_top*.png")) if causal_dir.exists() else []
if graph_paths:
    display(Markdown("### Causal graph figures"))
    for path in graph_paths:
        print(f"Displaying: {path.relative_to(REPO_ROOT)}")
        display(Image(filename=str(path)))
else:
    display(Markdown("No causal graph PNGs found yet. They will be generated by the full design-space sweep."))


## 8. Fixed-Point CINTAS Sensitivity

This subsection shows whether fixed-point CINTAS preserves the floating-point reference score. It supports the claim that compact fixed-point arithmetic is sufficient for edge SLM scoring.


In [ ]:
# Fixed-point CINTAS sensitivity.
if "tcad_summary" not in globals():
    raise RuntimeError("Run Section 5 first; it loads the saved TCAD result tables.")

fp_base = tcad_summary.copy()
if "sweep_source" not in fp_base.columns:
    fp_base["sweep_source"] = "standard_dse"
else:
    fp_base["sweep_source"] = fp_base["sweep_source"].fillna("standard_dse")
fp_sources = [fp_base]
if not globals().get("DROOP_ADAPTIVE_ALREADY_IN_TCAD_SUMMARY", False) and "droop_adaptive_summary" in globals() and not droop_adaptive_summary.empty:
    fp_sources.append(droop_adaptive_summary.assign(sweep_source="droop_adaptive_sweep"))
fp_all = pd.concat(fp_sources, ignore_index=True, sort=False)

fixed_point_sensitivity = (
    fp_all.groupby(["sweep_source", "fixed_point_q"], as_index=False)
    .agg(
        mean_fp_mae=("fp_mae", "mean"),
        p95_fp_mae=("fp_mae", lambda x: np.nanquantile(x, 0.95)),
        mean_fp_max_abs=("fp_max_abs", "mean"),
        max_fp_max_abs=("fp_max_abs", "max"),
        mean_mcc=("mcc", "mean"),
    )
    .sort_values(["sweep_source", "fixed_point_q"])
)
fp_path = save_table(fixed_point_sensitivity, "tcad_fixed_point_sensitivity.csv")

fig, ax = plt.subplots(figsize=(7.6, 4.2))
for source, group in fixed_point_sensitivity.groupby("sweep_source", sort=True):
    group = group.sort_values("fixed_point_q")
    ax.semilogy(group["fixed_point_q"], group["p95_fp_mae"], marker="o", linewidth=2.2, label=source.replace("_", " "))
ax.set_title("Fixed-Point CINTAS Sensitivity")
ax.set_xlabel("Fractional precision Q")
ax.set_ylabel("p95 score error")
ax.grid(True, which="both", alpha=0.25)
ax.legend(frameon=False)
fp_fig_path = save_show(fig, "fig_results_fixed_point_sensitivity.png")

display(Markdown("### Fixed-point sensitivity table"))
display(fixed_point_sensitivity.round(8))
print(f"Saved table: {fp_path.relative_to(REPO_ROOT)}")
print(f"Saved figure: {fp_fig_path.relative_to(REPO_ROOT)}")


## 9. CINTAS Hardware-Cost Analysis

This subsection summarizes operator count, estimated area, power, and latency as a function of feature budget. It is the result subsection that should eventually be replaced or complemented by RTL/FPGA synthesis numbers.


In [ ]:
# CINTAS hardware-cost analysis.
if "tcad_summary" not in globals():
    raise RuntimeError("Run Section 5 first; it loads the saved TCAD result tables.")

hw_cols = [
    "top_k", "hw_add_count", "hw_mult_count", "hw_estimated_serial_cycles", "hw_area_mm2",
    "hw_power_mw", "hw_setup_b_area_overhead_pct", "hw_idle_power_overhead_pct",
    "hw_median_workload_power_overhead_pct",
]
hw_cols = [c for c in hw_cols if c in tcad_summary.columns]
hardware_cost = (
    tcad_summary[hw_cols]
    .groupby("top_k", as_index=False)
    .mean(numeric_only=True)
    .sort_values("top_k")
)
hw_path = save_table(hardware_cost, "tcad_hardware_cost_analysis.csv")

fig, ax1 = plt.subplots(figsize=(7.8, 4.2))
ax1.plot(hardware_cost["top_k"], hardware_cost["hw_setup_b_area_overhead_pct"], marker="o", linewidth=2.2, label="area overhead")
ax1.set_xlabel("Selected feature budget k")
ax1.set_ylabel("Area overhead (%)")
ax1.grid(True, alpha=0.25)
ax2 = ax1.twinx()
ax2.plot(hardware_cost["top_k"], hardware_cost["hw_idle_power_overhead_pct"], marker="s", color="tab:orange", linewidth=2.2, label="idle-power overhead")
ax2.set_ylabel("Idle-power overhead (%)")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, frameon=False, loc="upper left")
ax1.set_title("CINTAS Hardware-Cost Scaling")
hw_fig_path = save_show(fig, "fig_results_hardware_cost_scaling.png")

if "paper_detection_table" in globals():
    selected_hw_cols = [c for c in [
        "setup", "scenario", "sweep_source", "top_k", "fixed_point_q", "window_size",
        "hw_setup_b_area_overhead_pct", "hw_idle_power_overhead_pct", "mcc", "fpr"
    ] if c in paper_detection_table.columns]
    selected_hw = paper_detection_table[selected_hw_cols].copy()
    selected_hw_path = save_table(selected_hw, "tcad_selected_operating_points_hardware_cost.csv")
    display(Markdown("### Selected operating points and estimated hardware cost"))
    display(selected_hw.round(8))
    print(f"Saved table: {selected_hw_path.relative_to(REPO_ROOT)}")

display(Markdown("### Hardware-cost scaling table"))
display(hardware_cost.round(8))
print(f"Saved table: {hw_path.relative_to(REPO_ROOT)}")
print(f"Saved figure: {hw_fig_path.relative_to(REPO_ROOT)}")


## 10. Lifecycle Drift and Recalibration


In [ ]:
if "tcad_cfg" not in globals():
    cfg_path = REPO_ROOT / "configs" / f"tcad_grid_{TCAD_PRESET}.json"
    tcad_cfg = notebook_tcad_config(TCAD_PRESET, config_path=cfg_path, seed=SEED)

if "summary" in globals() and isinstance(summary, pd.DataFrame) and not summary.empty:
    tcad_summary_for_tbd = summary.copy()
elif (TCAD_OUT / "tcad_ablation_summary.csv").exists():
    tcad_summary_for_tbd = pd.read_csv(TCAD_OUT / "tcad_ablation_summary.csv")
else:
    raise RuntimeError("Run Section 4 first, or make sure the TCAD summary CSV exists.")

if "CITADEL_SHARED_FEATURE_COUNT" not in globals():
    count_a, count_b = load_telemetry_two_setups(DATA_ROOT)
    count_a = clean_and_debias_telemetry(count_a)
    count_b = clean_and_debias_telemetry(count_b)
    feat_a = drop_constant_features(count_a)
    feat_b = drop_constant_features(count_b)
    CITADEL_SHARED_FEATURE_COUNT = len(sorted(set(feat_a) & set(feat_b)))
    del count_a, count_b

lifecycle_cfg = notebook_lifecycle_config(tcad_cfg)
print(f"Lifecycle config: top_k={lifecycle_cfg.top_k}, window_size={lifecycle_cfg.window_size}, lambda_res={lifecycle_cfg.lambda_res}")
print(f"CITADEL shared feature count: {CITADEL_SHARED_FEATURE_COUNT}")

lifecycle_artifacts = notebook_run_lifecycle_drift(
    data_root=DATA_ROOT,
    out_root=LIFECYCLE_OUT,
    cfg=lifecycle_cfg,
    seed=SEED,
    threads=THREADS,
)
lifecycle_summary = lifecycle_artifacts["summary"].copy()
print(f"Lifecycle summary: {lifecycle_artifacts['summary_path'].relative_to(REPO_ROOT)}")
print(f"Lifecycle manifest: {lifecycle_artifacts['manifest_path'].relative_to(REPO_ROOT)}")
display(lifecycle_summary.head(12))

paper_tbd_path = notebook_write_paper_tbd_replacements(
    tcad_summary=tcad_summary_for_tbd,
    lifecycle_summary=lifecycle_summary,
    total_features=CITADEL_SHARED_FEATURE_COUNT,
    out_path=RESULTS_ROOT / "paper_tbd_replacements.csv",
)
print(f"Paper TBD replacements: {paper_tbd_path.relative_to(REPO_ROOT)}")
display(pd.read_csv(paper_tbd_path))


## 11. TCAD Results Gallery: Five Tables and Six Figures

This gallery is the paper-facing result view. Run it after Sections 4--10. It displays the five main TCAD tables and six main TCAD figures inline, including the stable conditional telemetry graph network, and also saves the same artifacts under `results/notebook_run/tcad_ablation/paper_figures/`. Apple portability remains supplemental and is not mixed into these main CINTAS hardware-cost claims.


In [ ]:
# TCAD results gallery: five paper tables and six paper figures.
# Run after Sections 4--10. This cell displays the main TCAD result artifacts inline.

from IPython.display import Image, Markdown, display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcdefaults()
plt.rcParams.update({
    "font.size": 8.5,
    "axes.titlesize": 10.5,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 8.0,
    "figure.titlesize": 12.0,
    "axes.linewidth": 0.8,
    "savefig.dpi": 300,
})

GALLERY_DIR = TCAD_OUT / "paper_figures"
GALLERY_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = TCAD_OUT / "tcad_ablation_summary.csv"
FOLD_PATH = TCAD_OUT / "tcad_ablation_fold_results.csv"
DROOP_SUMMARY_PATH = RESULTS_ROOT / "droop_adaptive_ablation" / "droop_adaptive_summary_all_quantiles.csv"
LIFECYCLE_SUMMARY_PATH = LIFECYCLE_OUT / "lifecycle_recalibration_summary.csv"
LIFECYCLE_SCENARIO_PATH = LIFECYCLE_OUT / "lifecycle_recalibration_by_scenario.csv"

if "summary" in globals() and isinstance(summary, pd.DataFrame) and not summary.empty:
    gallery_summary = summary.copy()
elif SUMMARY_PATH.exists():
    gallery_summary = pd.read_csv(SUMMARY_PATH)
else:
    raise RuntimeError("Run Section 4 first, or make sure tcad_ablation_summary.csv exists.")

fold_cols = [
    "setup", "scenario", "workload", "sweep_source", "droop_adaptive_p_quantile", "window_size", "agg_mode", "lambda_res", "p_quantile",
    "n_splits", "requested_n_splits", "top_k", "n_selected_features", "weight_mode", "fixed_point_q",
    "auc_roc", "auc_pr", "f1", "bal_acc", "mcc", "fpr", "brier", "ece",
    "fp_mae", "fp_max_abs", "hw_setup_b_area_overhead_pct", "hw_idle_power_overhead_pct",
]
if "tcad_artifacts" in globals() and isinstance(tcad_artifacts, dict) and isinstance(tcad_artifacts.get("fold_results"), pd.DataFrame):
    gallery_folds = tcad_artifacts["fold_results"].copy()
    gallery_folds = gallery_folds[[c for c in fold_cols if c in gallery_folds.columns]]
elif FOLD_PATH.exists():
    gallery_folds = pd.read_csv(FOLD_PATH, usecols=lambda c: c in fold_cols)
else:
    raise RuntimeError("Run Section 4 first, or make sure tcad_ablation_fold_results.csv exists.")

GALLERY_DROOP_ALREADY_IN_SUMMARY = (
    "sweep_source" in gallery_summary.columns
    and gallery_summary["sweep_source"].astype(str).str.contains("droop_adaptive", case=False, na=False).any()
)
if GALLERY_DROOP_ALREADY_IN_SUMMARY:
    gallery_droop = pd.DataFrame()
elif "adaptive_summary" in globals() and isinstance(adaptive_summary, pd.DataFrame) and not adaptive_summary.empty:
    gallery_droop = adaptive_summary.copy()
elif DROOP_SUMMARY_PATH.exists():
    gallery_droop = pd.read_csv(DROOP_SUMMARY_PATH)
else:
    gallery_droop = pd.DataFrame()

gallery_lifecycle = pd.read_csv(LIFECYCLE_SUMMARY_PATH) if LIFECYCLE_SUMMARY_PATH.exists() else pd.DataFrame()
gallery_lifecycle_by_scenario = pd.read_csv(LIFECYCLE_SCENARIO_PATH) if LIFECYCLE_SCENARIO_PATH.exists() else pd.DataFrame()

rank_cols = [c for c in ["mcc", "bal_acc", "auc_roc", "f1", "fpr"] if c in gallery_summary.columns]
rank_ascending = [False, False, False, False, True][: len(rank_cols)]

def _best_rows(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    local_rank = [c for c in rank_cols if c in df.columns]
    local_asc = [rank_ascending[rank_cols.index(c)] for c in local_rank]
    if not local_rank:
        return df.groupby(group_cols, as_index=False).head(1).reset_index(drop=True)
    return (
        df.sort_values(local_rank, ascending=local_asc, kind="mergesort", na_position="last")
        .groupby(group_cols, as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

def _save_table(df: pd.DataFrame, name: str) -> Path:
    path = GALLERY_DIR / name
    df.to_csv(path, index=False)
    return path

def _save_show(fig, name: str) -> Path:
    path = GALLERY_DIR / name
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    display(Image(filename=str(path)))
    return path

def _show_table(title: str, df: pd.DataFrame, name: str, digits: int = 6) -> Path:
    path = _save_table(df, name)
    display(Markdown(f"### {title}"))
    display(df.round(digits) if not df.empty else df)
    print(f"Saved table: {path.relative_to(REPO_ROOT)}")
    return path


def _trim_white_margin(img: np.ndarray, threshold: float = 0.985, pad: int = 12) -> np.ndarray:
    """Crop empty white margins from imported PNGs before composing paper figures."""
    arr = np.asarray(img)
    rgb = arr[..., :3] if arr.ndim == 3 else arr
    mask = np.any(rgb < threshold, axis=2) if rgb.ndim == 3 else rgb < threshold
    coords = np.argwhere(mask)
    if coords.size == 0:
        return arr
    y0, x0 = np.maximum(coords.min(axis=0) - pad, 0)
    y1, x1 = np.minimum(coords.max(axis=0) + pad + 1, arr.shape[:2])
    return arr[y0:y1, x0:x1]

# -----------------------------
# Table 1: paper-selected operating points.
# -----------------------------
main_best = _best_rows(gallery_summary, ["setup", "scenario"])
if "sweep_source" not in main_best.columns:
    main_best["sweep_source"] = "standard_dse"
else:
    main_best["sweep_source"] = main_best["sweep_source"].fillna("standard_dse")
if "droop_adaptive_p_quantile" not in main_best.columns:
    main_best["droop_adaptive_p_quantile"] = np.nan

if not gallery_droop.empty:
    droop_best = _best_rows(gallery_droop, ["setup", "scenario"])
    droop_best["sweep_source"] = "droop_adaptive_sweep"
    droop_candidates = pd.concat([
        main_best[main_best["scenario"].astype(str).str.upper() == "DROOP"],
        droop_best,
    ], ignore_index=True, sort=False)
    paper_droop = _best_rows(droop_candidates, ["setup", "scenario"])
else:
    paper_droop = main_best[main_best["scenario"].astype(str).str.upper() == "DROOP"].copy()

paper_selected = pd.concat([
    main_best[main_best["scenario"].astype(str).str.upper() != "DROOP"],
    paper_droop,
], ignore_index=True, sort=False).sort_values(["setup", "scenario"]).reset_index(drop=True)

selected_cols = [
    "setup", "scenario", "sweep_source", "top_k", "window_size", "agg_mode", "lambda_res",
    "weight_mode", "fixed_point_q", "auc_roc", "auc_pr", "f1", "bal_acc", "mcc", "fpr",
    "fp_mae", "fp_max_abs", "hw_setup_b_area_overhead_pct", "hw_idle_power_overhead_pct",
]
selected_cols = [c for c in selected_cols if c in paper_selected.columns]
table1 = paper_selected[selected_cols].copy()
_show_table("Table 1. Paper-selected CITADEL operating points", table1, "gallery_table1_selected_operating_points.csv")

# -----------------------------
# Table 2: workload robustness.
# -----------------------------
per_workload = gallery_folds[gallery_folds["workload"].astype(str).str.upper() != "ALL"].copy()
cfg_cols = [c for c in [
    "setup", "scenario", "workload", "window_size", "agg_mode", "lambda_res", "p_quantile",
    "top_k", "n_selected_features", "weight_mode", "fixed_point_q"
] if c in per_workload.columns]
metric_cols = [c for c in [
    "auc_roc", "auc_pr", "f1", "bal_acc", "mcc", "fpr", "brier", "ece",
    "fp_mae", "fp_max_abs", "hw_setup_b_area_overhead_pct", "hw_idle_power_overhead_pct"
] if c in per_workload.columns]
per_workload_mean = per_workload.groupby(cfg_cols, as_index=False)[metric_cols].mean(numeric_only=True)
workload_best = _best_rows(per_workload_mean, ["setup", "scenario", "workload"])
table2 = (
    workload_best.groupby(["setup", "scenario"], as_index=False)
    .agg(
        workloads=("workload", "nunique"),
        mean_best_mcc=("mcc", "mean"),
        median_best_mcc=("mcc", "median"),
        min_best_mcc=("mcc", "min"),
        max_best_mcc=("mcc", "max"),
        mean_best_fpr=("fpr", "mean"),
        max_best_fpr=("fpr", "max"),
        median_top_k=("top_k", "median"),
        median_window_size=("window_size", "median"),
    )
)
_show_table("Table 2. Workload-level robustness summary", table2, "gallery_table2_workload_robustness.csv")

# -----------------------------
# Table 3: feature-budget trade-off.
# -----------------------------
table3 = (
    gallery_summary.groupby(["setup", "scenario", "top_k"], as_index=False)
    .agg(
        best_mcc=("mcc", "max"),
        best_bal_acc=("bal_acc", "max"),
        min_fpr=("fpr", "min"),
        mean_area_overhead_pct=("hw_setup_b_area_overhead_pct", "mean"),
        mean_idle_power_overhead_pct=("hw_idle_power_overhead_pct", "mean"),
    )
    .sort_values(["setup", "scenario", "top_k"])
)
_show_table("Table 3. Feature-budget and telemetry-cost trade-off", table3, "gallery_table3_feature_budget_tradeoff.csv")

# -----------------------------
# Table 4: stable conditional telemetry graph top features.
# -----------------------------
feature_rank_rows = []
for rank_path in sorted((TCAD_OUT / "causal").glob("SETUP_*_feature_ranks.csv")):
    setup_name = rank_path.stem.split("_")[1]
    ranks = pd.read_csv(rank_path).head(12).copy()
    ranks.insert(0, "rank", np.arange(1, len(ranks) + 1))
    ranks.insert(0, "setup", setup_name)
    feature_rank_rows.append(ranks)
if feature_rank_rows:
    table4_raw = pd.concat(feature_rank_rows, ignore_index=True)
else:
    table4_raw = pd.DataFrame(columns=["setup", "rank", "feature", "domain", "importance_score"])
table4_cols = [
    "setup", "rank", "feature", "domain", "importance_score",
    "edge_stability", "conditional_dependence", "cias_alignment",
    "graph_centrality", "anomaly_alignment",
]
# Keep only graph-ranking columns that are actually populated; this avoids paper tables with blank cost fields.
table4_keep = []
for col in table4_cols:
    if col in table4_raw.columns and (col in {"setup", "rank", "feature"} or not table4_raw[col].isna().all()):
        table4_keep.append(col)
table4 = table4_raw[table4_keep]
_show_table("Table 4. Stable conditional telemetry graph top features", table4, "gallery_table4_stable_graph_top_features.csv")

# -----------------------------
# Table 5: lifecycle recalibration summary, with deployment columns.
# -----------------------------
if not gallery_lifecycle.empty:
    table5_cols = [
        "setup", "method", "top_k", "window_size", "rank_jaccard", "n_eval_windows", "tpr", "fpr",
        "mcc", "benign_score_mean_delta", "n_initial_features", "n_recalibrated_features",
    ]
    table5 = gallery_lifecycle[[c for c in table5_cols if c in gallery_lifecycle.columns]].copy()
else:
    table5 = table1[[c for c in ["setup", "scenario", "top_k", "fixed_point_q", "mcc", "fpr", "fp_mae", "fp_max_abs", "hw_setup_b_area_overhead_pct", "hw_idle_power_overhead_pct"] if c in table1.columns]].copy()
_show_table("Table 5. Lifecycle recalibration and deployment feasibility", table5, "gallery_table5_lifecycle_deployment.csv")

# -----------------------------
# Figure 1: full DSE heatmap over k and N.
# -----------------------------
dse_envelope = (
    gallery_summary.groupby(["setup", "scenario", "top_k", "window_size"], as_index=False)
    .agg(best_mcc=("mcc", "max"), best_bal_acc=("bal_acc", "max"), min_fpr=("fpr", "min"))
)
pairs = sorted(set(zip(dse_envelope["setup"].astype(str), dse_envelope["scenario"].astype(str))))
ncols = 2 if len(pairs) > 1 else 1
nrows = int(np.ceil(max(len(pairs), 1) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 3.2 * nrows), squeeze=False)
vals = dse_envelope["best_mcc"].replace([np.inf, -np.inf], np.nan).dropna()
vmin = min(-0.05, float(vals.min())) if not vals.empty else -0.05
vmax = 1.0
last_im = None
for ax, (setup_name, scenario_name) in zip(axes.flat, pairs):
    sub = dse_envelope[(dse_envelope["setup"].astype(str) == setup_name) & (dse_envelope["scenario"].astype(str) == scenario_name)].copy()
    pivot = sub.pivot_table(index="top_k", columns="window_size", values="best_mcc", aggfunc="max").sort_index(ascending=False)
    last_im = ax.imshow(pivot.to_numpy(dtype=float), aspect="auto", cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(f"Setup {setup_name} / {scenario_name}", pad=4)
    ax.set_xlabel("Decision block N", labelpad=2)
    ax.set_ylabel("Feature budget k", labelpad=2)
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels([str(int(x)) for x in pivot.index])
    xt = np.arange(len(pivot.columns))
    keep = max(1, int(np.ceil(len(xt) / 8)))
    ax.set_xticks(xt[::keep])
    ax.set_xticklabels([str(int(x)) for x in pivot.columns[::keep]], rotation=35, ha="right")
    if not sub.empty and sub["best_mcc"].notna().any():
        best = sub.sort_values(["best_mcc", "best_bal_acc", "min_fpr"], ascending=[False, False, True], kind="mergesort").iloc[0]
        y = list(pivot.index).index(best["top_k"])
        x = list(pivot.columns).index(best["window_size"])
        ax.scatter([x], [y], marker="*", s=170, c="white", edgecolors="black", linewidths=0.8)
        ax.text(x, y, f" {best['best_mcc']:.2f}", va="center", ha="left", fontsize=8, color="white", fontweight="bold")
for ax in axes.flat[len(pairs):]:
    ax.axis("off")
fig.suptitle("Figure 1. Full DSE Accuracy Envelope", y=0.985, fontweight="bold")
fig.subplots_adjust(left=0.08, right=0.86, bottom=0.10, top=0.88, hspace=0.52, wspace=0.30)
if last_im is not None:
    cax = fig.add_axes([0.89, 0.17, 0.018, 0.64])
    cbar = fig.colorbar(last_im, cax=cax)
    cbar.set_label("Best MCC", labelpad=3)
fig1_path = _save_show(fig, "gallery_fig1_full_dse_heatmap.png")
print(f"Saved figure: {fig1_path.relative_to(REPO_ROOT)}")

# -----------------------------
# Figure 2: detection quality bars.
# -----------------------------
metrics_to_plot = [c for c in ["mcc", "f1", "bal_acc", "auc_roc"] if c in table1.columns]
labels = [f"{r.setup}-{r.scenario}" for r in table1.itertuples()]
x = np.arange(len(labels))
width = 0.18 if len(metrics_to_plot) >= 4 else 0.24
fig, ax = plt.subplots(figsize=(max(8.2, 1.25 * len(labels)), 4.2))
for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + (i - (len(metrics_to_plot)-1)/2) * width, table1[metric], width, label=metric.upper())
ax.set_title("Figure 2. Detection Quality Across Anomaly Classes", fontweight="bold")
ax.set_ylabel("Metric value")
ax.set_ylim(0.0, 1.02)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.grid(axis="y", alpha=0.25)
ax.legend(ncol=min(4, len(metrics_to_plot)), frameon=False)
fig2_path = _save_show(fig, "gallery_fig2_detection_quality.png")
print(f"Saved figure: {fig2_path.relative_to(REPO_ROOT)}")

# -----------------------------
# Figure 3: workload robustness heatmap.
# -----------------------------
workload_best = workload_best.copy()
workload_best["setup_scenario"] = workload_best["setup"].astype(str) + "-" + workload_best["scenario"].astype(str)
workload_order = workload_best.groupby("workload")["mcc"].mean().sort_values(ascending=False).index.tolist()
pair_order = workload_best.groupby("setup_scenario")["mcc"].mean().sort_values(ascending=True).index.tolist()
heat = workload_best.pivot_table(index="setup_scenario", columns="workload", values="mcc", aggfunc="max").reindex(index=pair_order, columns=workload_order)
fig = plt.figure(figsize=(max(10.0, 0.62 * len(workload_order)), max(4.0, 0.55 * len(pair_order))))
grid = fig.add_gridspec(1, 2, width_ratios=[5.0, 1.15], wspace=0.04)
ax = fig.add_subplot(grid[0, 0])
ax_bar = fig.add_subplot(grid[0, 1], sharey=ax)
im = ax.imshow(heat.to_numpy(dtype=float), aspect="auto", cmap="magma", vmin=vmin, vmax=vmax)
ax.set_title("Figure 3. Workload Sweep Robustness", fontweight="bold")
ax.set_xlabel("Workload")
ax.set_ylabel("Setup / anomaly")
ax.set_xticks(np.arange(len(workload_order)))
ax.set_xticklabels(workload_order, rotation=45, ha="right")
ax.set_yticks(np.arange(len(pair_order)))
ax.set_yticklabels(pair_order)
for y in range(heat.shape[0]):
    for x_i in range(heat.shape[1]):
        val = heat.iloc[y, x_i]
        if pd.notna(val):
            ax.text(x_i, y, f"{val:.2f}", ha="center", va="center", fontsize=7, color="white" if val < 0.62 else "black")
mean_by_pair = heat.mean(axis=1, skipna=True)
ax_bar.barh(np.arange(len(pair_order)), mean_by_pair.to_numpy(dtype=float), color="#4c78a8", alpha=0.9)
ax_bar.set_xlim(vmin, vmax)
ax_bar.set_xlabel("Mean")
ax_bar.grid(axis="x", alpha=0.25)
ax_bar.tick_params(axis="y", labelleft=False)
for y, val in enumerate(mean_by_pair):
    if pd.notna(val):
        ax_bar.text(val + 0.02, y, f"{val:.2f}", va="center", fontsize=8)
cbar = fig.colorbar(im, ax=[ax, ax_bar], shrink=0.88, pad=0.015)
cbar.set_label("Best MCC")
fig3_path = _save_show(fig, "gallery_fig3_workload_heatmap.png")
print(f"Saved figure: {fig3_path.relative_to(REPO_ROOT)}")

# -----------------------------
# Figure 4: stable conditional graph network.
# -----------------------------
graph_paths = sorted((TCAD_OUT / "causal").glob("SETUP_*_causal_graph_and_top*.png"))
if graph_paths:
    n_graphs = len(graph_paths)
    fig, axes = plt.subplots(1, n_graphs, figsize=(5.7 * n_graphs, 3.35), squeeze=False)
    for ax, path in zip(axes.flat, graph_paths):
        img = _trim_white_margin(plt.imread(path), threshold=0.985, pad=18)
        ax.imshow(img)
        ax.axis("off")
        setup_label = path.stem.replace("_causal_graph_and_top", " top-").replace("SETUP_", "Setup ")
        ax.set_title(setup_label, pad=3, fontweight="bold")
    for ax in axes.flat[len(graph_paths):]:
        ax.axis("off")
else:
    fig, ax = plt.subplots(figsize=(8.0, 2.4))
    ax.axis("off")
    ax.text(0.5, 0.5, "Run Section 4 or Section 7 to generate stable conditional telemetry graph PNGs.", ha="center", va="center", wrap=True)
fig.suptitle("Figure 4. Stable Conditional Telemetry Graph Network", y=0.985, fontweight="bold")
fig.subplots_adjust(left=0.01, right=0.99, bottom=0.02, top=0.88, wspace=0.04)
fig4_path = _save_show(fig, "gallery_fig4_stable_graph_network.png")
print(f"Saved figure: {fig4_path.relative_to(REPO_ROOT)}")

# -----------------------------
# Figure 5: stable graph feature map.
# -----------------------------
plot_features = table4.copy().head(24)
if plot_features.empty:
    fig, ax = plt.subplots(figsize=(8, 2.4))
    ax.axis("off")
    ax.text(0.5, 0.5, "Run Section 7 to generate stable graph feature ranks.", ha="center", va="center")
else:
    domain_palette = {"CORE": "#4c78a8", "MEMORY": "#f58518", "SENSOR": "#54a24b", "OTHER": "#b279a2"}
    plot_features["label"] = plot_features["setup"].astype(str) + ": " + plot_features["feature"].astype(str).str.slice(0, 42)
    plot_features = plot_features.iloc[::-1]
    fig, ax = plt.subplots(figsize=(9.4, max(5.0, 0.28 * len(plot_features))))
    colors = [domain_palette.get(str(d), "#777777") for d in plot_features.get("domain", "OTHER")]
    ax.barh(np.arange(len(plot_features)), plot_features["importance_score"].fillna(0.0), color=colors, alpha=0.84)
    if "edge_stability" in plot_features.columns and plot_features["edge_stability"].notna().any():
        ax.scatter(plot_features["edge_stability"].fillna(0.0), np.arange(len(plot_features)), s=42, color="black", label="edge stability", zorder=3)
    ax.set_yticks(np.arange(len(plot_features)))
    ax.set_yticklabels(plot_features["label"], fontsize=8)
    ax.set_xlabel("Normalized score")
    ax.set_title("Figure 5. Stable Conditional Feature Compass", fontweight="bold")
    ax.grid(axis="x", alpha=0.25)
    handles = [plt.Line2D([0], [0], color=color, lw=7, label=domain) for domain, color in domain_palette.items()]
    if "edge_stability" in plot_features.columns and plot_features["edge_stability"].notna().any():
        handles.append(plt.Line2D([0], [0], marker="o", color="black", lw=0, label="edge stability"))
    ax.legend(handles=handles, frameon=False, loc="lower right")
fig5_path = _save_show(fig, "gallery_fig5_stable_feature_compass.png")
print(f"Saved figure: {fig5_path.relative_to(REPO_ROOT)}")

# -----------------------------
# Figure 6: implementation and lifecycle feasibility.
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.1))
fp = (
    gallery_summary.groupby("fixed_point_q", as_index=False)
    .agg(p95_fp_mae=("fp_mae", lambda x: np.nanquantile(x, 0.95)), mean_fp_mae=("fp_mae", "mean"), mean_mcc=("mcc", "mean"))
    .sort_values("fixed_point_q")
)
axes[0].semilogy(fp["fixed_point_q"], fp["p95_fp_mae"], marker="o", linewidth=2.2, label="p95 MAE")
axes[0].semilogy(fp["fixed_point_q"], fp["mean_fp_mae"], marker="s", linewidth=2.0, label="mean MAE")
axes[0].set_title("Fixed-point sensitivity")
axes[0].set_xlabel("Q")
axes[0].set_ylabel("Score error")
axes[0].grid(True, which="both", alpha=0.25)
axes[0].legend(frameon=False)

hw = (
    gallery_summary.groupby("top_k", as_index=False)
    .agg(area=("hw_setup_b_area_overhead_pct", "mean"), power=("hw_idle_power_overhead_pct", "mean"), best_mcc=("mcc", "max"))
    .sort_values("top_k")
)
axes[1].plot(hw["top_k"], hw["area"], marker="o", linewidth=2.2, label="area overhead")
axes[1].plot(hw["top_k"], hw["power"], marker="s", linewidth=2.2, label="idle-power overhead")
axes[1].set_title("Hardware-cost scaling")
axes[1].set_xlabel("Feature budget k")
axes[1].set_ylabel("Overhead (%)")
axes[1].grid(True, alpha=0.25)
axes[1].legend(frameon=False)

if not gallery_lifecycle.empty and {"setup", "method", "fpr", "tpr"}.issubset(gallery_lifecycle.columns):
    life_plot = gallery_lifecycle.copy()
    life_plot["label"] = life_plot["setup"].astype(str) + "-" + life_plot["method"].astype(str)
    axes[2].scatter(life_plot["fpr"], life_plot["tpr"], s=90, c=np.arange(len(life_plot)), cmap="tab10", edgecolors="black", linewidths=0.6)
    for row in life_plot.itertuples():
        axes[2].text(row.fpr + 0.002, row.tpr, str(row.label), fontsize=7, va="center")
    axes[2].set_xlim(left=0.0)
    axes[2].set_ylim(0.0, 1.02)
    axes[2].set_xlabel("Benign FPR")
    axes[2].set_ylabel("Anomaly TPR")
    axes[2].set_title("Lifecycle recalibration")
    axes[2].grid(True, alpha=0.25)
else:
    axes[2].axis("off")
    axes[2].text(0.5, 0.5, "Run Section 10 for lifecycle drift/recalibration results.", ha="center", va="center", wrap=True)
fig.suptitle("Figure 6. Deployment Feasibility: Fixed-Point, Hardware Cost, and Lifecycle", y=1.04, fontsize=13, fontweight="bold")
fig6_path = _save_show(fig, "gallery_fig6_deployment_feasibility.png")
print(f"Saved figure: {fig6_path.relative_to(REPO_ROOT)}")

manifest = pd.DataFrame([
    {"type": "table", "paper_id": "Table 1", "artifact": "gallery_table1_selected_operating_points.csv"},
    {"type": "table", "paper_id": "Table 2", "artifact": "gallery_table2_workload_robustness.csv"},
    {"type": "table", "paper_id": "Table 3", "artifact": "gallery_table3_feature_budget_tradeoff.csv"},
    {"type": "table", "paper_id": "Table 4", "artifact": "gallery_table4_stable_graph_top_features.csv"},
    {"type": "table", "paper_id": "Table 5", "artifact": "gallery_table5_lifecycle_deployment.csv"},
    {"type": "figure", "paper_id": "Figure 1", "artifact": "gallery_fig1_full_dse_heatmap.png"},
    {"type": "figure", "paper_id": "Figure 2", "artifact": "gallery_fig2_detection_quality.png"},
    {"type": "figure", "paper_id": "Figure 3", "artifact": "gallery_fig3_workload_heatmap.png"},
    {"type": "figure", "paper_id": "Figure 4", "artifact": "gallery_fig4_stable_graph_network.png"},
    {"type": "figure", "paper_id": "Figure 5", "artifact": "gallery_fig5_stable_feature_compass.png"},
    {"type": "figure", "paper_id": "Figure 6", "artifact": "gallery_fig6_deployment_feasibility.png"},
])
manifest_path = _save_table(manifest, "gallery_manifest_5tables_6figures.csv")
legacy_manifest_path = _save_table(manifest, "gallery_manifest_5tables_5figures.csv")
display(Markdown("### Gallery Manifest"))
display(manifest)
print(f"Gallery manifest: {manifest_path.relative_to(REPO_ROOT)}")
print(f"Legacy manifest alias: {legacy_manifest_path.relative_to(REPO_ROOT)}")


## 12. Reproducibility Check


In [ ]:
if RUN_REPEAT_CHECK:
    repeat_artifacts = notebook_run_tcad_ablation(
        data_root=DATA_ROOT,
        out_root=TCAD_REPEAT_OUT,
        cfg=tcad_cfg,
        seed=SEED,
        threads=THREADS,
    )
    repeat_summary = repeat_artifacts["summary"].copy()
    same = summary.equals(repeat_summary)
    print(f"Repeated TCAD summary equals first run: {same}")
    if not same:
        diff_cols = [c for c in summary.columns if not summary[c].equals(repeat_summary[c])]
        raise AssertionError(f"Reproducibility check failed; differing columns: {diff_cols}")
else:
    print("RUN_REPEAT_CHECK=False, skipped repeat execution.")


## 13. Manifest And Artifact Audit


In [ ]:
def load_manifest(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

manifest_specs = [
    ("TCAD full design-space sweep", TCAD_OUT / "run_manifest.json"),
    ("Lifecycle drift and recalibration", LIFECYCLE_OUT / "run_manifest.json"),
]
for path in sorted((RESULTS_ROOT / "droop_adaptive_ablation").glob("p*/run_manifest.json")):
    manifest_specs.append((f"DROOP-adaptive sweep {path.parent.name}", path))

records = []
for run_name, path in manifest_specs:
    if path.exists():
        manifest = load_manifest(path)
        records.append({
            "run": run_name,
            "manifest": str(path.relative_to(REPO_ROOT)),
            "status": "OK",
            "git_commit": manifest.get("git_commit"),
            "data_files": len(manifest.get("data_files", [])),
            "artifacts": len(manifest.get("artifacts", [])),
            "python": manifest.get("python_version"),
        })
    else:
        records.append({
            "run": run_name,
            "manifest": str(path.relative_to(REPO_ROOT)),
            "status": "MISSING",
            "git_commit": None,
            "data_files": 0,
            "artifacts": 0,
            "python": None,
        })

manifest_overview = pd.DataFrame(records)
display(manifest_overview)

if (TCAD_OUT / "run_manifest.json").exists():
    tcad_manifest = load_manifest(TCAD_OUT / "run_manifest.json")
    display(pd.DataFrame(tcad_manifest.get("data_files", [])).head())


## 14. Supplemental Hardware-Cost Artifact Summary


In [ ]:

costs = OperatorCosts.from_csv(REPO_ROOT / "hardware" / "cintas_operator_costs.csv")
operator_table = pd.DataFrame([
    {"operator": "add", "area_mm2": costs.add_area_mm2, "power_mw_at_1ghz": costs.add_power_mw_at_1ghz, "delay_ps": costs.add_delay_ps, "cycles": costs.add_cycles},
    {"operator": "mult", "area_mm2": costs.mult_area_mm2, "power_mw_at_1ghz": costs.mult_power_mw_at_1ghz, "delay_ps": costs.mult_delay_ps, "cycles": costs.mult_cycles},
])
display(operator_table)

display(compute_tableIII_setupB(n_features=15, frequency_ghz=1.0))

hardware_cols = [
    "setup", "scenario", "top_k", "n_selected_features", "fixed_point_q",
    "hw_area_mm2", "hw_power_mw", "hw_setup_b_area_overhead_pct",
    "hw_idle_power_overhead_pct", "hw_add_count", "hw_mult_count",
]
display(summary[hardware_cols].drop_duplicates().head(12))


## 15. FPGA/RTL Laptop Workflow


In [ ]:
tools = []
for name, command in [
    ("verilator", ["verilator", "--version"]),
    ("iverilog", ["iverilog", "-V"]),
    ("yosys", ["yosys", "-V"]),
    ("gtkwave", ["gtkwave", "--version"]),
]:
    exe = shutil.which(command[0])
    row = {"tool": name, "available": exe is not None, "path": exe or ""}
    if exe:
        try:
            completed = subprocess.run(command, capture_output=True, text=True, timeout=10)
            first_line = (completed.stdout or completed.stderr).splitlines()[0] if (completed.stdout or completed.stderr) else ""
            row["version"] = first_line
        except Exception as exc:
            row["version"] = f"version check failed: {exc}"
    else:
        row["version"] = "not installed"
    tools.append(row)

display(pd.DataFrame(tools))

## 16. Export Fixed-Point Golden Vectors For RTL


In [ ]:
FPGA_OUT.mkdir(parents=True, exist_ok=True)
selected_feature_path = TCAD_OUT / "tcad_selected_features.csv"
if not selected_feature_path.exists():
    raise RuntimeError("Run Section 4 first, or make sure tcad_selected_features.csv exists.")

selected_features_df = pd.read_csv(selected_feature_path)
setup_a_candidates = selected_features_df[selected_features_df["setup"].astype(str).str.upper() == "A"].copy()
if setup_a_candidates.empty:
    raise RuntimeError("No Setup A selected-feature rows found in tcad_selected_features.csv.")

# Prefer the paper-selected Setup A operating point when Section 5 has been run.
if "paper_detection_table" in globals() and isinstance(paper_detection_table, pd.DataFrame):
    target = paper_detection_table[paper_detection_table["setup"].astype(str).str.upper() == "A"].copy()
    target = target[target["sweep_source"].astype(str) == "main_full_sweep"] if "sweep_source" in target.columns else target
    if not target.empty:
        t = target.sort_values(["mcc", "bal_acc", "f1", "fpr"], ascending=[False, False, False, True], kind="mergesort").iloc[0]
        for col in ["top_k", "lambda_res", "agg_mode", "weight_mode", "fixed_point_q"]:
            if col in setup_a_candidates.columns and col in t.index:
                setup_a_candidates = setup_a_candidates[setup_a_candidates[col].astype(str) == str(t[col])]

if setup_a_candidates.empty:
    setup_a_candidates = selected_features_df[selected_features_df["setup"].astype(str).str.upper() == "A"].copy()

setup_a_candidates = setup_a_candidates.sort_values(["top_k", "fixed_point_q"], ascending=[True, False], kind="mergesort")
selected_row = setup_a_candidates.iloc[0]
features = [f.strip() for f in str(selected_row["features"]).split(",") if f.strip()]

setup_a_raw, _ = load_telemetry_two_setups(DATA_ROOT)
setup_a = clean_and_debias_telemetry(setup_a_raw)
features = [f for f in features if f in setup_a.columns]
if not features:
    raise RuntimeError("Selected feature list does not match Setup A telemetry columns.")

lambda_res = float(selected_row.get("lambda_res", 0.5))
weight_mode = str(selected_row.get("weight_mode", "uniform"))
q_format = int(selected_row.get("fixed_point_q", 15))
model = fit_cintas_from_benign(setup_a, features, lambda_res=lambda_res, weight_mode=weight_mode)
fixed = FixedPointCINTAS.from_float_model(model, FixedPointConfig(q=q_format))

subset = setup_a[model.feature_cols].head(32).copy()
score_float, e1_float, e2_float = model.score_dataframe(subset)
score_q = fixed.score_dataframe(subset)

golden = subset.copy()
golden.insert(0, "sample_index", range(len(golden)))
golden["expected_score_q"] = score_q
golden["expected_score_float"] = score_float
golden["expected_e1_float"] = e1_float
golden["expected_e2_float"] = e2_float
golden_path = FPGA_OUT / f"cintas_setupA_q{q_format}_golden_vectors.csv"
golden.to_csv(golden_path, index=False)

print(f"Golden vectors: {golden_path.relative_to(REPO_ROOT)}")
print(f"Source selected features: {selected_feature_path.relative_to(REPO_ROOT)}")
print(f"Features exported: {len(model.feature_cols)}; lambda={lambda_res}; weight_mode={weight_mode}; q={q_format}")
display(golden.head())


## 17. Merge Future RTL/FPGA Results Into The TCAD Table


In [ ]:
rtl_summary_candidates = [
    RTL_SWEEP_OUT / "rtl_resource_summary.csv",
    REPO_ROOT / "results" / "rtl_sweep" / "rtl_resource_summary.csv",
]
existing = next((p for p in rtl_summary_candidates if p.exists()), None)

if existing is None:
    RTL_SWEEP_OUT.mkdir(parents=True, exist_ok=True)
    template = pd.DataFrame([
        {
            "setup": "A",
            "top_k": int(summary["top_k"].iloc[0]),
            "fixed_point_q": int(summary["fixed_point_q"].iloc[0]),
            "luts": None,
            "ffs": None,
            "dsps": None,
            "brams": None,
            "fmax_mhz": None,
            "latency_cycles": None,
            "energy_per_block_nj": None,
            "status": "fill after RTL simulation/synthesis",
        }
    ])
    existing = RTL_SWEEP_OUT / "rtl_resource_summary.csv"
    template.to_csv(existing, index=False)
    print(f"Created RTL summary template: {existing.relative_to(REPO_ROOT)}")

rtl_summary = pd.read_csv(existing)
print(f"RTL summary source: {existing.relative_to(REPO_ROOT)}")
display(rtl_summary.head())

merge_keys = [key for key in ["setup", "top_k", "fixed_point_q"] if key in rtl_summary.columns and key in summary.columns]
if merge_keys:
    merged = summary.merge(rtl_summary, on=merge_keys, how="left")
    display(merged.head())
else:
    print("RTL summary does not yet share merge keys with the TCAD summary.")

## 18. Supplemental Limited-Observability Apple Case Study

This supplemental study evaluates CITADEL as a portability and observability ablation on Apple M2 Pro host-visible telemetry. It uses the same benign calibration, stable conditional graph ranking, top-k feature budgets, and CINTAS block scoring, but it is **not** used for CINTAS area, power, or RTL/FPGA hardware-cost claims. Treat the result as evidence about observability limits when low-level hardware counters are unavailable.


In [ ]:
# Supplemental limited-observability Apple case study.
# This is an observability/portability ablation, not part of CINTAS area/power claims.

from dataclasses import dataclass
from IPython.display import Image, Markdown, display

APPLE_OUT = RESULTS_ROOT / "apple_limited_observability"
APPLE_FIG_DIR = APPLE_OUT / "paper_figures"
APPLE_OUT.mkdir(parents=True, exist_ok=True)
APPLE_FIG_DIR.mkdir(parents=True, exist_ok=True)

APPLE_SCENARIOS = ("ATOMIC", "BRANCH", "CACHE", "MEMBW", "TLB")
APPLE_FEATURE_BUDGETS = (5, 8, 10, 15)
APPLE_WINDOW_SIZES = (50, 100, 250)
APPLE_LAMBDA_VALUES = (0.25, 0.5, 0.75)
APPLE_AGG_MODES = ("max", "mean")
APPLE_WEIGHT_MODES = ("uniform", "inv_var")
APPLE_FIXED_POINT_Q = 15
APPLE_N_SPLITS = 3
APPLE_P_QUANTILE = 0.99
APPLE_GRAPH_BOOTSTRAPS = 4
APPLE_GRAPH_STABILITY_THRESHOLD = 0.50
APPLE_GRAPH_SUBSAMPLE_FRAC = 0.70
APPLE_DROP_TIME_COLUMNS = {
    "idx", "timestamp", "ts_unix_s", "t_rel_s", "uptime_s",
    "first_sample_time_ns", "last_sample_time_ns", "sample_span_ns",
}

@dataclass(frozen=True)
class AppleObservabilityConfig:
    scenarios_eval: tuple[str, ...] = APPLE_SCENARIOS
    feature_budgets: tuple[int, ...] = APPLE_FEATURE_BUDGETS
    window_sizes: tuple[int, ...] = APPLE_WINDOW_SIZES
    lambda_res_values: tuple[float, ...] = APPLE_LAMBDA_VALUES
    agg_modes: tuple[str, ...] = APPLE_AGG_MODES
    weight_modes: tuple[str, ...] = APPLE_WEIGHT_MODES
    fixed_point_q: int = APPLE_FIXED_POINT_Q
    n_splits: int = APPLE_N_SPLITS
    p_quantile: float = APPLE_P_QUANTILE
    graph_bootstraps: int = APPLE_GRAPH_BOOTSTRAPS
    graph_subsample_frac: float = APPLE_GRAPH_SUBSAMPLE_FRAC
    graph_stability_threshold: float = APPLE_GRAPH_STABILITY_THRESHOLD
    seed: int = SEED

apple_cfg = AppleObservabilityConfig()


def _parse_apple_file(path: Path) -> dict[str, str]:
    rel = path.relative_to(APPLE_DATA_ROOT)
    tier = rel.parts[0]
    workload_scenario = rel.parts[1]
    if "__" not in workload_scenario:
        raise ValueError(f"Cannot parse Apple workload/scenario from {path}")
    workload, scenario = workload_scenario.split("__", 1)
    stem = path.stem.lower()
    view = "core" if "core" in stem else "full"
    observability = f"{tier}_{view}"
    scenario_u = scenario.upper()
    scenario_u = "BENIGN" if scenario_u == "NOMINAL" else scenario_u
    return {
        "tier": tier,
        "view": view,
        "observability": observability,
        "workload": workload.upper(),
        "scenario": scenario_u,
    }


def _load_apple_observability_frames(root: Path) -> dict[str, pd.DataFrame]:
    files = sorted(Path(root).rglob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No Apple telemetry CSVs found under {root}")
    frames: dict[str, list[pd.DataFrame]] = {}
    for path in files:
        meta = _parse_apple_file(path)
        df = _sanitize_telemetry_columns(pd.read_csv(path))
        # Keep a monotonic time index but remove absolute/wall-clock columns from feature candidates.
        if "idx" in df.columns:
            time_idx = pd.to_numeric(df["idx"], errors="coerce")
            if time_idx.isna().any():
                time_idx = time_idx.fillna(pd.Series(np.arange(len(df)), index=df.index))
            df["time_idx"] = time_idx.astype(int)
        else:
            df["time_idx"] = np.arange(len(df), dtype=int)
        drop_cols = [c for c in APPLE_DROP_TIME_COLUMNS if c in df.columns]
        if drop_cols:
            df = df.drop(columns=drop_cols)
        obs = meta["observability"].upper()
        df["setup"] = obs
        df["observability"] = meta["observability"]
        df["tier"] = meta["tier"]
        df["view"] = meta["view"]
        df["scenario"] = meta["scenario"]
        df["workload"] = meta["workload"]
        df["is_anom"] = 0 if meta["scenario"] == "BENIGN" else 1
        df["label"] = df["is_anom"]
        frames.setdefault(obs, []).append(df)
    return {
        obs: pd.concat(parts, ignore_index=True).sort_values(["workload", "scenario", "time_idx"]).reset_index(drop=True)
        for obs, parts in frames.items()
    }


def _save_apple_table(df: pd.DataFrame, name: str) -> Path:
    path = APPLE_OUT / name
    df.to_csv(path, index=False)
    return path


def _save_apple_fig(fig, name: str) -> Path:
    path = APPLE_FIG_DIR / name
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    display(Image(filename=str(path)))
    return path

apple_progress = progress_start("Apple limited-observability", f"root={APPLE_DATA_ROOT.relative_to(REPO_ROOT)}")
apple_frames = _load_apple_observability_frames(APPLE_DATA_ROOT)
progress_update("Apple limited-observability", "loaded observability views", start=apple_progress, detail=", ".join(sorted(apple_frames)))

apple_inventory_rows = []
apple_fold_frames = []
apple_selected_rows = []

for obs_idx, (obs, df_obs_raw) in enumerate(sorted(apple_frames.items()), start=1):
    progress_update("Apple limited-observability", "processing observability view", start=apple_progress, current=obs_idx, total=len(apple_frames), detail=obs)
    df_obs = clean_and_debias_telemetry(df_obs_raw)
    feature_cols = drop_constant_features(df_obs)
    if len(feature_cols) < 2:
        continue
    scenario_count = df_obs["scenario"].nunique()
    workload_count = df_obs["workload"].nunique()
    apple_inventory_rows.append({
        "observability": obs,
        "rows": len(df_obs),
        "workloads": workload_count,
        "scenarios_including_benign": scenario_count,
        "available_features": len(feature_cols),
    })

    base_model = fit_cintas_from_benign(df_obs, feature_cols, lambda_res=0.5, weight_mode="uniform")
    df_obs = df_obs.copy()
    df_obs["cias_sample_score"] = score_samples(df_obs, base_model)
    _, _, ranks = build_causal_and_rank_features_for_setup(
        setup=obs,
        df=df_obs,
        feature_cols=feature_cols,
        out_root=APPLE_OUT / obs.lower(),
        corr_threshold=0.35,
        top_k_plot=min(max(APPLE_FEATURE_BUDGETS), len(feature_cols)),
        score_col="cias_sample_score",
        graph_bootstraps=APPLE_GRAPH_BOOTSTRAPS,
        graph_subsample_frac=APPLE_GRAPH_SUBSAMPLE_FRAC,
        graph_stability_threshold=APPLE_GRAPH_STABILITY_THRESHOLD,
        seed=SEED + obs_idx,
    )

    for top_k in APPLE_FEATURE_BUDGETS:
        feats = _selected_features(ranks, min(int(top_k), len(feature_cols)), feature_cols)
        for lambda_res in APPLE_LAMBDA_VALUES:
            for weight_mode in APPLE_WEIGHT_MODES:
                model = fit_cintas_from_benign(df_obs, feats, lambda_res=float(lambda_res), weight_mode=weight_mode)
                fp_err = _fixed_point_error(df_obs, model, APPLE_FIXED_POINT_Q)
                for agg_mode in APPLE_AGG_MODES:
                    eval_df = run_exact_eval_for_setup(
                        setup=obs,
                        df=df_obs,
                        model=model,
                        scenarios_eval=APPLE_SCENARIOS,
                        window_sizes=APPLE_WINDOW_SIZES,
                        n_splits_list=(APPLE_N_SPLITS,),
                        default_p_quantile=APPLE_P_QUANTILE,
                        agg_mode=agg_mode,
                        seed=SEED,
                    )
                    if eval_df.empty:
                        continue
                    eval_df["observability"] = obs
                    eval_df["top_k"] = int(len(feats))
                    eval_df["n_selected_features"] = int(len(feats))
                    eval_df["available_features"] = int(len(feature_cols))
                    eval_df["lambda_res"] = float(lambda_res)
                    eval_df["weight_mode"] = str(weight_mode)
                    eval_df["fixed_point_q"] = int(APPLE_FIXED_POINT_Q)
                    eval_df["fp_mae"] = fp_err["fp_mae"]
                    eval_df["fp_max_abs"] = fp_err["fp_max_abs"]
                    apple_fold_frames.append(eval_df)
                    apple_selected_rows.append({
                        "observability": obs,
                        "top_k": int(len(feats)),
                        "lambda_res": float(lambda_res),
                        "agg_mode": str(agg_mode),
                        "weight_mode": str(weight_mode),
                        "fixed_point_q": int(APPLE_FIXED_POINT_Q),
                        "available_features": int(len(feature_cols)),
                        "features": ",".join(feats),
                    })

apple_inventory = pd.DataFrame(apple_inventory_rows).sort_values("observability")
apple_fold_results = pd.concat(apple_fold_frames, ignore_index=True) if apple_fold_frames else pd.DataFrame()
if apple_fold_results.empty:
    raise RuntimeError("Apple supplemental study produced no evaluation rows.")

metric_cols = ["auc_roc", "auc_pr", "f1", "bal_acc", "mcc", "fpr", "brier", "ece", "fp_mae", "fp_max_abs"]
global_rows = apple_fold_results[apple_fold_results["workload"].astype(str).str.upper() == "ALL"].copy()
apple_summary = (
    global_rows.groupby([
        "observability", "setup", "scenario", "window_size", "agg_mode", "lambda_res",
        "top_k", "weight_mode", "fixed_point_q", "n_selected_features", "available_features",
    ], as_index=False)[metric_cols]
    .mean(numeric_only=True)
)
rank_cols = ["mcc", "bal_acc", "auc_roc", "f1", "fpr"]
apple_best = (
    apple_summary.sort_values(rank_cols, ascending=[False, False, False, False, True], kind="mergesort")
    .groupby(["observability", "scenario"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

per_workload = apple_fold_results[apple_fold_results["workload"].astype(str).str.upper() != "ALL"].copy()
workload_group_cols = [
    "observability", "scenario", "workload", "window_size", "agg_mode", "lambda_res",
    "top_k", "weight_mode", "fixed_point_q", "n_selected_features", "available_features",
]
apple_workload_mean = per_workload.groupby(workload_group_cols, as_index=False)[metric_cols].mean(numeric_only=True)
apple_workload_best = (
    apple_workload_mean.sort_values(rank_cols, ascending=[False, False, False, False, True], kind="mergesort")
    .groupby(["observability", "scenario", "workload"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
apple_workload_summary = (
    apple_workload_best.groupby(["observability", "scenario"], as_index=False)
    .agg(
        workloads=("workload", "nunique"),
        mean_mcc=("mcc", "mean"),
        min_mcc=("mcc", "min"),
        mean_fpr=("fpr", "mean"),
        max_fpr=("fpr", "max"),
        median_top_k=("top_k", "median"),
        median_window_size=("window_size", "median"),
    )
)

inventory_path = _save_apple_table(apple_inventory, "apple_observability_inventory.csv")
fold_path = _save_apple_table(apple_fold_results, "apple_observability_fold_results.csv")
summary_path = _save_apple_table(apple_summary, "apple_observability_summary.csv")
best_path = _save_apple_table(apple_best, "apple_observability_best_by_scenario.csv")
workload_best_path = _save_apple_table(apple_workload_best, "apple_workload_best_by_observability.csv")
workload_summary_path = _save_apple_table(apple_workload_summary, "apple_workload_summary.csv")
selected_path = _save_apple_table(pd.DataFrame(apple_selected_rows).drop_duplicates(), "apple_selected_features.csv")

obs_order = [x for x in ["TIER0_FULL", "TIER1_ALT_CORE", "TIER1_ALT_FULL", "TIER2_CORE", "TIER2_FULL"] if x in set(apple_best["observability"])]
scenario_order = [s for s in APPLE_SCENARIOS if s in set(apple_best["scenario"])]
heat = apple_best.pivot_table(index="observability", columns="scenario", values="mcc", aggfunc="max").reindex(index=obs_order, columns=scenario_order)
fig, ax = plt.subplots(figsize=(8.6, max(3.4, 0.55 * len(obs_order))))
im = ax.imshow(heat.to_numpy(dtype=float), aspect="auto", cmap="cividis", vmin=-0.05, vmax=1.0)
ax.set_title("Supplemental Apple Observability Ablation: Best MCC", fontweight="bold")
ax.set_xlabel("Stress condition")
ax.set_ylabel("Observability view")
ax.set_xticks(np.arange(len(scenario_order)))
ax.set_xticklabels(scenario_order, rotation=30, ha="right")
ax.set_yticks(np.arange(len(obs_order)))
ax.set_yticklabels(obs_order)
for y in range(heat.shape[0]):
    for x in range(heat.shape[1]):
        val = heat.iloc[y, x]
        if pd.notna(val):
            ax.text(x, y, f"{val:.2f}", ha="center", va="center", fontsize=8, color="white" if val < 0.55 else "black")
cbar = fig.colorbar(im, ax=ax, shrink=0.86, pad=0.02)
cbar.set_label("Best MCC")
apple_obs_fig = _save_apple_fig(fig, "fig_apple_observability_mcc_heatmap.png")

apple_workload_best["obs_scenario"] = apple_workload_best["observability"].astype(str) + " / " + apple_workload_best["scenario"].astype(str)
wl_order = apple_workload_best.groupby("workload")["mcc"].mean().sort_values(ascending=False).index.tolist()
row_order = apple_workload_best.groupby("obs_scenario")["mcc"].mean().sort_values(ascending=True).index.tolist()
wl_heat = apple_workload_best.pivot_table(index="obs_scenario", columns="workload", values="mcc", aggfunc="max").reindex(index=row_order, columns=wl_order)
fig, ax = plt.subplots(figsize=(max(8.5, 0.70 * len(wl_order)), max(6.0, 0.35 * len(row_order))))
im = ax.imshow(wl_heat.to_numpy(dtype=float), aspect="auto", cmap="magma", vmin=-0.05, vmax=1.0)
ax.set_title("Supplemental Apple Workload Robustness", fontweight="bold")
ax.set_xlabel("Workload")
ax.set_ylabel("Observability / condition")
ax.set_xticks(np.arange(len(wl_order)))
ax.set_xticklabels(wl_order, rotation=45, ha="right")
ax.set_yticks(np.arange(len(row_order)))
ax.set_yticklabels(row_order, fontsize=8)
for y in range(wl_heat.shape[0]):
    for x in range(wl_heat.shape[1]):
        val = wl_heat.iloc[y, x]
        if pd.notna(val):
            ax.text(x, y, f"{val:.2f}", ha="center", va="center", fontsize=6, color="white" if val < 0.62 else "black")
cbar = fig.colorbar(im, ax=ax, shrink=0.88, pad=0.01)
cbar.set_label("Best MCC")
apple_workload_fig = _save_apple_fig(fig, "fig_apple_workload_mcc_heatmap.png")

manifest_path = write_run_manifest(
    APPLE_OUT,
    repo_root=REPO_ROOT,
    data_root=APPLE_DATA_ROOT,
    cfg=apple_cfg,
    seed=SEED,
    artifact_paths=(
        inventory_path,
        fold_path,
        summary_path,
        best_path,
        workload_best_path,
        workload_summary_path,
        selected_path,
        apple_obs_fig,
        apple_workload_fig,
    ),
)

display(Markdown("### Apple observability inventory"))
display(apple_inventory)
display(Markdown("### Apple supplemental best results by observability and condition"))
display(apple_best.round(6))
display(Markdown("### Apple workload robustness summary"))
display(apple_workload_summary.round(6))
print(f"Apple summary: {summary_path.relative_to(REPO_ROOT)}")
print(f"Apple best table: {best_path.relative_to(REPO_ROOT)}")
print(f"Apple workload summary: {workload_summary_path.relative_to(REPO_ROOT)}")
print(f"Apple manifest: {manifest_path.relative_to(REPO_ROOT)}")
progress_update("Apple limited-observability", "DONE", start=apple_progress, detail=manifest_path.relative_to(REPO_ROOT))


## 19. Paper-Ready Output Checklist


In [ ]:
golden_candidate = globals().get("golden_path", FPGA_OUT / "cintas_setupA_q15_golden_vectors.csv")
rtl_candidate = globals().get("existing", RTL_SWEEP_OUT / "rtl_resource_summary.csv")

final_artifacts = [
    TCAD_OUT / "run_manifest.json",
    TCAD_OUT / "tcad_ablation_summary.csv",
    TCAD_OUT / "tcad_ablation_fold_results.csv",
    TCAD_OUT / "tcad_selected_features.csv",
    TCAD_OUT / "paper_figures" / "tcad_detection_quality_paper_selected.csv",
    TCAD_OUT / "paper_figures" / "fig_results_detection_quality.png",
    TCAD_OUT / "paper_figures" / "gallery_manifest_5tables_6figures.csv",
    TCAD_OUT / "paper_figures" / "gallery_fig1_full_dse_heatmap.png",
    TCAD_OUT / "paper_figures" / "gallery_fig4_stable_graph_network.png",
    TCAD_OUT / "paper_figures" / "gallery_fig6_deployment_feasibility.png",
    LIFECYCLE_OUT / "run_manifest.json",
    RESULTS_ROOT / "paper_tbd_replacements.csv",
    PROGRESS_LOG,
    golden_candidate,
    rtl_candidate,
    RESULTS_ROOT / "apple_limited_observability" / "apple_observability_summary.csv",
    RESULTS_ROOT / "apple_limited_observability" / "paper_figures" / "fig_apple_observability_mcc_heatmap.png",
]

for artifact in final_artifacts:
    print(f"{'OK' if Path(artifact).exists() else 'MISSING'}  {Path(artifact).relative_to(REPO_ROOT)}")

print("\nCITADEL TCAD notebook run complete.")
